In [ ]:
# Import the relevant libraries
import copy
import importlib.util
import re
from datetime import datetime
from pathlib import Path

import cmasher as cmr
import matplotlib.pyplot as plt
import netCDF4 as nc
import numpy as np
from astropy.io import fits
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D


def load_time_distance_config(config_file):

    '''
    Purpose
    -------
    Load the user-defined parameters for the plotting notebook.

    Inputs
    ------
    config_file: pathlib.Path
        Path to the Python configuration file.

    Outputs
    -------
    config: dict
        Dictionary containing the plotting and time-distance parameters.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Load the Python configuration file
    spec = importlib.util.spec_from_file_location('config', config_file)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    return module.config


def load_local_module(module_name, file_path):

    '''
    Purpose
    -------
    Load a local Python module from a file path.

    Inputs
    ------
    module_name: str
        Name that will be assigned to the loaded module.

    file_path: pathlib.Path
        Path to the Python source file.

    Outputs
    -------
    module: python module
        Imported module object.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Load the requested local module
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    return module


def resolve_config_file():

    '''
    Purpose
    -------
    Find the shared configuration file from the notebook or project root.

    Inputs
    ------
    None

    Outputs
    -------
    config_file: pathlib.Path
        Resolved path to the shared configuration file.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Search the common notebook and project-root locations
    config_candidates = [
        Path('config.py'),
        Path('Code/Time-Distance/config.py')]

    for candidate in config_candidates:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError('Could not find config.py.')


def load_project_modules(config_file):

    '''
    Purpose
    -------
    Load project modules.

    Inputs
    ------
    config_file: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    module_dir = config_file.parent
    agw_filter_module = load_local_module('agw_filter', module_dir / 'agw_filter.py')
    time_distance_module = load_local_module('time_distance_pipeline', module_dir / 'time_distance.py')

    return {
        'agw_filter_module': agw_filter_module,
        'time_distance_module': time_distance_module,
        'naming_module': None,
        'AGWFilter': agw_filter_module.AGWFilter,
        'spectral_analysis': agw_filter_module.spectral_analysis,
        'isothermal_dispersion_equations': agw_filter_module.isothermal_dispersion_equations}


def _normalize_source_type(source_type_value):

    '''
    Purpose
    -------
    Normalize the configured source-type label to the canonical runtime name.

    Inputs
    ------
    source_type_value: object
        Raw source-type value from the configuration module.

    Outputs
    -------
    normalized_source_type: str
        Canonical source-type label used by the pipeline.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize the label before validating it downstream.
    normalized_source_type = str(source_type_value).strip().lower()

    # Preserve compatibility with the older single-cube alias.
    if normalized_source_type == 'single_netcdf_cube':
        return 'single_cube'

    return normalized_source_type



def _normalize_element(element):

    '''
    Purpose
    -------
    Normalize a spectral-element token to title case.

    Inputs
    ------
    element: object
        Raw element label.

    Outputs
    -------
    normalized_element: str
        Element label with only the first character capitalized.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Build the requested result from the provided inputs.
    if element in ['', None]:
        return ''

    element = str(element).strip()

    if element == '':
        return ''

    return element[0].upper() + element[1:].lower()



def _slugify(value):

    '''
    Purpose
    -------
    Convert a free-form label into a filesystem-friendly slug.

    Inputs
    ------
    value: object
        Raw string-like value to normalize.

    Outputs
    -------
    slug: str
        Lowercase slug with non-alphanumeric runs collapsed to underscores.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Build the requested result from the provided inputs.
    value = str(value).strip().lower()
    value = value.replace('å', 'a')
    value = re.sub(r'angstrom', 'a', value, flags = re.IGNORECASE)
    value = re.sub(r'[^0-9a-z]+', '_', value)
    value = re.sub(r'_+', '_', value)

    return value.strip('_')



def _join_slug(parts):

    '''
    Purpose
    -------
    Join multiple slug fragments while dropping empty entries.

    Inputs
    ------
    parts: iterable
        Slug fragments or raw values.

    Outputs
    -------
    joined_slug: str
        Underscore-delimited slug assembled from the non-empty parts.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize each fragment into slug form and drop whitespace-only entries.
    parts = [_slugify(part) for part in parts if str(part).strip() != '']

    # Drop any fragments that collapsed to an empty slug.
    parts = [part for part in parts if part != '']

    return '_'.join(parts).strip('_')



def _load_netcdf_coordinate(variable):

    '''
    Purpose
    -------
    Load a NetCDF coordinate and replace fill-value placeholders with `NaN`.

    Inputs
    ------
    variable: netCDF4.Variable
        Coordinate variable to read.

    Outputs
    -------
    values: np.array, float
        Coordinate values with invalid placeholders converted to `NaN`.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Load the coordinate values as float64 so the later checks behave consistently.
    values = np.asarray(variable[:], dtype = np.float64)
    fill_value = getattr(variable, '_FillValue', None)

    # Replace the declared NetCDF fill value with `NaN`.
    if fill_value is not None:
        values = np.where(values == fill_value, np.nan, values)

    # Treat extremely large placeholder values as invalid coordinates.
    values = np.where(np.abs(values) > 1.0e30, np.nan, values)

    return values



def _convert_netcdf_length_array_to_Mm(values, units, label):

    '''
    Purpose
    -------
    Convert a NetCDF length coordinate into megameters.

    Inputs
    ------
    values: np.array
        Length coordinate values.

    units: str
        Unit string stored in the NetCDF metadata.

    label: str
        Coordinate name used in error messages.

    Outputs
    -------
    values_Mm: np.array, float
        Coordinate values converted to Mm.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize the unit string before matching it to the supported conversions.
    unit = str(units or '').strip().lower()

    # Select the conversion factor that maps the native coordinate units into Mm.
    if unit in ['', 'mm']:
        factor = 1.0
    elif unit == 'cm':
        factor = 1.0e-8
    elif unit == 'm':
        factor = 1.0e-6
    elif unit == 'km':
        factor = 1.0e-3
    else:
        raise ValueError(
            f'Unsupported length unit {units!r} for NetCDF coordinate {label}. '
            f'Use cm, m, km, or Mm.')

    # Apply the unit conversion in a single vectorized operation.
    return np.asarray(values, dtype = np.float64)*factor



def _normalize_single_cube_height_values_km(height_values_km):

    '''
    Purpose
    -------
    Re-reference single-cube heights so the first layer is treated as zero height.

    Inputs
    ------
    height_values_km: np.array
        Physical heights in km.

    Outputs
    -------
    normalized_height_values_km: np.array, float
        Heights shifted so that `xc3[0]` becomes the reference level.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Convert the height coordinate into float64 before checking and shifting it.
    height_values_km = np.asarray(height_values_km, dtype = np.float64)

    # Reject empty coordinate arrays before referencing the first layer.
    if height_values_km.size == 0:
        raise ValueError('Could not normalize single_cube heights because the xc3 coordinate is empty.')

    # Use the first layer as the photospheric reference height.
    photosphere_height_km = float(height_values_km[0])

    # Reject invalid reference heights so all later differences remain physical.
    if not np.isfinite(photosphere_height_km):
        raise ValueError(
            'Could not normalize single_cube heights because xc3[0] is invalid. '
            'The photospheric reference height must be finite.')

    # Shift all heights so the first layer is treated as zero height.
    return height_values_km - photosphere_height_km



def _convert_netcdf_time_array_to_seconds(values, units):

    '''
    Purpose
    -------
    Convert a NetCDF time coordinate into seconds.

    Inputs
    ------
    values: np.array
        Time coordinate values.

    units: str
        Unit string stored in the NetCDF metadata.

    Outputs
    -------
    values_seconds: np.array, float
        Time values converted to seconds.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Normalize the unit string before matching it to the supported conversions.
    unit = str(units or '').strip().lower()

    # Select the conversion factor that maps the native time units into seconds.
    if unit in ['', 's', 'sec', 'secs', 'second', 'seconds']:
        factor = 1.0
    elif unit in ['ms', 'millisecond', 'milliseconds']:
        factor = 1.0e-3
    elif unit in ['min', 'mins', 'minute', 'minutes']:
        factor = 60.0
    elif unit in ['h', 'hr', 'hrs', 'hour', 'hours']:
        factor = 3600.0
    else:
        raise ValueError(
            f'Unsupported time unit {units!r} for the NetCDF time coordinate. '
            f'Use s, ms, min, or h.')

    # Apply the unit conversion in a single vectorized operation.
    return np.asarray(values, dtype = np.float64)*factor



def _infer_uniform_step(values, label):

    '''
    Purpose
    -------
    Infer a uniform sampling step from a finite coordinate array.

    Inputs
    ------
    values: np.array
        Coordinate samples.

    label: str
        Coordinate label used in error messages.

    Outputs
    -------
    reference_step: float
        Inferred uniform sampling interval.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Keep only the finite samples before estimating the coordinate step.
    finite_values = np.asarray(values[np.isfinite(values)], dtype = np.float64)

    # A uniform step cannot be inferred from fewer than two valid samples.
    if finite_values.size < 2:
        raise ValueError(f'Need at least two finite samples to infer {label}.')

    # Compute the nonzero finite step sizes between adjacent samples.
    steps = np.diff(finite_values)
    steps = steps[np.isfinite(steps)]
    steps = steps[np.abs(steps) > 0.0]

    # Reject coordinates that contain only repeated or invalid values.
    if steps.size == 0:
        raise ValueError(f'Could not infer a nonzero step size for {label}.')

    # Use the median absolute step as the reference spacing.
    reference_step = float(np.median(np.abs(steps)))

    # Enforce uniform sampling so the Fourier metadata remain reliable.
    if not np.allclose(np.abs(steps), reference_step, rtol = 1.0e-9, atol = 1.0e-12):
        raise ValueError(f'{label} is not uniformly sampled in the NetCDF file.')

    return reference_step



def infer_netcdf_time_step_seconds(file_path):

    '''
    Purpose
    -------
    Infer netcdf time step seconds.

    Inputs
    ------
    file_path: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    require_netcdf4()

    # Resolve the file path once before checking its existence and opening it.
    file_path = Path(file_path).expanduser().resolve()

    # Reject missing files before trying to open the dataset.
    if not file_path.exists():
        raise FileNotFoundError(f'NetCDF cube file not found: {file_path}')

    # Open the NetCDF file and select the time coordinate used by the cube.
    with nc.Dataset(file_path) as netcdf_file:
        if 'time' in netcdf_file.variables:
            time_variable_name = 'time'
        elif 't' in netcdf_file.variables:
            time_variable_name = 't'
        else:
            raise ValueError(
                f'Could not find a time coordinate in {file_path}. '
                f'Expected a variable named time or t.')

        # Load the coordinate values and convert them into seconds.
        time_variable = netcdf_file.variables[time_variable_name]
        time_values = _load_netcdf_coordinate(time_variable)
        time_seconds = _convert_netcdf_time_array_to_seconds(time_values, getattr(time_variable, 'units', ''))

    # Infer the uniform cadence from the converted time coordinate.
    return _infer_uniform_step(time_seconds, 'dt')



def infer_netcdf_pixel_scale_Mm(file_path):

    '''
    Purpose
    -------
    Infer netcdf pixel scale Mm.

    Inputs
    ------
    file_path: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    require_netcdf4()

    # Resolve the file path once before checking its existence and opening it.
    file_path = Path(file_path).expanduser().resolve()

    # Reject missing files before trying to open the dataset.
    if not file_path.exists():
        raise FileNotFoundError(f'NetCDF cube file not found: {file_path}')

    # Open the NetCDF file and inspect the candidate horizontal coordinate arrays.
    with nc.Dataset(file_path) as netcdf_file:
        horizontal_axes = []
        for axis_name in ['xb1', 'xb2', 'xc1', 'xc2']:

            # Skip coordinate names that are not present in this file.
            if axis_name not in netcdf_file.variables:
                continue

            # Load the current coordinate axis and discard invalid or placeholder values.
            axis_variable = netcdf_file.variables[axis_name]
            axis_values = _load_netcdf_coordinate(axis_variable)

            # Ignore axes that do not contain enough valid samples to define a spacing.
            if np.count_nonzero(np.isfinite(axis_values)) < 2:
                continue

            # Convert the coordinate to Mm and store its inferred step size.
            axis_values_Mm = _convert_netcdf_length_array_to_Mm(axis_values, getattr(axis_variable, 'units', ''), axis_name)
            horizontal_axes.append((axis_name, _infer_uniform_step(axis_values_Mm, axis_name)))

    # Reject files that do not expose any valid horizontal coordinate.
    if len(horizontal_axes) == 0:
        raise ValueError(
            f'Could not infer dx from {file_path}. The NetCDF file does not contain a valid '
            f'horizontal coordinate array in xb1, xb2, xc1, or xc2. '
            f'If this is one of the older uncorrected CO5BOLD files, regenerate it with the '
            f'coordinate-fixing download script first.')

    # Use the first valid horizontal axis as the reference pixel scale.
    dx_Mm = float(horizontal_axes[0][1])

    # Enforce consistent spacing across every valid horizontal coordinate array.
    for axis_name, axis_step in horizontal_axes[1:]:
        if not np.isclose(axis_step, dx_Mm, rtol = 1.0e-9, atol = 1.0e-12):
            raise ValueError(
                f'Horizontal coordinate spacing is inconsistent in {file_path}: '
                f'{horizontal_axes[0][0]} gives {dx_Mm:g} Mm while {axis_name} gives {axis_step:g} Mm.')

    return dx_Mm



def infer_netcdf_sampling(file_path):

    '''
    Purpose
    -------
    Infer netcdf sampling.

    Inputs
    ------
    file_path: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    file_path = Path(file_path).expanduser().resolve()

    # Reject missing files before trying to infer any sampling metadata.
    if not file_path.exists():
        raise FileNotFoundError(f'NetCDF cube file not found: {file_path}')

    # Return both the cadence and pixel scale in one convenience dictionary.
    return {
        'dt_seconds': infer_netcdf_time_step_seconds(file_path),
              'dx_Mm': infer_netcdf_pixel_scale_Mm(file_path)}



def infer_netcdf_height_pair_km(file_path, h1, h2):

    '''
    Purpose
    -------
    Infer netcdf height pair km.

    Inputs
    ------
    file_path: object
        Input parameter.
    h1: object
        Input parameter.
    h2: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    require_netcdf4()

    # Resolve the file path once before checking its existence and opening it.
    file_path = Path(file_path).expanduser().resolve()

    # Reject missing files before trying to infer any height metadata.
    if not file_path.exists():
        raise FileNotFoundError(f'NetCDF cube file not found: {file_path}')

    # Parse the requested heights as integer layer indices.
    try:
        h1_index = int(str(h1).strip())
        h2_index = int(str(h2).strip())
    except ValueError as exc:
        raise ValueError('single_cube heights must be given as integer z indices in h1 and h2.') from exc

    # Load the full physical height coordinate so the requested layers can be validated.
    height_values_km = infer_netcdf_height_coordinates_km(file_path)

    # Reject layer indices that fall outside the available height range.
    if h1_index < 0 or h1_index >= height_values_km.size:
        raise IndexError(f'h1 = {h1_index} is out of range for xc3 with length {height_values_km.size}.')
    if h2_index < 0 or h2_index >= height_values_km.size:
        raise IndexError(f'h2 = {h2_index} is out of range for xc3 with length {height_values_km.size}.')

    # Reject requested layers whose physical heights are invalid.
    if not np.isfinite(height_values_km[h1_index]) or not np.isfinite(height_values_km[h2_index]):
        raise ValueError(
            f'Could not infer physical heights from {file_path}. The requested xc3 entries for h1 or h2 are invalid.')

    # Return both the indices and their physical heights for downstream metadata.
    return {
        'h1_index': h1_index,
        'h2_index': h2_index,
           'h1_km': float(height_values_km[h1_index]),
           'h2_km': float(height_values_km[h2_index])}



def parse_spectral_line(value):

    '''
    Purpose
    -------
    Extract a spectral-line identifier from a filename or metadata string.

    Inputs
    ------
    value: object
        Candidate filename, path, or metadata token.

    Outputs
    -------
    spectral_line: str
        Parsed line identifier, or an empty string when no match is found.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Build the requested result from the provided inputs.
    if value in ['', None]:
        return ''

    value = str(value)
    basename = Path(value).name

    # HMI Doppler and continuum products are Fe6173 even when the filename omits the line.
    if re.search(r'hmi[\s_.-]*(?:dop|cont)', basename, flags = re.IGNORECASE) or re.search(r'hmi[\s_.-]*(?:dop|cont)', value, flags = re.IGNORECASE):
        return '6173'

    patterns = [
        r'fe[\s_.-]*(\d{4,5})',
        r'([a-z]{1,2})[\s_.-]*(\d{4,5})',
        r'(\d{4,5})(?=\D*(?:fits|fit|fts|h5|hdf5|nc|$))',
        r'(\d{4,5})']

    for pattern in patterns:
        match = re.search(pattern, basename, flags = re.IGNORECASE)
        if match is not None:
            return match.group(match.lastindex)

    for pattern in patterns:
        match = re.search(pattern, value, flags = re.IGNORECASE)
        if match is not None:
            return match.group(match.lastindex)

    return ''



def parse_spectral_identifier(value):

    '''
    Purpose
    -------
    Parse a spectral element and line identifier from a filename or metadata string.

    Inputs
    ------
    value: object
        Candidate filename, path, or metadata token.

    Outputs
    -------
    spectral_id: dict
        Dictionary containing the normalized element and line tokens.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Build the requested result from the provided inputs.
    spectral_id = {
        'element': '',
           'line': ''}

    if value in ['', None]:
        return spectral_id

    value = str(value)
    basename = Path(value).name
    candidates = [basename, value]

    element_line_patterns = [
        r'(?:^|[^A-Za-z])([A-Za-z]{1,2})[\s_.-]*(\d{4,5})(?:\s*(?:a|å|angstrom))?(?=[^0-9]|$)',
        r'(?:^|[^A-Za-z])([A-Za-z]{1,2})\s+[A-Za-z]*\s*(\d{4,5})(?:\s*(?:a|å|angstrom))?(?=[^0-9]|$)']

    for candidate in candidates:
        for pattern in element_line_patterns:
            match = re.search(pattern, candidate, flags = re.IGNORECASE)
            if match is not None:
                spectral_id['element'] = _normalize_element(match.group(1))
                spectral_id['line'] = match.group(2)
                return spectral_id

    token = str(value).lower()
    if re.search(r'hmi[\s_.-]*(?:dop|cont)', token, flags = re.IGNORECASE):
        spectral_id['element'] = 'Fe'
        spectral_id['line'] = '6173'
        return spectral_id

    spectral_id['line'] = parse_spectral_line(value)

    return spectral_id



def read_fits_spectral_identifier(path):

    '''
    Purpose
    -------
    Read spectral-identification hints from the FITS header.

    Inputs
    ------
    path: str or pathlib.Path
        FITS file to inspect.

    Outputs
    -------
    spectral_id: dict
        Dictionary containing any element and line tokens inferred from the header.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Build the requested result from the provided inputs.
    spectral_id = {
        'element': '',
           'line': ''}

    path = Path(path)

    if str(path) in ['', '.'] or path.suffix.lower() not in ['.fits', '.fit', '.fts']:
        return spectral_id

    try:
        header = fits.getheader(path)
    except Exception:
        return spectral_id

    header_keys = [
        'LINE',
        'LINENAME',
        'WAVELNTH',
        'WAVELENG',
        'WAVE_LEN',
        'WAVELEN',
        'CONTENT',
        'BANDPASS',
        'SPECTRAL']

    for key in header_keys:
        if key in header:
            parsed = parse_spectral_identifier(header[key])

            if spectral_id['element'] == '' and parsed['element'] != '':
                spectral_id['element'] = parsed['element']

            if spectral_id['line'] == '' and parsed['line'] != '':
                spectral_id['line'] = parsed['line']

    return spectral_id



def resolve_paired_cube_spectral_identifier(file_path):

    '''
    Purpose
    -------
    Resolve the paired-cube spectral identifier from the filename and, when needed, the FITS header.

    Inputs
    ------
    file_path: str or pathlib.Path
        Input paired-cube file.

    Outputs
    -------
    spectral_id: dict
        Dictionary containing the resolved element and line tokens.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Build the requested result from the provided inputs.
    spectral_id = parse_spectral_identifier(file_path)

    if spectral_id['element'] != '' and spectral_id['line'] != '':
        return spectral_id

    header_id = read_fits_spectral_identifier(file_path)

    if spectral_id['element'] == '' and header_id['element'] != '':
        spectral_id['element'] = header_id['element']

    if spectral_id['line'] == '' and header_id['line'] != '':
        spectral_id['line'] = header_id['line']

    return spectral_id



def normalize_observable_slug(observable):

    '''
    Purpose
    -------
    Normalize an observable name to the canonical slug used in output filenames.

    Inputs
    ------
    observable: object
        Raw observable token.

    Outputs
    -------
    observable_slug: str
        Canonical observable slug.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Build the requested result from the provided inputs.
    alias_lookup = {
        'bx': 'bb1',
        'by': 'bb2',
        'bz': 'bb3',
        'b1': 'bb1',
        'b2': 'bb2',
        'b3': 'bb3'}

    observable = str(observable or '').strip().lower()

    return alias_lookup.get(observable, observable)



def infer_observational_context(file_path):

    '''
    Purpose
    -------
    Infer a human-readable context label for an observational paired-cube file.

    Inputs
    ------
    file_path: str or pathlib.Path
        Input observational file path.

    Outputs
    -------
    context: dict
        Title and slug labels derived from the observation date or parent directory.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Build the requested result from the provided inputs.
    file_path = str(file_path)
    match = re.search(r'(\d{1,2}[A-Za-z]{3}\d{4})', file_path)

    if match is not None:
        date_token = match.group(1)
        date_obj = datetime.strptime(date_token, '%d%b%Y')
        return {
            'title': date_obj.strftime('%d %B %Y'),
             'slug': date_obj.strftime('%d%b%Y').lower()}

    path = Path(file_path)

    return {
        'title': path.parent.name,
         'slug': _slugify(path.parent.name)}



def infer_simulation_context(file_path):

    '''
    Purpose
    -------
    Infer a human-readable context label for a simulation cube path.

    Inputs
    ------
    file_path: str or pathlib.Path
        Input simulation file path.

    Outputs
    -------
    context: dict
        Title and slug labels derived from the simulation component and field strength.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    # Build the requested result from the provided inputs.
    path = Path(file_path)
    component = ''
    strength = ''

    for part in path.parts:
        lower_part = part.lower()
        if lower_part in ['hx', 'vx', 'z0']:
            component = lower_part
        if re.fullmatch(r'\d+g', lower_part):
            strength = lower_part

    if component == '' or strength == '':
        match = re.search(r'(hx|vx|z0)[_\-/](\d+g)', str(path), flags = re.IGNORECASE)
        if match is not None:
            component = match.group(1).lower()
            strength = match.group(2).lower()

    title_parts = []
    slug_parts = []

    if component != '':
        title_parts.append(component)
        slug_parts.append(component)
    if strength != '':
        title_parts.append(strength.upper())
        slug_parts.append(strength)

    if len(title_parts) == 0:
        title_parts.append(path.stem)
        slug_parts.append(_slugify(path.stem))

    return {
        'title': ' '.join(title_parts),
         'slug': _join_slug(slug_parts)}



def build_context_labels(config):

    '''
    Purpose
    -------
    Build context labels.

    Inputs
    ------
    config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    data = config['data']
    source_type = _normalize_source_type(data['source_type'])

    if source_type == 'paired_cubes':
        context = infer_observational_context(data.get('v1', data['paired_cubes'].get('v1', '')))
    elif source_type == 'single_cube':
        context = infer_simulation_context(data.get('file', data['single_cube'].get('file', '')))
    else:
        raise ValueError(f'Unsupported source_type: {source_type}')

    return {
        'title_prefix': context['title'],
        'title_suffix': '',
        'slug_prefix': context['slug'],
        'slug_suffix': ''}



def build_magnetic_mask_slug(config):

    '''
    Purpose
    -------
    Build magnetic mask slug.

    Inputs
    ------
    config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    filtering = config.get('filtering', {})
    filter_sequence = filtering.get('filter_sequence', [])
    magnetogram_config = filtering.get('magnetogram', {})

    if not filtering.get('enabled', False):
        return ''

    if 'magnetogram' not in filter_sequence:
        return ''

    if not magnetogram_config.get('enabled', False):
        return ''

    selection = magnetogram_config.get('selection', 'nonmagnetic').lower()
    threshold_G = float(magnetogram_config['threshold_G'])
    threshold_slug = _slugify(f'{threshold_G:g}')

    if selection == 'nonmagnetic':
        relation = 'le'
    elif selection == 'magnetic':
        relation = 'gt'
    else:
        raise ValueError("magnetogram selection must be either 'magnetic' or 'nonmagnetic'.")

    return f'b_{relation}_{threshold_slug}g'


PLOT_FLAG_MAP = {
            'cross_correlation': 'time_distance',
            'phase_difference': 'phase_difference',
                      'komega': 'komega_diagram',
              'gaussian_filter': 'gaussian_filter',
                  'dopplergram': 'filtered_dopplergram',
                    'composite': 'composite_diagnostics',
 'correlation_radius_animation': 'correlation_radius_animation',
'correlation_vertical_animation': 'correlation_vertical_animation',
       'phase_radius_animation': 'phase_radius_animation',
     'phase_vertical_animation': 'phase_vertical_animation'}


PLOT_DEFAULTS = {
    'time_distance': {
         'plot_type': 'time_distance',
         'axis_type': 'time_lag',
           'figsize': [10, 6],
              'cmap': 'viridis',
       'value_label': 'Normalized Cross-Correlation',
 'normalize_display': True,
              'ylim': [-60.0, 60.0],
       'output_file': ''},

    'phase_difference': {
         'plot_type': 'phase_difference',
         'axis_type': 'frequency',
           'figsize': [10, 6],
              'cmap': 'seismic_r',
       'value_label': r'$\Delta \phi$ [deg]',
              'ylim': [-10.0, 10.0],
              'vmin': -50.0,
              'vmax': 50.0,
       'output_file': ''},

    'komega_diagram': {
         'plot_type': 'komega_diagram',
           'figsize': [10, 6],
              'cmap': cmr.fusion,
       'value_label': r'$\Delta \phi$ [deg]',
              'xlim': [0.0, 7.0],
              'ylim': [0.0, 10.0],
              'vmin': -20.0,
              'vmax': 80.0,
    'overlay_curves': True,
       'output_file': ''},

    'gaussian_filter': {
         'plot_type': 'gaussian_filter',
           'dataset': 'v1',
           'figsize': [8, 5],
              'cmap': 'viridis',
              'xlim': [0.0, 7.0],
              'ylim': [0.0, 10.0],
 'overlay_lamb_line': True,
 'sound_speed_km_s': 7.0,
       'output_file': ''},

    'filtered_dopplergram': {
         'plot_type': 'filtered_dopplergram',
           'dataset': 'v1',
        'time_index': 0,
           'figsize': [8, 5],
              'cmap': 'RdBu_r',
       'output_file': ''},

    'composite_diagnostics': {
         'plot_type': 'composite_diagnostics',
           'figsize': [27.5, 8.0],
       'output_file': ''},


    'field_strength_comparison': {
         'plot_type': 'field_strength_comparison',
         'cube_paths': [],
         'observable': '',
                'h1': None,
                'h2': None,
           'figsize': [31.5, 14.5],
   'execution_mode': 'load',
'missing_data_behavior': 'error',
              'show': True,
            'komega': {},
 'cross_correlation': {},
   'phase_difference': {},
     'gaussian_filter': {},
       'output_file': ''},

    'field_orientation_comparison': {
         'plot_type': 'field_orientation_comparison',
         'cube_paths': [],
         'observable': '',
                'h1': None,
                'h2': None,
           'figsize': [24.0, 14.5],
   'execution_mode': 'load',
'missing_data_behavior': 'error',
              'show': True,
            'komega': {},
 'cross_correlation': {},
   'phase_difference': {},
     'gaussian_filter': {},
       'output_file': ''},

    'gaussian_filter_comparison': {
         'plot_type': 'gaussian_filter_comparison',
         'cube_paths': [],
 'filter_params_list': [],
         'observable': '',
                'h1': None,
                'h2': None,
           'figsize': None,
   'execution_mode': 'load',
'missing_data_behavior': 'error',
              'show': True,
            'komega': {},
 'cross_correlation': {},
   'phase_difference': {},
     'gaussian_filter': {},
       'output_file': ''},



    'correlation_radius_animation': {
         'plot_type': 'correlation_radius_animation',
         'axis_type': 'time_lag',
           'figsize': [14, 5],
              'cmap': 'viridis',
       'value_label': 'Cross-Correlation',
  'line_value_label': 'Cross-Correlation',
'colorbar_value_label': 'Normalized Cross-Correlation',
'normalize_map_display': True,
'normalize_line_display': False,
       'interval_ms': 800,
              'ylim': [-60.0, 60.0],
       'output_file': ''},

    'correlation_vertical_animation': {
         'plot_type': 'correlation_vertical_animation',
         'axis_type': 'time_lag',
           'figsize': [14, 5],
              'cmap': 'viridis',
       'value_label': 'Cross-Correlation',
  'line_value_label': 'Cross-Correlation',
'colorbar_value_label': 'Normalized Cross-Correlation',
'normalize_map_display': True,
'normalize_line_display': False,
       'interval_ms': 60,
              'ylim': [-60.0, 60.0],
       'output_file': ''},

    'phase_radius_animation': {
         'plot_type': 'phase_radius_animation',
         'axis_type': 'frequency',
           'figsize': [14, 5],
              'cmap': 'seismic_r',
       'value_label': r'$\Delta \phi$ [deg]',
       'interval_ms': 800,
              'ylim': [-10.0, 10.0],
              'vmin': -50.0,
              'vmax': 50.0,
       'output_file': ''},

    'phase_vertical_animation': {
         'plot_type': 'phase_vertical_animation',
         'axis_type': 'frequency',
           'figsize': [14, 5],
              'cmap': 'seismic_r',
       'value_label': r'$\Delta \phi$ [deg]',
       'interval_ms': 60,
              'ylim': [-10.0, 10.0],
              'vmin': -50.0,
              'vmax': 50.0,
       'output_file': ''}}



def _join_title(parts):

    '''
    Purpose
    -------
    Join title.

    Inputs
    ------
    parts: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    parts = [str(part).strip() for part in parts if str(part).strip() != '']

    return ' '.join(parts).strip()



def _format_height_km(height_km):

    '''
    Purpose
    -------
    Format height km.

    Inputs
    ------
    height_km: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    height_km = float(height_km)

    if height_km.is_integer():
        return str(int(height_km))

    return f'{height_km:g}'



def _format_filter_title(filter_name):

    '''
    Purpose
    -------
    Format filter title.

    Inputs
    ------
    filter_name: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    return str(filter_name).replace('_', ' ').title()



def normalize_observable_name(observable_name):

    '''
    Purpose
    -------
    Normalize observable name.

    Inputs
    ------
    observable_name: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    observable_name = str(observable_name).strip()

    if observable_name == '':
        return ''

    alias_lookup = {
           'bx': 'bb1',
           'by': 'bb2',
           'bz': 'bb3',
           'b1': 'bb1',
           'b2': 'bb2',
           'b3': 'bb3'}

    return alias_lookup.get(observable_name.lower(), observable_name)



def build_magnetic_mask_title(config):

    '''
    Purpose
    -------
    Build magnetic mask title.

    Inputs
    ------
    config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    filtering = config.get('filtering', {})
    filter_sequence = filtering.get('filter_sequence', [])
    magnetogram_config = filtering.get('magnetogram', {})

    if not filtering.get('enabled', False):
        return ''

    if 'magnetogram' not in filter_sequence:
        return ''

    if not magnetogram_config.get('enabled', False):
        return ''

    selection = magnetogram_config.get('selection', 'nonmagnetic').lower()
    threshold_G = float(magnetogram_config['threshold_G'])

    if selection == 'nonmagnetic':
        relation = r'\leq'
    elif selection == 'magnetic':
        relation = '>'
    else:
        raise ValueError("magnetogram selection must be either 'magnetic' or 'nonmagnetic'.")

    return rf'$|B| {relation} {threshold_G:g}\ \mathrm{{G}}$'



def build_gaussian_parameter_slug(config):

    '''
    Purpose
    -------
    Build gaussian parameter slug.

    Inputs
    ------
    config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    filtering = config.get('filtering', {})
    filter_sequence = filtering.get('filter_sequence', [])
    gaussian_config = filtering.get('gaussian', {})

    if not filtering.get('enabled', False):
        return ''

    if 'gaussian' not in filter_sequence:
        return ''

    if not gaussian_config.get('enabled', False):
        return ''

    return _join_slug([
        'gauss',
        'ck', f"{float(gaussian_config['central_k']):g}",
        'wk', f"{float(gaussian_config['width_k']):g}",
        'cf', f"{float(gaussian_config['central_f']):g}",
        'wf', f"{float(gaussian_config['width_f']):g}"])



def build_processing_labels(config):

    '''
    Purpose
    -------
    Build processing labels.

    Inputs
    ------
    config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    filtering = config.get('filtering', {})
    filter_sequence = filtering.get('filter_sequence', [])
    enabled_filters = []

    for filter_name in filter_sequence:
        filter_config = filtering.get(filter_name, {})
        if filter_config.get('enabled', True):
            enabled_filters.append(filter_name)

    if filtering.get('enabled', False) and len(enabled_filters) > 0:
        filter_title = ' + '.join([_format_filter_title(filter_name) for filter_name in enabled_filters])
        filter_slug = '_'.join([_slugify(filter_name) for filter_name in enabled_filters])
        magnetic_criterion = build_magnetic_mask_title(config)
        magnetic_slug = build_magnetic_mask_slug(config)
        gaussian_parameter_slug = build_gaussian_parameter_slug(config)
        processing_title = f'{filter_title} Filtered'
        processing_slug_parts = [f'{filter_slug}_filtered']

        if magnetic_criterion != '':
            processing_title = f'{processing_title}, {magnetic_criterion}'

        if gaussian_parameter_slug != '':
            processing_slug_parts.append(gaussian_parameter_slug)

        if magnetic_slug != '':
            processing_slug_parts.append(magnetic_slug)

        return {
            'title': f'({processing_title})',
            'slug': _join_slug(processing_slug_parts),
            'flag_title': processing_title}

    return {
        'title': '(Unfiltered)',
        'slug': 'unfiltered',
        'flag_title': 'Unfiltered'}



def build_filter_sequence_title(config):

    '''
    Purpose
    -------
    Build filter sequence title.

    Inputs
    ------
    config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    filtering = config.get('filtering', {})

    if not filtering.get('enabled', False):
        return ''

    enabled_filters = []
    for filter_name in filtering.get('filter_sequence', []):
        if filtering.get(filter_name, {}).get('enabled', True):
            enabled_filters.append(_format_filter_title(filter_name))

    if len(enabled_filters) == 0:
        return ''

    return _join_title(enabled_filters + ['Filtered'])



def build_source_labels(config, dataset = 'pair'):

    '''
    Purpose
    -------
    Build source labels.

    Inputs
    ------
    config: object
        Input parameter.
    dataset: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    data = config['data']
    source_type = _normalize_source_type(data['source_type'])

    if source_type == 'paired_cubes':
        spectral_id_v1 = resolve_paired_cube_spectral_identifier(data.get('v1', ''))
        spectral_id_v2 = resolve_paired_cube_spectral_identifier(data.get('v2', ''))

        line_v1 = spectral_id_v1['line']
        line_v2 = spectral_id_v2['line']
        element_v1 = spectral_id_v1['element']
        element_v2 = spectral_id_v2['element']

        if line_v1 == '' or line_v2 == '' or element_v1 == '' or element_v2 == '':
            raise ValueError(
                'Could not determine the spectral-line element and wavelength for '
                'source_type = paired_cubes from the data-file names or FITS headers.')

        source_lookup = {
            'v1': {'title': f'{element_v1}{line_v1} Å', 'slug': f'{element_v1.lower()}{line_v1}'},
            'v2': {'title': f'{element_v2}{line_v2} Å', 'slug': f'{element_v2.lower()}{line_v2}'},
            'pair': {'title': f'{element_v1}{line_v1} Å vs {element_v2}{line_v2} Å', 'slug': f'{element_v1.lower()}{line_v1}_{element_v2.lower()}{line_v2}'}}
    elif source_type == 'single_cube':
        h1_km = data.get('resolved_h1_km', '')
        h2_km = data.get('resolved_h2_km', '')
        observable_slug = normalize_observable_name(data.get('observable', ''))

        if h1_km in ['', None] or h2_km in ['', None]:
            raise ValueError(
                'source_type = single_cube requires physical heights to be inferable from '
                'the file xc3 coordinate using h1 and h2.')

        h1_title = f'{_format_height_km(h1_km)} km'
        h2_title = f'{_format_height_km(h2_km)} km'
        h1_slug = f'{_slugify(_format_height_km(h1_km))}km'
        h2_slug = f'{_slugify(_format_height_km(h2_km))}km'

        source_lookup = {
            'v1': {'title': h1_title, 'slug': _join_slug([observable_slug, h1_slug])},
            'v2': {'title': h2_title, 'slug': _join_slug([observable_slug, h2_slug])},
            'pair': {'title': f'{h1_title} vs {h2_title}', 'slug': _join_slug([observable_slug, h1_slug, h2_slug])}}
    else:
        raise ValueError(f'Unsupported source_type: {source_type}')

    if dataset not in source_lookup:
        raise ValueError(f'Unsupported dataset label: {dataset}')

    return source_lookup[dataset]



def build_base_labels(config, include_processing = True):

    '''
    Purpose
    -------
    Build base labels.

    Inputs
    ------
    config: object
        Input parameter.
    include_processing: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    context = build_context_labels(config)
    source = build_source_labels(config, dataset = 'pair')
    processing = build_processing_labels(config)

    title_parts = [context['title_prefix'], source['title']]
    slug_parts = [context['slug_prefix'], source['slug']]

    if include_processing:
        title_parts.append(processing['title'])
        slug_parts.append(processing['slug'])

    title_parts.append(context['title_suffix'])
    slug_parts.append(context['slug_suffix'])

    return {
        'title': _join_title(title_parts),
        'slug': _join_slug(slug_parts)}



def build_composite_suptitle(config):

    '''
    Purpose
    -------
    Build composite suptitle.

    Inputs
    ------
    config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    context = build_context_labels(config)
    source = build_source_labels(config, dataset = 'pair')
    filter_title = build_filter_sequence_title(config)
    magnetic_title = ''

    filtering = config.get('filtering', {})
    magnetogram_config = filtering.get('magnetogram', {})
    if filtering.get('enabled', False) and 'magnetogram' in filtering.get('filter_sequence', []) and magnetogram_config.get('enabled', False):
        threshold_G = float(magnetogram_config['threshold_G'])
        selection = magnetogram_config.get('selection', 'nonmagnetic').lower()
        relation = '<' if selection == 'nonmagnetic' else '>'
        magnetic_title = rf'$|B| {relation} {threshold_G:g}\ \mathrm{{G}}$'

    title = _join_title([
        context['title_prefix'],
        source['title'],
        filter_title,
        context['title_suffix']])

    if magnetic_title != '':
        title = _join_title([title + ',', magnetic_title])

    return title



def build_plot_title(config, product, dataset = 'pair', time_index = None):

    '''
    Purpose
    -------
    Build plot title.

    Inputs
    ------
    config: object
        Input parameter.
    product: object
        Input parameter.
    dataset: object
        Input parameter.
    time_index: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    context = build_context_labels(config)
    source_type = _normalize_source_type(config['data']['source_type'])
    source_dataset = dataset

    if source_type == 'single_cube' and product in ['gaussian_filter', 'filtered_dopplergram']:
        source_dataset = 'pair'

    source = build_source_labels(config, dataset = source_dataset)
    base = build_base_labels(config, include_processing = True)
    processing = build_processing_labels(config)

    if product == 'time_distance':
        return base['title']

    if product == 'phase_difference':
        return base['title']

    if product == 'komega_diagram':
        return base['title']

    if product == 'gaussian_filter':
        pair_source = build_source_labels(config, dataset = 'pair')
        return _join_title([context['title_prefix'], pair_source['title'], context['title_suffix']])

    if product == 'filtered_dopplergram':
        dopplergram_title = _join_title([
            context['title_prefix'],
            source['title'],
            processing['flag_title'],
            context['title_suffix']])

        if time_index is not None:
            dopplergram_title = f'{dopplergram_title} (t = {int(time_index)})'

        return dopplergram_title

    if product == 'composite_diagnostics':
        return build_composite_suptitle(config)

    raise ValueError(f'Unsupported product for title generation: {product}')



def build_output_stem(config, product, dataset = 'pair', time_index = None):

    '''
    Purpose
    -------
    Build output stem.

    Inputs
    ------
    config: object
        Input parameter.
    product: object
        Input parameter.
    dataset: object
        Input parameter.
    time_index: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    if product == 'time_distance':
        return _join_slug([build_base_labels(config, include_processing = True)['slug'], 'xc'])

    if product == 'phase_difference':
        return _join_slug([build_base_labels(config, include_processing = True)['slug'], 'phase_diff'])

    if product == 'komega_diagram':
        return _join_slug([build_base_labels(config, include_processing = True)['slug'], 'komega'])

    source_type = _normalize_source_type(config['data']['source_type'])

    if product == 'gaussian_filter':
        base_slug = build_base_labels(config, include_processing = False)['slug']
        dataset_slug = build_source_labels(config, dataset = dataset)['slug']

        if source_type == 'single_cube':
            dataset_slug = _join_slug([normalize_observable_name(config['data'].get('observable', '')), dataset_slug])

        stem_parts = [base_slug, dataset_slug, 'gaussian_filter']
        gaussian_parameter_slug = build_gaussian_parameter_slug(config)

        if gaussian_parameter_slug != '':
            stem_parts.append(gaussian_parameter_slug)

        return _join_slug(stem_parts)

    if product == 'filtered_dopplergram':
        stem_parts = [
            build_base_labels(config, include_processing = True)['slug'],
            build_source_labels(config, dataset = dataset)['slug'],
            'dopplergram']

        if time_index is not None:
            stem_parts.append(f't{int(time_index):04d}')

        return _join_slug(stem_parts)

    if product == 'composite_diagnostics':
        return _join_slug([build_base_labels(config, include_processing = True)['slug'], 'composite'])

    raise ValueError(f'Unsupported product for file-name generation: {product}')



def get_figure_dir(config):

    '''
    Purpose
    -------
    Get figure dir.

    Inputs
    ------
    config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    paths = config.get('paths', {})

    return paths.get('figure_dir', '')



def get_animation_dir(config):

    '''
    Purpose
    -------
    Get animation dir.

    Inputs
    ------
    config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    paths = config.get('paths', {})

    return paths.get('animation_dir', '')



def prepare_runtime_config(config):

    '''
    Purpose
    -------
    Prepare runtime config.

    Inputs
    ------
    config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    runtime_config = copy.deepcopy(config)
    data = runtime_config['data']
    paths = runtime_config.get('paths', {})
    source_type = _normalize_source_type(data['source_type'])
    data['source_type'] = source_type

    if source_type == 'paired_cubes':
        paired = copy.deepcopy(data.get('paired_cubes', {}))
        data['data_dir'] = paired.get('data_dir', data.get('data_dir', ''))
        data['v1'] = paired.get('v1', data.get('v1', ''))
        data['v2'] = paired.get('v2', data.get('v2', ''))
        data['h1'] = paired.get('h1', data.get('h1', ''))
        data['h2'] = paired.get('h2', data.get('h2', ''))

        if 'p_dx_Mm' not in paired or 'dt' not in paired or 'delta_z_km' not in paired:
            raise ValueError(
                "source_type = 'paired_cubes' requires data['paired_cubes']['delta_z_km'], "
                "data['paired_cubes']['p_dx_Mm'], and data['paired_cubes']['dt'].")

        data['resolved_delta_z_km'] = abs(float(paired['delta_z_km']))
        runtime_config['time_distance']['p_dx_Mm'] = float(paired['p_dx_Mm'])
        runtime_config['time_distance']['dt'] = float(paired['dt'])
    elif source_type == 'single_cube':
        simulation = copy.deepcopy(data.get('single_cube', {}))
        data['file'] = simulation.get('file', data.get('file', ''))
        data['observable'] = simulation.get('observable', data.get('observable', ''))
        data['h1'] = simulation.get('h1', data.get('h1', ''))
        data['h2'] = simulation.get('h2', data.get('h2', ''))
        sampling = infer_netcdf_sampling(data['file'])
        resolved_heights = infer_netcdf_height_pair_km(data['file'], data['h1'], data['h2'])
        runtime_config['time_distance']['dt'] = float(sampling['dt_seconds'])
        runtime_config['time_distance']['p_dx_Mm'] = float(sampling['dx_Mm'])
        data['h1'] = int(resolved_heights['h1_index'])
        data['h2'] = int(resolved_heights['h2_index'])
        data['resolved_h1_index'] = int(resolved_heights['h1_index'])
        data['resolved_h2_index'] = int(resolved_heights['h2_index'])
        data['resolved_h1_km'] = float(resolved_heights['h1_km'])
        data['resolved_h2_km'] = float(resolved_heights['h2_km'])
        data['resolved_delta_z_km'] = abs(float(resolved_heights['h2_km']) - float(resolved_heights['h1_km']))
    else:
        raise ValueError(f'Unsupported source_type: {source_type}')

    data_output_dir = Path(paths['data_output_dir']).expanduser().resolve()
    data['outfile'] = str(data_output_dir / f"{build_output_stem(runtime_config, 'time_distance')}.fits")
    data['phase_outfile'] = str(data_output_dir / f"{build_output_stem(runtime_config, 'phase_difference')}.fits")
    data['komega_outfile'] = str(data_output_dir / f"{build_output_stem(runtime_config, 'komega_diagram')}.fits")

    return runtime_config



def build_plot_request_defaults(plot_type, config):

    '''
    Purpose
    -------
    Build plot request defaults.

    Inputs
    ------
    plot_type: object
        Input parameter.
    config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    if plot_type not in PLOT_DEFAULTS:
        raise KeyError(f'Unsupported plot type: {plot_type}')

    request = copy.deepcopy(PLOT_DEFAULTS[plot_type])

    if plot_type in ['time_distance', 'correlation_radius_animation', 'correlation_vertical_animation']:
        request['data_file'] = config['data']['outfile']
    elif plot_type in ['phase_difference', 'phase_radius_animation', 'phase_vertical_animation']:
        request['data_file'] = config['data']['phase_outfile']
    elif plot_type == 'komega_diagram':
        request['data_file'] = config['data']['komega_outfile']

    return request


def load_time_distance_data(data_file, dt, dx_Mm, axis_type = 'time_lag'):

    '''
    Purpose
    -------
    Load a saved time-distance array and reconstruct the plotting axes.

    Inputs
    ------
    data_file: str or pathlib.Path
        Path to the saved FITS file.

    dt: float
        Cadence of the time series in seconds.

    dx_Mm: float
        Spatial sampling in Mm per pixel.

    axis_type: str, optional
        Name of the second axis. Supported values are 'time_lag' and 'frequency'.

    Outputs
    -------
    data: dict
        Dictionary containing the saved array, plotting axes, labels, and file path.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Read the saved data from disk
    data_file = Path(data_file).expanduser().resolve()
    values = np.asarray(fits.getdata(data_file), dtype = np.float64)
    radii = np.arange(values.shape[0], dtype = np.float64)*float(dx_Mm)

    # Reconstruct the second axis from the saved array size
    if axis_type == 'time_lag':
        vertical_axis = ((np.arange(values.shape[1]) - values.shape[1]//2)*float(dt))/60.0
        vertical_label = 'Time Lag [min]'
        frame_label = 'time lag'
        frame_unit = 'min'
        frame_name = 'time_lag'
    elif axis_type == 'frequency':
        vertical_axis = np.fft.fftshift(np.fft.fftfreq(values.shape[1], d = float(dt)))*1.0e3
        vertical_label = r'$\nu / 2\pi$ [mHz]'
        frame_label = 'frequency'
        frame_unit = 'mHz'
        frame_name = 'frequency'
    else:
        raise ValueError("axis_type must be either 'time_lag' or 'frequency'.")

    return {
        'data_file': data_file,
        'values': values,
        'radii': radii,
        'vertical_axis': vertical_axis,
        'vertical_label': vertical_label,
        'frame_label': frame_label,
        'frame_unit': frame_unit,
        'frame_name': frame_name}


def normalize_cross_correlation_display(values):

    '''
    Purpose
    -------
    Normalize a cross-correlation map for display without changing the
    saved pipeline output. The normalization is applied independently
    at each radius so the 2D map matches the legacy MATLAB-style
    display convention.

    Inputs
    ------
    values: np.array, float
        Cross-correlation map.

    Outputs
    -------
    normalized_values: np.array, float
        Display-normalized cross-correlation map.
    '''

    # Build the requested result from the provided inputs.
    values = np.asarray(values, dtype = np.float64)
    normalized_values = np.array(values, copy = True)

    if values.ndim == 1:
        finite_values = values[np.isfinite(values)]
        if finite_values.size == 0:
            return normalized_values

        vmax = float(np.nanmax(np.abs(finite_values)))
        if not np.isfinite(vmax) or vmax <= 0.0:
            return normalized_values

        return values/vmax

    for radius_index in range(values.shape[0]):
        row = values[radius_index]
        finite_row = row[np.isfinite(row)]
        if finite_row.size == 0:
            continue

        vmax = float(np.nanmax(np.abs(finite_row)))
        if not np.isfinite(vmax) or vmax <= 0.0:
            continue

        normalized_values[radius_index] = row/vmax

    return normalized_values


def load_komega_data(data_file, config):

    '''
    Purpose
    -------
    Load a saved k-omega FITS file and reconstruct its plotting axes.

    Inputs
    ------
    data_file: str or pathlib.Path
        Path to the saved k-omega FITS file.

    config: dict
        Shared configuration dictionary.

    Outputs
    -------
    data: dict
        Dictionary containing the saved array, axes, labels, and file path.

    Author(s)
    ---------
    Julio M. Morales, March 20th, 2026
    '''

    # Build the requested result from the provided inputs.
    data_file = Path(data_file).expanduser().resolve()
    values = np.asarray(fits.getdata(data_file), dtype = np.float64)
    header = fits.getheader(data_file)

    if 'CRVAL1' in header and 'CDELT1' in header:
        k_axis = header['CRVAL1'] + np.arange(values.shape[0], dtype = np.float64)*header['CDELT1']
    else:
        k_nyquist = np.pi/float(config['time_distance']['p_dx_Mm'])
        k_axis = np.linspace(0.0, k_nyquist, values.shape[0], endpoint = True, dtype = np.float64)

    if 'CRVAL2' in header and 'CDELT2' in header:
        nu_axis = header['CRVAL2'] + np.arange(values.shape[1], dtype = np.float64)*header['CDELT2']
    else:
        nu_nyquist = (np.pi/float(config['time_distance']['dt']))/(2.0*np.pi)*1.0e3
        nu_axis = np.linspace(0.0, nu_nyquist, values.shape[1], endpoint = True, dtype = np.float64)

    orientation = {}
    for header_key, metadata_key in [('THAVG1', 'theta_mean_h1_deg'), ('THAVG2', 'theta_mean_h2_deg'), ('PHAVG1', 'phi_mean_h1_deg'), ('PHAVG2', 'phi_mean_h2_deg')]:
        if header_key in header:
            orientation[metadata_key] = float(header[header_key])

    for header_key, metadata_key in [('BAVG_G', 'mean_field_strength_G_between_heights'), ('RHOAVG', 'mean_density_cgs_between_heights'), ('CSAVG', 'mean_sound_speed_km_s_between_heights'), ('CAAVG', 'alfven_speed_km_s'), ('CACSRAT', 'alfven_to_sound_speed_ratio')]:
        if header_key in header:
            orientation[metadata_key] = float(header[header_key])

    if 'PHICIRC' in header:
        orientation['phi_mean_method'] = 'circular' if bool(header['PHICIRC']) else 'linear'

    return {
        'data_file': data_file,
        'values': values,
        'k_axis': np.asarray(k_axis, dtype = np.float64),
        'nu_axis': np.asarray(nu_axis, dtype = np.float64),
        'header': header,
        'orientation': orientation,
        'horizontal_label': r'$k_h$ [Mm$^{-1}$]',
        'vertical_label': r'$\nu / 2\pi$ [mHz]'}


def format_komega_orientation_title(data, config):

    '''
    Purpose
    -------
    Format komega orientation title.

    Inputs
    ------
    data: object
        Input parameter.
    config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    orientation = data.get('orientation', {})
    if len(orientation) == 0:
        return ''

    data_config = config.get('data', {})
    height_specs = [
        (data_config.get('resolved_h1_km', None), orientation.get('theta_mean_h1_deg', np.nan), orientation.get('phi_mean_h1_deg', np.nan)),
        (data_config.get('resolved_h2_km', None), orientation.get('theta_mean_h2_deg', np.nan), orientation.get('phi_mean_h2_deg', np.nan))]
    title_parts = []

    for height_km, theta_mean_deg, phi_mean_deg in height_specs:
        height_label = f"{_format_height_km(height_km)} km" if height_km not in ['', None] else 'unknown height'
        theta_label = '--' if not np.isfinite(theta_mean_deg) else f"$\\langle \\theta \\rangle = {theta_mean_deg:.1f}^{{\\circ}}$"
        phi_label = '--' if not np.isfinite(phi_mean_deg) else f"$\\langle \\phi \\rangle = {phi_mean_deg:.1f}^{{\\circ}}$"
        title_parts.append(f"{height_label}: {theta_label}, {phi_label}")

    if np.isfinite(float(orientation.get('alfven_to_sound_speed_ratio', np.nan))):
        title_parts.append(f"$c_A / c_s = {orientation['alfven_to_sound_speed_ratio']:.3g}$")

    return ' | '.join(title_parts)


def load_reference_atmosphere_model():

    '''
    Purpose
    -------
    Load the reference solar atmosphere used by the original k-omega notebook.

    Inputs
    ------
    None

    Outputs
    -------
    atmosphere: dict
        Dictionary containing the model columns needed for the dispersion curves.

    Author(s)
    ---------
    Julio M. Morales, March 20th, 2026
    '''

    # Build the requested result from the provided inputs.
    model_file = resolve_config_file().parent / 'Oana_codes' / 'CSM_A.dat'
    model_data = np.loadtxt(model_file, dtype = np.float64)

    return {
        'height_Mm': model_data[:, 0],
        'gravity_cgs': model_data[:, 1],
        'dP_dz': model_data[:, 2],
        'density_cgs': model_data[:, 3],
        'dRho_dz': model_data[:, 4],
        'cs_cgs': model_data[:, 5]}


def compute_komega_dispersion_curves(k_axis):

    '''
    Purpose
    -------
    Compute the dispersion curves overlaid on the standalone and composite k-omega plots.

    Inputs
    ------
    k_axis: np.array, float
        Positive horizontal-wavenumber axis in 1/Mm.

    Outputs
    -------
    curves: dict
        Dictionary containing the model dispersion curves in mHz.

    Author(s)
    ---------
    Julio M. Morales, March 20th, 2026
    '''

    # Build the requested result from the provided inputs.
    atmosphere = load_reference_atmosphere_model()
    height_index = 1005
    gravity_cgs = 27400.0
    gravity_km = 0.274
    cs_cgs = atmosphere['cs_cgs']
    density_cgs = atmosphere['density_cgs']
    dRho_dz = atmosphere['dRho_dz']
    dP_dz = atmosphere['dP_dz']
    cs_km = cs_cgs*1.0e-5
    kx_km = np.asarray(k_axis, dtype = np.float64)/1000.0

    surface_fmode = np.sqrt(gravity_km*kx_km)*1000.0/(2.0*np.pi)
    lamb_curve = cs_km[height_index]*kx_km*1000.0/(2.0*np.pi)

    density_scale_height_value = -1.0*(dRho_dz[height_index]/density_cgs[height_index])
    density_scale_height_value = density_scale_height_value**(-1)
    acoustic_cutoff_frequency_squared = (cs_cgs[height_index]**2)/(4.0*density_scale_height_value**2)
    brunt_vaisala_squared = gravity_cgs*(
        dP_dz[height_index]/((cs_cgs[height_index]**2)*density_cgs[height_index])
        - dRho_dz[height_index]/density_cgs[height_index]
    )

    quadratic_b = -(acoustic_cutoff_frequency_squared + cs_km[height_index]**2*kx_km**2)
    quadratic_c = brunt_vaisala_squared*cs_km[height_index]**2*kx_km**2
    discriminant = np.maximum(quadratic_b**2 - 4.0*quadratic_c, 0.0)
    qval1 = (-quadratic_b + np.sqrt(discriminant))/2.0
    qval2 = (-quadratic_b - np.sqrt(discriminant))/2.0
    regime_1 = np.sqrt(np.maximum(qval1, 0.0))*1000.0/(2.0*np.pi)
    regime_2 = np.sqrt(np.maximum(qval2, 0.0))*1000.0/(2.0*np.pi)

    return {
        'surface_fmode': surface_fmode,
        'lamb_curve': lamb_curve,
        'regime_1': regime_1,
        'regime_2': regime_2}


def add_komega_dispersion_overlays(ax, k_axis):

    '''
    Purpose
    -------
    Overlay the reference dispersion curves on a k-omega axis.

    Inputs
    ------
    ax: matplotlib.axes.Axes
        Axis that will receive the overlays.

    k_axis: np.array, float
        Positive horizontal-wavenumber axis in 1/Mm.

    Outputs
    -------
    None

    Author(s)
    ---------
    Julio M. Morales, March 20th, 2026
    '''

    # Build the requested result from the provided inputs.
    curves = compute_komega_dispersion_curves(k_axis)
    ax.plot(k_axis, curves['regime_1'], color = 'k', linewidth = 2.0)
    ax.plot(k_axis, curves['regime_2'], color = 'k', linewidth = 2.0)
    ax.plot(k_axis, curves['surface_fmode'], color = 'gray', linewidth = 2.0)
    ax.plot(k_axis, curves['lamb_curve'], color = 'gray', linewidth = 2.0, linestyle = '--')


def get_axis_limits(axis, fallback_step):

    '''
    Purpose
    -------
    Compute the outer pixel-edge limits for a plotted axis.

    Inputs
    ------
    axis: np.array, float
        Axis-center values.

    fallback_step: float
        Step size used when the axis has only one element.

    Outputs
    -------
    limits: tuple
        Minimum and maximum axis limits at the outer pixel edges.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Expand the axis limits to the outer pixel edges
    if axis.size > 1:
        step = float(np.median(np.diff(axis)))
    else:
        step = float(fallback_step)

    return axis[0] - 0.5*step, axis[-1] + 0.5*step


def get_resolved_delta_z_km(config):

    '''
    Purpose
    -------
    Get resolved delta z km.

    Inputs
    ------
    config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    delta_z_km = config.get('data', {}).get('resolved_delta_z_km', None)

    if delta_z_km in ['', None]:
        return None

    return float(delta_z_km)


def radius_Mm_to_angle_deg(radius_Mm, delta_z_km):

    '''
    Purpose
    -------
    Radius Mm to angle deg.

    Inputs
    ------
    radius_Mm: object
        Input parameter.
    delta_z_km: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    radius_km = np.asarray(radius_Mm, dtype = np.float64)*1000.0
    delta_z_km = float(delta_z_km)
    angle_deg = np.empty_like(radius_km, dtype = np.float64)

    zero_mask = np.isclose(radius_km, 0.0)
    nonzero_mask = ~zero_mask
    angle_deg[zero_mask] = 90.0 if delta_z_km > 0.0 else 0.0
    angle_deg[nonzero_mask] = np.degrees(np.arctan(delta_z_km/radius_km[nonzero_mask]))

    return angle_deg


def format_angle_tick_label(angle_deg):

    '''
    Purpose
    -------
    Format angle tick label.

    Inputs
    ------
    angle_deg: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    if np.isclose(angle_deg, round(angle_deg), atol = 0.05):
        return f'{int(round(angle_deg))}'

    return f'{angle_deg:.1f}'


def attach_propagation_angle_axis(ax, config, label = r'$\theta$ [deg]'):

    '''
    Purpose
    -------
    Attach propagation angle axis.

    Inputs
    ------
    ax: object
        Input parameter.
    config: object
        Input parameter.
    label: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    delta_z_km = get_resolved_delta_z_km(config)
    if delta_z_km is None:
        return None

    top_ax = ax.twiny()
    top_ax.set_xlim(*ax.get_xlim())
    bottom_ticks = np.asarray(ax.get_xticks(), dtype = np.float64)
    x0, x1 = ax.get_xlim()
    xmin, xmax = min(x0, x1), max(x0, x1)
    tick_mask = np.isfinite(bottom_ticks) & (bottom_ticks >= xmin - 1.0e-9) & (bottom_ticks <= xmax + 1.0e-9)
    tick_positions = bottom_ticks[tick_mask]

    if tick_positions.size == 0:
        tick_positions = np.asarray([xmin, xmax], dtype = np.float64)

    angle_labels = [format_angle_tick_label(angle) for angle in radius_Mm_to_angle_deg(tick_positions, delta_z_km)]
    top_ax.set_xticks(tick_positions)
    top_ax.set_xticklabels(angle_labels)
    top_ax.set_xlabel(label)
    top_ax.patch.set_alpha(0.0)
    top_ax.xaxis.set_ticks_position('top')
    top_ax.xaxis.set_label_position('top')
    top_ax.tick_params(axis = 'x', which = 'both', direction = 'out')

    return top_ax


def propagation_angle_to_radius_Mm(angle_deg, delta_z_km):

    '''
    Purpose
    -------
    Propagation angle to radius Mm.

    Inputs
    ------
    angle_deg: object
        Input parameter.
    delta_z_km: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    angle_deg = float(angle_deg)
    delta_z_km = float(delta_z_km)

    if delta_z_km <= 0.0:
        return None

    angle_rad = np.radians(angle_deg)
    tangent = np.tan(angle_rad)

    if not np.isfinite(tangent) or np.isclose(tangent, 0.0):
        return None

    return delta_z_km/(1000.0*tangent)


def add_reference_angle_line(ax, config, angle_deg = 15.0, color = 'k', linewidth = 1.5):

    '''
    Purpose
    -------
    Add reference angle line.

    Inputs
    ------
    ax: object
        Input parameter.
    config: object
        Input parameter.
    angle_deg: object
        Input parameter.
    color: object
        Input parameter.
    linewidth: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    delta_z_km = get_resolved_delta_z_km(config)
    if delta_z_km is None:
        return None

    radius_Mm = propagation_angle_to_radius_Mm(angle_deg, delta_z_km)
    if radius_Mm is None or not np.isfinite(radius_Mm):
        return None

    x0, x1 = ax.get_xlim()
    xmin, xmax = min(x0, x1), max(x0, x1)
    if radius_Mm < xmin - 1.0e-9 or radius_Mm > xmax + 1.0e-9:
        return None

    return ax.axvline(radius_Mm, color = color, linewidth = linewidth)


def maybe_save_figure(fig, output_file, default_name = None, figure_dir = None):

    '''
    Purpose
    -------
    Save a matplotlib figure to disk as a JPEG in the shared figures folder.

    Inputs
    ------
    fig: matplotlib.figure.Figure
        Figure that may be written to disk.

    output_file: str or pathlib.Path
        Requested output file name. The final still figure is always written as a JPEG.

    default_name: str, optional
        Default JPEG file name used when output_file is empty.

    figure_dir: str or pathlib.Path, optional
        Folder where the JPEG file should be written.

    Outputs
    -------
    saved_file: pathlib.Path or None
        Saved file path when a file is written.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Save all still figures as JPEG files in the shared figures folder
    if figure_dir in ['', None]:
        return None

    figure_dir = Path(figure_dir).expanduser().resolve()

    if output_file in ['', None]:
        if default_name is None:
            return None
        output_file = Path(default_name)
    else:
        output_file = Path(output_file)

    output_file = figure_dir / output_file.name
    output_file = output_file.with_suffix('.jpeg')
    output_file.parent.mkdir(parents = True, exist_ok = True)
    fig.savefig(output_file, format = 'jpeg')

    return output_file


def resolve_animation_output_file(output_file, data_file, suffix, animation_dir = None):

    '''
    Purpose
    -------
    Resolve the animation output path from a requested or default file name.

    Inputs
    ------
    output_file: str or pathlib.Path
        Requested output file path.

    data_file: pathlib.Path
        Source FITS file path.

    suffix: str
        Suffix appended to the data-file stem when no output path is provided.

    Outputs
    -------
    resolved_output: pathlib.Path
        Final MP4 output path.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Use the requested output path or fall back to the animation folder
    if output_file in ['', None]:
        if animation_dir in ['', None]:
            output_file = data_file.with_name(f'{data_file.stem}_{suffix}.mp4')
        else:
            output_file = Path(animation_dir).expanduser().resolve() / f'{data_file.stem}_{suffix}.mp4'
    else:
        output_file = Path(output_file).expanduser().resolve()

    output_file.parent.mkdir(parents = True, exist_ok = True)

    return output_file


def load_raw_dopplergrams(config, AGWFilter, plot_cache):

    '''
    Purpose
    -------
    Load and cache the unfiltered Dopplergram cubes.

    Inputs
    ------
    config: dict
        Shared configuration dictionary.

    AGWFilter: class
        Filtering class used by the time-distance pipeline.

    plot_cache: dict
        In-memory cache for the plotting notebook.

    Outputs
    -------
    v1: np.array, float
        First Dopplergram cube in [t, y, x] order.

    v2: np.array, float
        Second Dopplergram cube in [t, y, x] order.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Reuse the raw cubes when multiple plots need them
    if 'raw_dopplergrams' not in plot_cache:
        plot_cache['raw_dopplergrams'] = AGWFilter(config).load_dopplergrams()

    return plot_cache['raw_dopplergrams']


def load_filtered_dopplergrams(config, AGWFilter, plot_cache):

    '''
    Purpose
    -------
    Load and cache the Dopplergram cubes after the configured filtering sequence.

    Inputs
    ------
    config: dict
        Shared configuration dictionary.

    AGWFilter: class
        Filtering class used by the time-distance pipeline.

    plot_cache: dict
        In-memory cache for the plotting notebook.

    Outputs
    -------
    v1_filt: np.array, float
        Filtered first Dopplergram cube in [t, y, x] order.

    v2_filt: np.array, float
        Filtered second Dopplergram cube in [t, y, x] order.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Reuse the filtered cubes when multiple plots need them
    if 'filtered_dopplergrams' not in plot_cache:
        plot_cache['filtered_dopplergrams'] = AGWFilter(config).run()

    return plot_cache['filtered_dopplergrams']



def convert_velocity_to_km_s(velocity_data, config):

    '''
    Purpose
    -------
    Convert Dopplergram values to km/s before plotting.

    Inputs
    ------
    velocity_data: np.array, float
        Dopplergram array in the native source units.

    config: dict
        Shared configuration dictionary.

    Outputs
    -------
    velocity_km_s: np.array, float
        Dopplergram array converted to km/s for plotting.

    Author(s)
    ---------
    Julio M. Morales, March 19th, 2026
    '''

    # Build the requested result from the provided inputs.
    velocity_km_s = np.asarray(velocity_data, dtype = np.float64)

    if config.get('data', {}).get('source_type', 'paired_cubes') == 'single_cube':
        velocity_km_s = velocity_km_s/1.0e5

    return velocity_km_s


def load_magnetogram_removed_masks(config, AGWFilter, plot_cache):

    '''
    Purpose
    -------
    Load and cache the boolean masks used by the magnetogram filter.

    Inputs
    ------
    config: dict
        Shared configuration dictionary.

    AGWFilter: class
        Filtering class used by the time-distance pipeline.

    plot_cache: dict
        In-memory cache for the plotting notebook.

    Outputs
    -------
    removed_mask_v1: np.array, bool or None
        Boolean mask marking the first Dopplergram pixels removed by the magnetic filter.

    removed_mask_v2: np.array, bool or None
        Boolean mask marking the second Dopplergram pixels removed by the magnetic filter.

    mask_metadata: dict
        Dictionary containing the masking selection, threshold, and magnetogram arrays.

    Author(s)
    ---------
    Julio M. Morales, March 18th, 2026
    '''

    # Build the requested result from the provided inputs.
    filtering = config.get('filtering', {})
    magnetogram_config = filtering.get('magnetogram', {})

    if not filtering.get('enabled', False):
        return None, None, {}

    if 'magnetogram' not in filtering.get('filter_sequence', []):
        return None, None, {}

    if not magnetogram_config.get('enabled', False):
        return None, None, {}

    if 'magnetogram_removed_masks' not in plot_cache:
        plot_cache['magnetogram_removed_masks'] = AGWFilter(config).load_magnetogram_filter_masks()

    return plot_cache['magnetogram_removed_masks']



def build_request_title(request, config, naming_module, dataset = 'pair', time_index = None):

    '''
    Purpose
    -------
    Build request title.

    Inputs
    ------
    request: object
        Input parameter.
    config: object
        Input parameter.
    naming_module: object
        Input parameter.
    dataset: object
        Input parameter.
    time_index: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    plot_type = request.get('plot_type', '')
    product_lookup = {
        'time_distance': 'time_distance',
        'phase_difference': 'phase_difference',
        'komega_diagram': 'komega_diagram',
        'correlation_radius_animation': 'time_distance',
        'correlation_vertical_animation': 'time_distance',
        'phase_radius_animation': 'phase_difference',
        'phase_vertical_animation': 'phase_difference',
        'gaussian_filter': 'gaussian_filter',
        'filtered_dopplergram': 'filtered_dopplergram',
        'composite_diagnostics': 'composite_diagnostics'}
    product = product_lookup.get(plot_type, '')

    if product == '':
        if request.get('axis_type', '') == 'frequency':
            product = 'phase_difference'
        else:
            product = 'time_distance'

    return build_plot_title(config, product, dataset = dataset, time_index = time_index)


def build_animation_suptitle(base_title, frame_label, frame_value, frame_unit):

    '''
    Purpose
    -------
    Build the single figure-level title for an animation frame.

    Inputs
    ------
    base_title: str
        Standardized plot title for the underlying data product.

    frame_label: str
        Name of the animated coordinate.

    frame_value: float
        Current coordinate value displayed by the animation.

    frame_unit: str
        Unit label for the animated coordinate.

    Outputs
    -------
    suptitle: str
        Figure-level title string for the current animation frame.

    Author(s)
    ---------
    Julio M. Morales, March 18th, 2026
    '''

    # Build the requested result from the provided inputs.
    unit = f' {frame_unit}' if str(frame_unit).strip() != '' else ''

    return f'{base_title} | {str(frame_label).lower()} = {frame_value:.2f}{unit}'



def set_centered_figure_title(fig, title, y = 0.98):

    '''
    Purpose
    -------
    Apply a single figure-level title centered on the full canvas.

    Inputs
    ------
    fig: matplotlib.figure.Figure
        Figure that will receive the shared title.

    title: str
        Text string that should be centered on the full figure width.

    y: float, default = 0.98
        Vertical location of the title in figure coordinates.

    Outputs
    -------
    suptitle: matplotlib.text.Text
        Figure-level title artist.

    Author(s)
    ---------
    Julio M. Morales, March 19th, 2026
    '''

    # Build the requested result from the provided inputs.
    if getattr(fig, '_suptitle', None) is not None:
        fig._suptitle.remove()
        fig._suptitle = None

    for ax in fig.axes:
        ax.set_title('')

    return fig.suptitle(title, x = 0.5, y = y, ha = 'center')



def resolve_phase_cmap(cmap_name):

    '''
    Purpose
    -------
    Reverse the phase colormap so negative values are red and positive values are blue.

    Inputs
    ------
    cmap_name: str
        Requested matplotlib colormap name.

    Outputs
    -------
    cmap_name: str
        Colormap name with the reversed sign convention.

    Author(s)
    ---------
    Julio M. Morales, March 19th, 2026
    '''

    # Build the requested result from the provided inputs.
    cmap_name = str(cmap_name).strip()

    if cmap_name == '':
        cmap_name = 'seismic'

    return cmap_name if cmap_name.endswith('_r') else f'{cmap_name}_r'



# Match the portrait panel geometry used in the publication-style reference plot.
TARGET_PANEL_BOX_ASPECT = 1.55



def make_centered_single_panel_figure(figsize, panel_bottom = 0.14, panel_top = 0.84, colorbar_width = 0.022, colorbar_gap = 0.012):

    '''
    Purpose
    -------
    Create a centered single-panel figure whose colorbar is attached to the real plot box.

    Inputs
    ------
    figsize: tuple
        Requested matplotlib figure size.

    panel_bottom: float, default = 0.14
        Lower edge of the plotting box in figure coordinates.

    panel_top: float, default = 0.84
        Upper edge of the plotting box in figure coordinates.

    colorbar_width: float, default = 0.022
        Width of the dedicated colorbar axis in figure coordinates.

    colorbar_gap: float, default = 0.012
        Horizontal gap between the plot and the colorbar axes in figure coordinates.

    Outputs
    -------
    fig: matplotlib.figure.Figure
        Figure object containing the centered layout.

    ax: matplotlib.axes.Axes
        Main plotting axis.

    cax: matplotlib.axes.Axes
        Dedicated colorbar axis.

    Author(s)
    ---------
    Julio M. Morales, March 19th, 2026
    '''

    # Build the requested result from the provided inputs.
    fig = plt.figure(figsize = tuple(figsize))
    fig_width, fig_height = fig.get_size_inches()
    panel_height = panel_top - panel_bottom
    panel_width = panel_height*fig_height/(fig_width*TARGET_PANEL_BOX_ASPECT)
    total_width = panel_width + colorbar_gap + colorbar_width
    left = 0.5 - 0.5*total_width
    ax = fig.add_axes([left, panel_bottom, panel_width, panel_height])
    cax = fig.add_axes([left + panel_width + colorbar_gap, panel_bottom, colorbar_width, panel_height])
    ax.set_box_aspect(TARGET_PANEL_BOX_ASPECT)

    return fig, ax, cax



def make_centered_map_panel_figure(figsize, panel_bottom = 0.14, panel_top = 0.84, colorbar_width = 0.022, colorbar_gap = 0.012):

    '''
    Purpose
    -------
    Create a centered rectangular single-panel figure with an attached colorbar axis.

    Inputs
    ------
    figsize: tuple
        Requested matplotlib figure size.

    panel_bottom: float, default = 0.14
        Lower edge of the main plotting axis in figure coordinates.

    panel_top: float, default = 0.84
        Upper edge of the main plotting axis in figure coordinates.

    colorbar_width: float, default = 0.022
        Width of the dedicated colorbar axis in figure coordinates.

    colorbar_gap: float, default = 0.012
        Horizontal gap between the plot and the colorbar axes in figure coordinates.

    Outputs
    -------
    fig: matplotlib.figure.Figure
        Figure object containing the centered layout.

    ax: matplotlib.axes.Axes
        Main plotting axis.

    cax: matplotlib.axes.Axes
        Dedicated colorbar axis.
    '''

    # Build the requested result from the provided inputs.
    fig = plt.figure(figsize = tuple(figsize))
    panel_height = panel_top - panel_bottom
    fig_width, fig_height = fig.get_size_inches()
    panel_width = panel_height*fig_height/(fig_width*TARGET_PANEL_BOX_ASPECT)
    total_width = panel_width + colorbar_gap + colorbar_width
    left = 0.5 - 0.5*total_width
    ax = fig.add_axes([left, panel_bottom, panel_width, panel_height])
    cax = fig.add_axes([left + panel_width + colorbar_gap, panel_bottom, colorbar_width, panel_height])
    ax.set_box_aspect(TARGET_PANEL_BOX_ASPECT)

    return fig, ax, cax



def plot_saved_map(request, config, naming_module, figure_dir = None):

    '''
    Purpose
    -------
    Plot a saved time-distance or phase-difference FITS file.

    Inputs
    ------
    request: dict
        Dictionary containing the data path and plot parameters.

    config: dict
        Shared configuration dictionary.

    naming_module: module
        Naming helper module used by the plotting notebook.

    Outputs
    -------
    result: dict
        Dictionary containing the plotted figure, axes, and optional saved file path.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Build the requested result from the provided inputs.
    dt = config['time_distance']['dt']
    dx_Mm = config['time_distance']['p_dx_Mm']

    # Load the saved map and reconstruct the plotting axes
    data = load_time_distance_data(request['data_file'], dt, dx_Mm, axis_type = request['axis_type'])
    radius_limits = get_axis_limits(data['radii'], dx_Mm)
    vertical_step = float(dt)/60.0 if request['axis_type'] == 'time_lag' else 1.0e3/(data['values'].shape[1]*float(dt))
    vertical_limits = get_axis_limits(data['vertical_axis'], vertical_step)

    display_values = np.asarray(data['values'], dtype = np.float64)
    if request.get('normalize_display', False):
        display_values = normalize_cross_correlation_display(display_values)

    # Plot the saved 2D map
    fig, ax, cax = make_centered_map_panel_figure(request.get('figsize', [10, 6]))
    mesh = ax.pcolormesh(
        data['radii'],
        data['vertical_axis'],
        display_values.T,
        cmap = request.get('cmap', 'viridis'),
        vmin = request.get('vmin', None),
        vmax = request.get('vmax', None),
        shading = 'auto')

    xlim = request.get('xlim', None)
    ylim = request.get('ylim', None)
    ax.set_xlim(*radius_limits if xlim in ['', None] else xlim)
    ax.set_ylim(*vertical_limits if ylim in ['', None] else ylim)
    ax.set_xlabel('Annulus Radius [Mm]')
    ax.set_ylabel(data['vertical_label'])
    add_reference_angle_line(ax, config)
    attach_propagation_angle_axis(ax, config)
    set_centered_figure_title(fig, build_request_title(request, config, naming_module), y = 0.98)
    fig.colorbar(mesh, cax = cax, label = request.get('value_label', 'Value'))

    # Save and display the figure when requested
    saved_file = maybe_save_figure(
        fig,
        request.get('output_file', ''),
        default_name = f"{data['data_file'].stem}.jpeg",
        figure_dir = figure_dir)
    plt.show()

    return {'figure': fig, 'axes': ax, 'saved_file': saved_file, 'data_file': data['data_file']}



def save_radius_animation(request, config, naming_module, animation_dir = None):

    '''
    Purpose
    -------
    Save the radius-scan animation for a saved FITS file.

    Inputs
    ------
    request: dict
        Dictionary containing the data path and animation parameters.

    config: dict
        Shared configuration dictionary.

    naming_module: module
        Naming helper module used by the plotting notebook.

    Outputs
    -------
    output_file: pathlib.Path
        Saved MP4 file path.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Build the requested result from the provided inputs.
    dt = config['time_distance']['dt']
    dx_Mm = config['time_distance']['p_dx_Mm']

    # Load the saved map and set up the animation axes
    data = load_time_distance_data(request['data_file'], dt, dx_Mm, axis_type = request['axis_type'])
    raw_values = np.asarray(data['values'], dtype = np.float64)
    line_values = np.array(raw_values, copy = True)
    map_values = np.array(raw_values, copy = True)
    if request.get('normalize_line_display', request.get('normalize_display', False)):
        line_values = normalize_cross_correlation_display(line_values)
    if request.get('normalize_map_display', request.get('normalize_display', False)):
        map_values = normalize_cross_correlation_display(map_values)
    radii = data['radii']
    vertical_axis = data['vertical_axis']
    output_file = resolve_animation_output_file(
        request.get('output_file', ''),
        data['data_file'],
        'radius_animation',
        animation_dir = animation_dir)

    ymin = 1.05*np.min(line_values)
    ymax = 1.05*np.max(line_values)
    radius_limits = get_axis_limits(radii, dx_Mm)
    vertical_step = float(dt)/60.0 if request['axis_type'] == 'time_lag' else 1.0e3/(raw_values.shape[1]*float(dt))
    vertical_limits = get_axis_limits(vertical_axis, vertical_step)
    request_ylim = request.get('ylim', None)
    fig, axes = plt.subplots(1, 2, figsize = tuple(request.get('figsize', [14, 5])), gridspec_kw = {'width_ratios': [1.0, 1.25]})
    ax_line, ax_map = axes
    fig.subplots_adjust(left = 0.08, right = 0.90, bottom = 0.14, top = 0.84, wspace = 0.24)
    cax = ax_map.inset_axes([1.03, 0.0, 0.035, 1.0], transform = ax_map.transAxes)

    base_title = build_request_title(request, config, naming_module)
    line, = ax_line.plot(vertical_axis, line_values[0, :], color = 'k')
    ax_line.axhline(0.0, color = '0.6', linestyle = '--', linewidth = 1)
    ax_line.set_xlim(*(vertical_limits if request_ylim in ['', None] else request_ylim))
    ax_line.set_ylim(ymin, ymax)
    ax_line.set_xlabel(data['vertical_label'])
    ax_line.set_ylabel(request.get('line_value_label', request.get('value_label', 'Value')))

    mesh = ax_map.pcolormesh(
        radii,
        vertical_axis,
        map_values.T,
        cmap = request.get('cmap', 'viridis'),
        vmin = request.get('vmin', None),
        vmax = request.get('vmax', None),
        shading = 'auto')
    marker = ax_map.axvline(radii[0], color = 'red', linewidth = 2)
    ax_map.set_xlim(*radius_limits)
    ax_map.set_ylim(*(vertical_limits if request_ylim in ['', None] else request_ylim))
    ax_map.set_xlabel('Annulus Radius [Mm]')
    ax_map.set_ylabel(data['vertical_label'])
    add_reference_angle_line(ax_map, config)
    attach_propagation_angle_axis(ax_map, config)
    suptitle = set_centered_figure_title(fig, build_animation_suptitle(base_title, 'Radius', radii[0], 'Mm'), y = 0.98)
    fig.colorbar(mesh, cax = cax, label = request.get('colorbar_value_label', request.get('value_label', 'Value')))

    def update(radius_index):

        # Update the 1D slice and the synchronized radius marker
        line.set_ydata(line_values[radius_index, :])
        suptitle.set_text(build_animation_suptitle(base_title, 'Radius', radii[radius_index], 'Mm'))
        marker.set_xdata([radii[radius_index], radii[radius_index]])

        return line, marker, suptitle

    # Save the animation to disk
    interval_ms = float(request.get('interval_ms', 800))
    anim = FuncAnimation(fig, update, frames = np.arange(line_values.shape[0], dtype = int), interval = interval_ms, blit = False, repeat = True)
    writer = FFMpegWriter(fps = 1000.0/interval_ms)
    anim.save(output_file, writer = writer)
    plt.close(fig)

    return output_file



def save_vertical_animation(request, config, naming_module, animation_dir = None):

    '''
    Purpose
    -------
    Save the second-axis-scan animation for a saved FITS file.

    Inputs
    ------
    request: dict
        Dictionary containing the data path and animation parameters.

    config: dict
        Shared configuration dictionary.

    naming_module: module
        Naming helper module used by the plotting notebook.

    Outputs
    -------
    output_file: pathlib.Path
        Saved MP4 file path.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Build the requested result from the provided inputs.
    dt = config['time_distance']['dt']
    dx_Mm = config['time_distance']['p_dx_Mm']

    # Load the saved map and choose the animation frame order
    data = load_time_distance_data(request['data_file'], dt, dx_Mm, axis_type = request['axis_type'])
    raw_values = np.asarray(data['values'], dtype = np.float64)
    line_values = np.array(raw_values, copy = True)
    map_values = np.array(raw_values, copy = True)
    if request.get('normalize_line_display', request.get('normalize_display', False)):
        line_values = normalize_cross_correlation_display(line_values)
    if request.get('normalize_map_display', request.get('normalize_display', False)):
        map_values = normalize_cross_correlation_display(map_values)
    radii = data['radii']
    vertical_axis = data['vertical_axis']
    output_file = resolve_animation_output_file(
        request.get('output_file', ''),
        data['data_file'],
        'time_lag_animation' if request['axis_type'] == 'time_lag' else 'frequency_animation',
        animation_dir = animation_dir)

    if request['axis_type'] == 'time_lag':
        frame_indices = np.arange(raw_values.shape[1], dtype = int)
    else:
        zero_index = raw_values.shape[1]//2
        frame_indices = np.concatenate([
            np.arange(0, zero_index + 1, dtype = int),
            np.arange(zero_index + 1, raw_values.shape[1], dtype = int)])

    frame_index0 = int(frame_indices[0])
    ymin = 1.05*np.min(line_values)
    ymax = 1.05*np.max(line_values)
    radius_limits = get_axis_limits(radii, dx_Mm)
    vertical_step = float(dt)/60.0 if request['axis_type'] == 'time_lag' else 1.0e3/(raw_values.shape[1]*float(dt))
    vertical_limits = get_axis_limits(vertical_axis, vertical_step)
    request_ylim = request.get('ylim', None)

    fig, axes = plt.subplots(1, 2, figsize = tuple(request.get('figsize', [14, 5])), gridspec_kw = {'width_ratios': [1.0, 1.25]})
    ax_line, ax_map = axes
    fig.subplots_adjust(left = 0.08, right = 0.90, bottom = 0.14, top = 0.84, wspace = 0.24)
    cax = ax_map.inset_axes([1.03, 0.0, 0.035, 1.0], transform = ax_map.transAxes)

    base_title = build_request_title(request, config, naming_module)
    line, = ax_line.plot(radii, line_values[:, frame_index0], color = 'k')
    ax_line.axhline(0.0, color = '0.6', linestyle = '--', linewidth = 1)
    ax_line.set_xlim(radii.min(), radii.max())
    ax_line.set_ylim(ymin, ymax)
    ax_line.set_xlabel('Radius [Mm]')
    ax_line.set_ylabel(request.get('line_value_label', request.get('value_label', 'Value')))
    add_reference_angle_line(ax_line, config)
    attach_propagation_angle_axis(ax_line, config)

    mesh = ax_map.pcolormesh(radii, vertical_axis, map_values.T, cmap = request.get('cmap', 'viridis'), shading = 'auto')
    marker = ax_map.axhline(vertical_axis[frame_index0], color = 'red', linewidth = 2)
    ax_map.set_xlim(*radius_limits)
    ax_map.set_ylim(*(vertical_limits if request_ylim in ['', None] else request_ylim))
    ax_map.set_xlabel('Annulus Radius [Mm]')
    ax_map.set_ylabel(data['vertical_label'])
    add_reference_angle_line(ax_map, config)
    attach_propagation_angle_axis(ax_map, config)
    suptitle = set_centered_figure_title(
        fig,
        build_animation_suptitle(base_title, data['frame_label'], vertical_axis[frame_index0], data['frame_unit']),
        y = 0.98)
    fig.colorbar(mesh, cax = cax, label = request.get('colorbar_value_label', request.get('value_label', 'Value')))

    def update(frame_index):

        # Update the 1D slice and the synchronized second-axis marker
        line.set_ydata(line_values[:, frame_index])
        suptitle.set_text(
            build_animation_suptitle(base_title, data['frame_label'], vertical_axis[frame_index], data['frame_unit']))
        marker.set_ydata([vertical_axis[frame_index], vertical_axis[frame_index]])

        return line, marker, suptitle

    # Save the animation to disk
    interval_ms = float(request.get('interval_ms', 60))
    anim = FuncAnimation(fig, update, frames = frame_indices, interval = interval_ms, blit = False, repeat = True)
    writer = FFMpegWriter(fps = 1000.0/interval_ms)
    anim.save(output_file, writer = writer)
    plt.close(fig)

    return output_file


def make_time_distance_plot(request, config, naming_module):

    '''
    Purpose
    -------
    Make the saved time-distance diagram.

    Inputs
    ------
    request: dict
        Dictionary containing the plot parameters.

    config: dict
        Shared configuration dictionary.

    Outputs
    -------
    result: dict
        Dictionary containing the plotted figure, axes, and optional saved file path.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Plot the saved cross-correlation map
    request = copy.deepcopy(request)
    request['ylim'] = [-60.0, 60.0]

    return plot_saved_map(
        request,
        config,
        naming_module,
        figure_dir = get_figure_dir(config))


def make_phase_difference_plot(request, config, naming_module):

    '''
    Purpose
    -------
    Make the saved phase-difference diagram.

    Inputs
    ------
    request: dict
        Dictionary containing the plot parameters.

    config: dict
        Shared configuration dictionary.

    Outputs
    -------
    result: dict
        Dictionary containing the plotted figure, axes, and optional saved file path.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Plot the saved phase-difference map
    request = copy.deepcopy(request)
    request['cmap'] = resolve_phase_cmap(request.get('cmap', 'seismic'))

    return plot_saved_map(
        request,
        config,
        naming_module,
        figure_dir = get_figure_dir(config))


def make_komega_plot(request, config, naming_module):

    '''
    Purpose
    -------
    Plot the saved k-omega phase-difference FITS file.

    Inputs
    ------
    request: dict
        Dictionary containing the plot parameters.

    config: dict
        Shared configuration dictionary.

    Outputs
    -------
    result: dict
        Dictionary containing the plotted figure, axes, and optional saved file path.

    Author(s)
    ---------
    Julio M. Morales, March 20th, 2026
    '''

    # Build the requested result from the provided inputs.
    request = copy.deepcopy(request)
    data = load_komega_data(request['data_file'], config)
    k_limits = get_axis_limits(data['k_axis'], np.median(np.diff(data['k_axis'])) if data['k_axis'].size > 1 else 1.0)
    nu_limits = get_axis_limits(data['nu_axis'], np.median(np.diff(data['nu_axis'])) if data['nu_axis'].size > 1 else 1.0)

    fig, ax, cax = make_centered_single_panel_figure(request.get('figsize', [10, 6]))
    mesh = ax.pcolormesh(
        data['k_axis'],
        data['nu_axis'],
        data['values'].T,
        cmap = request.get('cmap', cmr.fusion),
        norm = TwoSlopeNorm(
            vmin = request.get('vmin', -20.0),
            vcenter = 0.0,
            vmax = request.get('vmax', 80.0)),
        shading = 'auto')

    if request.get('overlay_curves', True):
        add_komega_dispersion_overlays(ax, data['k_axis'])

    xlim = request.get('xlim', None)
    ylim = request.get('ylim', None)
    ax.set_xlim(*(k_limits if xlim in ['', None] else xlim))
    ax.set_ylim(*(nu_limits if ylim in ['', None] else ylim))
    ax.set_xlabel(data['horizontal_label'])
    ax.set_ylabel(data['vertical_label'])
    ax.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
    base_title = build_request_title(request, config, naming_module)
    orientation_title = format_komega_orientation_title(data, config)
    if orientation_title != '':
        base_title = f'{base_title}\n{orientation_title}'
    set_centered_figure_title(fig, base_title, y = 0.98)
    fig.colorbar(mesh, cax = cax, label = request.get('value_label', 'Value'))
    saved_file = maybe_save_figure(
        fig,
        request.get('output_file', ''),
        default_name = f"{build_output_stem(config, 'komega_diagram')}.jpeg",
        figure_dir = get_figure_dir(config))
    plt.show()

    return {
        'figure': fig,
        'axes': ax,
        'saved_file': saved_file,
        'data_file': data['data_file']}


def make_correlation_radius_animation(request, config, naming_module):

    '''
    Purpose
    -------
    Save the radius animation for the cross-correlation map.

    Inputs
    ------
    request: dict
        Dictionary containing the animation parameters.

    config: dict
        Shared configuration dictionary.

    Outputs
    -------
    output_file: pathlib.Path
        Saved MP4 file path.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Save the radius animation for the correlation map
    request = copy.deepcopy(request)
    request['ylim'] = [-60.0, 60.0]
    if request.get('data_file', '') in ['', None]:
        request['data_file'] = config['data']['outfile']
    return save_radius_animation(
        request,
        config,
        naming_module,
        animation_dir = get_animation_dir(config))


def make_correlation_vertical_animation(request, config, naming_module):

    '''
    Purpose
    -------
    Save the time-lag animation for the cross-correlation map.

    Inputs
    ------
    request: dict
        Dictionary containing the animation parameters.

    config: dict
        Shared configuration dictionary.

    Outputs
    -------
    output_file: pathlib.Path
        Saved MP4 file path.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Save the second-axis animation for the correlation map
    request = copy.deepcopy(request)
    request['ylim'] = [-60.0, 60.0]
    if request.get('data_file', '') in ['', None]:
        request['data_file'] = config['data']['outfile']
    return save_vertical_animation(
        request,
        config,
        naming_module,
        animation_dir = get_animation_dir(config))


def make_phase_radius_animation(request, config, naming_module):

    '''
    Purpose
    -------
    Save the radius animation for the phase-difference map.

    Inputs
    ------
    request: dict
        Dictionary containing the animation parameters.

    config: dict
        Shared configuration dictionary.

    Outputs
    -------
    output_file: pathlib.Path
        Saved MP4 file path.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Save the radius animation for the phase map
    request = copy.deepcopy(request)
    request['cmap'] = resolve_phase_cmap(request.get('cmap', 'seismic'))
    if request.get('data_file', '') in ['', None]:
        request['data_file'] = config['data']['phase_outfile']
    return save_radius_animation(
        request,
        config,
        naming_module,
        animation_dir = get_animation_dir(config))


def make_phase_vertical_animation(request, config, naming_module):

    '''
    Purpose
    -------
    Save the frequency animation for the phase-difference map.

    Inputs
    ------
    request: dict
        Dictionary containing the animation parameters.

    config: dict
        Shared configuration dictionary.

    Outputs
    -------
    output_file: pathlib.Path
        Saved MP4 file path.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Save the second-axis animation for the phase map
    request = copy.deepcopy(request)
    request['cmap'] = resolve_phase_cmap(request.get('cmap', 'seismic'))
    if request.get('data_file', '') in ['', None]:
        request['data_file'] = config['data']['phase_outfile']
    return save_vertical_animation(
        request,
        config,
        naming_module,
        animation_dir = get_animation_dir(config))


def make_gaussian_filter_plot(request, config, AGWFilter, spectral_analysis, isothermal_dispersion_equations, naming_module, plot_cache):

    '''
    Purpose
    -------
    Plot the Gaussian AGW filter in frequency-wavenumber space.

    Inputs
    ------
    request: dict
        Dictionary containing the plot parameters.

    config: dict
        Shared configuration dictionary.

    AGWFilter: class
        Filtering class used by the time-distance pipeline.

    spectral_analysis: module
        Local spectral-analysis module.

    isothermal_dispersion_equations: module
        Local dispersion-relation module.

    plot_cache: dict
        In-memory cache for the plotting notebook.

    Outputs
    -------
    result: dict
        Dictionary containing the plotted figure, axes, and optional saved file path.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Load the requested cube and build the Gaussian filter
    dataset = request.get('dataset', 'v1').lower()
    v1, v2 = load_raw_dopplergrams(config, AGWFilter, plot_cache)
    cube = v1 if dataset == 'v1' else v2
    filtering = AGWFilter(config)
    filter3D = filtering.build_gaussian_filter(cube)
    cube_xyt = np.transpose(cube, (2, 1, 0))
    _, kx_array, _, freq_array = spectral_analysis.make_fourier_grid(
        config['time_distance']['dt'],
        config['time_distance']['p_dx_Mm'],
        cube_xyt,
        threeD = True)

    # Take the middle-ky filter slice and plot only positive horizontal wavenumbers
    filter_slice = np.asarray(filter3D[:, filter3D.shape[1]//2, :], dtype = np.float64)
    positive_k = np.asarray(kx_array) >= 0.0
    k_plot = np.asarray(kx_array, dtype = np.float64)[positive_k]
    filter_slice = filter_slice[positive_k, :]
    k_limits = get_axis_limits(k_plot, np.median(np.diff(k_plot)))
    freq_limits = get_axis_limits(freq_array, np.median(np.diff(freq_array)))

    fig, ax, cax = make_centered_single_panel_figure(request.get('figsize', [8, 5]))
    image = ax.imshow(
        filter_slice.T,
        origin = 'lower',
        aspect = 'auto',
        extent = [k_limits[0], k_limits[1], freq_limits[0], freq_limits[1]],
        cmap = request.get('cmap', 'viridis'))

    # Overlay the same reference curves used in the k-omega diagram
    if request.get('overlay_lamb_line', True):
        add_komega_dispersion_overlays(ax, k_plot)

    # Format, save, and display the figure
    xlim = request.get('xlim', None)
    ylim = request.get('ylim', None)
    ax.set_xlim(*k_limits if xlim in ['', None] else xlim)
    ax.set_ylim(*freq_limits if ylim in ['', None] else ylim)
    ax.set_xlabel(r'$k_h$ [Mm$^{-1}$]')
    ax.set_ylabel(r'$\nu / 2\pi$ [mHz]')
    ax.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
    default_title = build_request_title(request, config, naming_module, dataset = dataset)
    default_name = f"{build_output_stem(config, 'gaussian_filter', dataset = dataset)}.jpeg"
    set_centered_figure_title(fig, default_title, y = 0.98)
    fig.colorbar(image, cax = cax, label = 'Filter Transmission')
    saved_file = maybe_save_figure(
        fig,
        request.get('output_file', ''),
        default_name = default_name,
        figure_dir = get_figure_dir(config))
    plt.show()

    return {'figure': fig, 'axes': ax, 'saved_file': saved_file}


def make_filtered_dopplergram_plot(request, config, AGWFilter, naming_module, plot_cache):

    '''
    Purpose
    -------
    Plot a Dopplergram map after the configured filtering sequence.

    Inputs
    ------
    request: dict
        Dictionary containing the plot parameters.

    config: dict
        Shared configuration dictionary.

    AGWFilter: class
        Filtering class used by the time-distance pipeline.

    plot_cache: dict
        In-memory cache for the plotting notebook.

    Outputs
    -------
    result: dict
        Dictionary containing the plotted figure, axes, and optional saved file path.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Load the requested filtered cube and extract the selected time slice
    dataset = request.get('dataset', 'v1').lower()
    v1_filt, v2_filt = load_filtered_dopplergrams(config, AGWFilter, plot_cache)
    removed_mask_v1, removed_mask_v2, mask_metadata = load_magnetogram_removed_masks(config, AGWFilter, plot_cache)
    cube = v1_filt if dataset == 'v1' else v2_filt
    removed_mask = removed_mask_v1 if dataset == 'v1' else removed_mask_v2
    time_index = int(np.clip(request.get('time_index', 0), 0, cube.shape[0] - 1))
    dx_Mm = float(config['time_distance']['p_dx_Mm'])

    x = (np.arange(cube.shape[2], dtype = np.float64) - cube.shape[2]//2)*dx_Mm
    y = (np.arange(cube.shape[1], dtype = np.float64) - cube.shape[1]//2)*dx_Mm
    x_limits = get_axis_limits(x, dx_Mm)
    y_limits = get_axis_limits(y, dx_Mm)

    velocity_slice = convert_velocity_to_km_s(cube[time_index, :, :], config)

    if removed_mask is None:
        display_slice = np.ma.array(velocity_slice, mask = np.zeros_like(velocity_slice, dtype = bool))
    else:
        if np.ndim(removed_mask) == 2:
            mask_slice = np.asarray(removed_mask, dtype = bool)
        else:
            mask_slice = np.asarray(removed_mask[time_index, :, :], dtype = bool)
        display_slice = np.ma.array(velocity_slice, mask = mask_slice)

    finite_values = np.asarray(display_slice.compressed(), dtype = np.float64)
    vmax = np.max(np.abs(finite_values)) if finite_values.size > 0 else 1.0

    if vmax == 0.0:
        vmax = 1.0

    norm = TwoSlopeNorm(vmin = -vmax, vcenter = 0.0, vmax = vmax)
    cmap = copy.copy(plt.get_cmap(request.get('cmap', 'RdBu_r')))
    cmap.set_bad('black')

    # Plot the selected spatial map and show the masked pixels in black
    fig, ax, cax = make_centered_single_panel_figure(request.get('figsize', [8, 5]))
    image = ax.pcolormesh(x, y, display_slice, cmap = cmap, norm = norm, shading = 'auto')
    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
    ax.set_xlabel('X [Mm]')
    ax.set_ylabel('Y [Mm]')
    default_title = build_request_title(request, config, naming_module, dataset = dataset, time_index = time_index)
    default_name = f"{build_output_stem(config, 'filtered_dopplergram', dataset = dataset, time_index = time_index)}.jpeg"
    set_centered_figure_title(fig, default_title, y = 0.98)
    fig.colorbar(image, cax = cax, label = r'$v_z$ [km s$^{-1}$]')
    saved_file = maybe_save_figure(
        fig,
        request.get('output_file', ''),
        default_name = default_name,
        figure_dir = get_figure_dir(config))
    plt.show()

    return {
        'figure': fig,
        'axes': ax,
        'saved_file': saved_file,
        'mask_metadata': mask_metadata}


def make_composite_diagnostics_plot(request, config, AGWFilter, spectral_analysis, isothermal_dispersion_equations, naming_module, plot_cache):

    '''
    Purpose
    -------
    Build the composite diagnostics figure with the filter, k-omega, cross-correlation, phase-difference, and Dopplergram panels.

    Inputs
    ------
    request: dict
        Dictionary containing the composite-figure parameters.

    config: dict
        Shared configuration dictionary.

    AGWFilter: class
        Filtering class used by the time-distance pipeline.

    spectral_analysis: module
        Local spectral-analysis module.

    isothermal_dispersion_equations: module
        Local dispersion-relation module.

    naming_module: module
        Shared naming utility module.

    plot_cache: dict
        In-memory cache for the plotting notebook.

    Outputs
    -------
    result: dict
        Dictionary containing the plotted figure, axes, and optional saved file path.

    Author(s)
    ---------
    Julio M. Morales, March 18th, 2026
    '''

    # Reuse the individual plot settings inside the combined diagnostics figure
    gaussian_request = build_plot_request_defaults('gaussian_filter', config)
    komega_request = build_plot_request_defaults('komega_diagram', config)
    time_distance_request = build_plot_request_defaults('time_distance', config)
    phase_request = build_plot_request_defaults('phase_difference', config)
    phase_request['cmap'] = resolve_phase_cmap(phase_request.get('cmap', 'seismic'))
    doppler_request = build_plot_request_defaults('filtered_dopplergram', config)

    fig, axes = plt.subplots(1, 5, figsize = tuple(request.get('figsize', [27.5, 8.0])), constrained_layout = False)
    fig.subplots_adjust(left = 0.04, right = 0.992, bottom = 0.14, top = 0.90, wspace = 0.78)
    ax_filter, ax_komega, ax_xc, ax_phase, ax_dopp = axes

    # Attach one colorbar axis to each panel so every colorbar matches its plot height
    cax_filter = ax_filter.inset_axes([1.03, 0.0, 0.035, 1.0], transform = ax_filter.transAxes)
    cax_komega = ax_komega.inset_axes([1.03, 0.0, 0.035, 1.0], transform = ax_komega.transAxes)
    cax_xc = ax_xc.inset_axes([1.03, 0.0, 0.035, 1.0], transform = ax_xc.transAxes)
    cax_phase = ax_phase.inset_axes([1.03, 0.0, 0.035, 1.0], transform = ax_phase.transAxes)
    cax_dopp = ax_dopp.inset_axes([1.03, 0.0, 0.035, 1.0], transform = ax_dopp.transAxes)

    # Draw the Gaussian filter panel when that filter is part of the active run
    filtering = config.get('filtering', {})
    gaussian_enabled = filtering.get('enabled', False) and ('gaussian' in filtering.get('filter_sequence', [])) and filtering.get('gaussian', {}).get('enabled', False)

    if gaussian_enabled:
        dataset = gaussian_request.get('dataset', 'v1').lower()
        v1, v2 = load_raw_dopplergrams(config, AGWFilter, plot_cache)
        cube = v1 if dataset == 'v1' else v2
        filtering_module = AGWFilter(config)
        filter3D = filtering_module.build_gaussian_filter(cube)
        cube_xyt = np.transpose(cube, (2, 1, 0))
        _, kx_array, _, freq_array = spectral_analysis.make_fourier_grid(
            config['time_distance']['dt'],
            config['time_distance']['p_dx_Mm'],
            cube_xyt,
            threeD = True)

        filter_slice = np.asarray(filter3D[:, filter3D.shape[1]//2, :], dtype = np.float64)
        positive_k = np.asarray(kx_array) >= 0.0
        k_plot = np.asarray(kx_array, dtype = np.float64)[positive_k]
        filter_slice = filter_slice[positive_k, :]
        k_limits = get_axis_limits(k_plot, np.median(np.diff(k_plot)))
        freq_limits = get_axis_limits(freq_array, np.median(np.diff(freq_array)))

        filter_image = ax_filter.imshow(
            filter_slice.T,
            origin = 'lower',
            aspect = 'auto',
            extent = [k_limits[0], k_limits[1], freq_limits[0], freq_limits[1]],
            cmap = gaussian_request.get('cmap', 'viridis'))

        if gaussian_request.get('overlay_lamb_line', True):
            add_komega_dispersion_overlays(ax_filter, k_plot)

        filter_xlim = gaussian_request.get('xlim', None)
        filter_ylim = gaussian_request.get('ylim', None)
        ax_filter.set_xlim(*k_limits if filter_xlim in ['', None] else filter_xlim)
        ax_filter.set_ylim(*freq_limits if filter_ylim in ['', None] else filter_ylim)
        ax_filter.set_xlabel(r'$k_h$ [Mm$^{-1}$]')
        ax_filter.set_ylabel(r'$\nu / 2\pi$ [mHz]')
        ax_filter.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
        fig.colorbar(filter_image, cax = cax_filter, label = 'Filter Transmission')
    else:
        ax_filter.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
        ax_filter.set_xlabel(r'$k_h$ [Mm$^{-1}$]')
        ax_filter.set_ylabel(r'$\nu / 2\pi$ [mHz]')
        ax_filter.text(0.5, 0.5, 'Gaussian disabled', ha = 'center', va = 'center', transform = ax_filter.transAxes)
        cax_filter.axis('off')

    # Draw the k-omega phase-difference panel from the saved pipeline output
    komega_data = load_komega_data(config['data']['komega_outfile'], config)
    komega_k_limits = get_axis_limits(komega_data['k_axis'], np.median(np.diff(komega_data['k_axis'])) if komega_data['k_axis'].size > 1 else 1.0)
    komega_nu_limits = get_axis_limits(komega_data['nu_axis'], np.median(np.diff(komega_data['nu_axis'])) if komega_data['nu_axis'].size > 1 else 1.0)
    komega_xlim = komega_request.get('xlim', None)
    komega_ylim = komega_request.get('ylim', None)
    komega_mesh = ax_komega.pcolormesh(
        komega_data['k_axis'],
        komega_data['nu_axis'],
        komega_data['values'].T,
        cmap = komega_request.get('cmap', cmr.fusion),
        norm = TwoSlopeNorm(
            vmin = komega_request.get('vmin', -20.0),
            vcenter = 0.0,
            vmax = komega_request.get('vmax', 80.0)),
        shading = 'auto')
    if komega_request.get('overlay_curves', True):
        add_komega_dispersion_overlays(ax_komega, komega_data['k_axis'])
    ax_komega.set_xlim(*(komega_k_limits if komega_xlim in ['', None] else komega_xlim))
    ax_komega.set_ylim(*(komega_nu_limits if komega_ylim in ['', None] else komega_ylim))
    ax_komega.set_xlabel(komega_data['horizontal_label'])
    ax_komega.set_ylabel(komega_data['vertical_label'])
    ax_komega.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
    fig.colorbar(komega_mesh, cax = cax_komega, label = komega_request.get('value_label', 'Value'))

    # Draw the cross-correlation panel from the saved notebook output
    xc_data = load_time_distance_data(config['data']['outfile'], config['time_distance']['dt'], config['time_distance']['p_dx_Mm'], axis_type = time_distance_request['axis_type'])
    xc_values = np.asarray(xc_data['values'], dtype = np.float64)
    if time_distance_request.get('normalize_display', False):
        xc_values = normalize_cross_correlation_display(xc_values)
    xc_radius_limits = get_axis_limits(xc_data['radii'], config['time_distance']['p_dx_Mm'])
    xc_vertical_step = float(config['time_distance']['dt'])/60.0
    xc_vertical_limits = get_axis_limits(xc_data['vertical_axis'], xc_vertical_step)
    xc_mesh = ax_xc.pcolormesh(
        xc_data['radii'],
        xc_data['vertical_axis'],
        xc_values.T,
        cmap = time_distance_request.get('cmap', 'viridis'),
        shading = 'auto')
    ax_xc.set_xlim(*xc_radius_limits)
    ax_xc.set_ylim(*xc_vertical_limits)
    ax_xc.set_xlabel('Radius [Mm]')
    ax_xc.set_ylabel(xc_data['vertical_label'])
    ax_xc.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
    add_reference_angle_line(ax_xc, config)
    fig.colorbar(xc_mesh, cax = cax_xc, label = time_distance_request.get('value_label', 'Value'))

    # Draw the phase-difference panel from the saved notebook output
    phase_data = load_time_distance_data(config['data']['phase_outfile'], config['time_distance']['dt'], config['time_distance']['p_dx_Mm'], axis_type = phase_request['axis_type'])
    phase_radius_limits = get_axis_limits(phase_data['radii'], config['time_distance']['p_dx_Mm'])
    phase_vertical_step = 1.0e3/(phase_data['values'].shape[1]*float(config['time_distance']['dt']))
    phase_vertical_limits = get_axis_limits(phase_data['vertical_axis'], phase_vertical_step)
    phase_ylim = phase_request.get('ylim', None)
    phase_mesh = ax_phase.pcolormesh(
        phase_data['radii'],
        phase_data['vertical_axis'],
        phase_data['values'].T,
        cmap = phase_request.get('cmap', 'seismic'),
        vmin = phase_request.get('vmin', None),
        vmax = phase_request.get('vmax', None),
        shading = 'auto')
    ax_phase.set_xlim(*phase_radius_limits)
    ax_phase.set_ylim(*(phase_vertical_limits if phase_ylim in ['', None] else phase_ylim))
    ax_phase.set_xlabel('Radius [Mm]')
    ax_phase.set_ylabel(phase_data['vertical_label'])
    ax_phase.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
    add_reference_angle_line(ax_phase, config)
    fig.colorbar(phase_mesh, cax = cax_phase, label = phase_request.get('value_label', 'Value'))

    # Draw the Dopplergram panel and keep the masked pixels black
    dataset = doppler_request.get('dataset', 'v1').lower()
    v1_filt, v2_filt = load_filtered_dopplergrams(config, AGWFilter, plot_cache)
    removed_mask_v1, removed_mask_v2, _ = load_magnetogram_removed_masks(config, AGWFilter, plot_cache)
    cube = v1_filt if dataset == 'v1' else v2_filt
    removed_mask = removed_mask_v1 if dataset == 'v1' else removed_mask_v2
    time_index = int(np.clip(doppler_request.get('time_index', 0), 0, cube.shape[0] - 1))
    dx_Mm = float(config['time_distance']['p_dx_Mm'])

    x = (np.arange(cube.shape[2], dtype = np.float64) - cube.shape[2]//2)*dx_Mm
    y = (np.arange(cube.shape[1], dtype = np.float64) - cube.shape[1]//2)*dx_Mm
    x_limits = get_axis_limits(x, dx_Mm)
    y_limits = get_axis_limits(y, dx_Mm)
    velocity_slice = convert_velocity_to_km_s(cube[time_index, :, :], config)

    if removed_mask is None:
        display_slice = np.ma.array(velocity_slice, mask = np.zeros_like(velocity_slice, dtype = bool))
    else:
        if np.ndim(removed_mask) == 2:
            mask_slice = np.asarray(removed_mask, dtype = bool)
        else:
            mask_slice = np.asarray(removed_mask[time_index, :, :], dtype = bool)
        display_slice = np.ma.array(velocity_slice, mask = mask_slice)

    finite_values = np.asarray(display_slice.compressed(), dtype = np.float64)
    vmax = np.max(np.abs(finite_values)) if finite_values.size > 0 else 1.0
    if vmax == 0.0:
        vmax = 1.0

    doppler_norm = TwoSlopeNorm(vmin = -vmax, vcenter = 0.0, vmax = vmax)
    doppler_cmap = copy.copy(plt.get_cmap(doppler_request.get('cmap', 'RdBu_r')))
    doppler_cmap.set_bad('black')
    doppler_image = ax_dopp.pcolormesh(x, y, display_slice, cmap = doppler_cmap, norm = doppler_norm, shading = 'auto')
    ax_dopp.set_xlim(*x_limits)
    ax_dopp.set_ylim(*y_limits)
    ax_dopp.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
    ax_dopp.set_xlabel('X [Mm]')
    ax_dopp.set_ylabel('Y [Mm]')
    fig.colorbar(doppler_image, cax = cax_dopp, label = r'$v_z$ [km s$^{-1}$]')

    # Add the shared publication title and write the figure to disk
    set_centered_figure_title(fig, build_request_title(request, config, naming_module), y = 0.98)
    saved_file = maybe_save_figure(
        fig,
        request.get('output_file', ''),
        default_name = f"{build_output_stem(config, 'composite_diagnostics')}.jpeg",
        figure_dir = get_figure_dir(config))
    plt.show()

    return {
        'figure': fig,
        'axes': [ax_filter, ax_komega, ax_xc, ax_phase, ax_dopp],
        'saved_file': saved_file}


COMPARISON_PLOT_TYPES = {
    'field_strength_comparison',
    'field_orientation_comparison',
    'gaussian_filter_comparison'}
COMPARISON_OUTPUT_KEYS = ['outfile', 'phase_outfile', 'komega_outfile']



def is_comparison_plot_type(plot_type):

    '''
    Purpose
    -------
    Is comparison plot type.

    Inputs
    ------
    plot_type: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    return str(plot_type).strip() in COMPARISON_PLOT_TYPES



def normalize_comparison_execution_mode(execution_mode = None, use_cached_data = None):

    '''
    Purpose
    -------
    Normalize comparison execution mode.

    Inputs
    ------
    execution_mode: object
        Input parameter.
    use_cached_data: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    if execution_mode in ['', None]:
        if use_cached_data in [True, False]:
            execution_mode = 'load' if bool(use_cached_data) else 'recompute'
        else:
            execution_mode = 'load'

    execution_mode = str(execution_mode).strip().lower()

    if execution_mode not in ['load', 'recompute']:
        raise ValueError("Comparison execution_mode must be 'load' or 'recompute'.")

    return execution_mode



def normalize_comparison_missing_data_behavior(missing_data_behavior = None, recompute_on_missing_data = None):

    '''
    Purpose
    -------
    Normalize comparison missing data behavior.

    Inputs
    ------
    missing_data_behavior: object
        Input parameter.
    recompute_on_missing_data: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    if isinstance(missing_data_behavior, bool):
        missing_data_behavior = 'recompute' if bool(missing_data_behavior) else 'error'
    elif missing_data_behavior in ['', None]:
        if recompute_on_missing_data in [True, False]:
            missing_data_behavior = 'recompute' if bool(recompute_on_missing_data) else 'error'
        else:
            missing_data_behavior = 'error'

    missing_data_behavior = str(missing_data_behavior).strip().lower()

    if missing_data_behavior not in ['error', 'recompute']:
        raise ValueError("Comparison missing_data_behavior must be 'error' or 'recompute'.")

    return missing_data_behavior



def resolve_comparison_execution_settings(request):

    '''
    Purpose
    -------
    Resolve comparison execution settings.

    Inputs
    ------
    request: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    return {
        'execution_mode': normalize_comparison_execution_mode(
            request.get('execution_mode', None),
            use_cached_data = request.get('use_cached_data', None),
        ),
        'missing_data_behavior': normalize_comparison_missing_data_behavior(
            request.get('missing_data_behavior', request.get('on_missing_data', None)),
            recompute_on_missing_data = request.get('recompute_on_missing_data', None),
        )}



def apply_comparison_execution_defaults(request, execution_mode = 'load', missing_data_behavior = 'error', explicit_request = None):

    '''
    Purpose
    -------
    Apply comparison execution defaults.

    Inputs
    ------
    request: object
        Input parameter.
    execution_mode: object
        Input parameter.
    missing_data_behavior: object
        Input parameter.
    explicit_request: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    request = copy.deepcopy(request)
    explicit_request = {} if explicit_request is None else explicit_request

    if not is_comparison_plot_type(request.get('plot_type', '')):
        return request

    if explicit_request.get('execution_mode', None) in ['', None] and explicit_request.get('use_cached_data', None) in ['', None]:
        request['execution_mode'] = normalize_comparison_execution_mode(execution_mode)

    if (
        explicit_request.get('missing_data_behavior', None) in ['', None]
        and explicit_request.get('on_missing_data', None) in ['', None]
        and explicit_request.get('recompute_on_missing_data', None) in ['', None]
    ):
        request['missing_data_behavior'] = normalize_comparison_missing_data_behavior(missing_data_behavior)

    return request



def comparison_runtime_output_paths(runtime_config):

    '''
    Purpose
    -------
    Comparison runtime output paths.

    Inputs
    ------
    runtime_config: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    data = runtime_config.get('data', {})
    output_paths = {}
    missing_keys = []

    for output_key in COMPARISON_OUTPUT_KEYS:
        output_file = data.get(output_key, '')
        if output_file in ['', None]:
            missing_keys.append(output_key)
            continue
        output_paths[output_key] = Path(output_file).expanduser().resolve()

    if len(missing_keys) > 0:
        raise KeyError(
            'Comparison plots require runtime output paths for outfile, phase_outfile, and komega_outfile. '
            f"Missing keys: {', '.join(missing_keys)}."
        )

    return output_paths



def describe_comparison_runtime_config(runtime_config, modules):

    '''
    Purpose
    -------
    Describe comparison runtime config.

    Inputs
    ------
    runtime_config: object
        Input parameter.
    modules: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    data = runtime_config.get('data', {})
    if data.get('source_type', '') == 'single_cube':
        return (
            f"{Path(data.get('file', '')).name} | {data.get('observable', '')} | "
            f"{data.get('resolved_h1_km', data.get('h1', '?'))} km vs "
            f"{data.get('resolved_h2_km', data.get('h2', '?'))} km"
        )

    return str(data.get('source_type', 'comparison run'))




def build_missing_comparison_outputs_error(plot_label, missing_records):

    '''
    Purpose
    -------
    Build missing comparison outputs error.

    Inputs
    ------
    plot_label: object
        Input parameter.
    missing_records: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    preview_lines = []
    for record in missing_records[:8]:
        preview_lines.append(f"{record['label']}: {', '.join(record['missing_outputs'])}")

    preview = '\n'.join(preview_lines)

    return (
        f"{plot_label} requires existing pipeline outputs when execution_mode = 'load'. Missing files include:\n"
        f"{preview}\n"
        "Set execution_mode = 'recompute' or missing_data_behavior = 'recompute' to regenerate them."
    )



def recompute_comparison_runtime_records(records, modules, config_file = None):

    '''
    Purpose
    -------
    Recompute comparison runtime records.

    Inputs
    ------
    records: object
        Input parameter.
    modules: object
        Input parameter.
    config_file: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    if len(records) == 0:
        return

    time_distance_module = modules.get('time_distance_module', None)
    if time_distance_module is None:
        raise KeyError('The time_distance_module is required to recompute comparison products.')

    resolved_config_file = resolve_config_file() if config_file in ['', None] else Path(config_file).expanduser().resolve()

    for record_index, record in enumerate(records, start = 1):
        print(f"[{record_index}/{len(records)}] recomputing {record['label']}")
        results = time_distance_module.run_time_distance(
            config_file = resolved_config_file,
            config_override = copy.deepcopy(record['runtime_config']),
        )

        for output_key in COMPARISON_OUTPUT_KEYS:
            if output_key in results:
                record['runtime_config']['data'][output_key] = str(Path(results[output_key]).expanduser().resolve())

        record['output_paths'] = comparison_runtime_output_paths(record['runtime_config'])
        record['missing_outputs'] = [
            str(path) for path in record['output_paths'].values() if not path.exists()
        ]

        if len(record['missing_outputs']) > 0:
            raise FileNotFoundError(
                f"Recomputed comparison outputs are still missing for {record['label']}: "
                f"{', '.join(record['missing_outputs'])}"
            )



def ensure_comparison_runtime_products(runtime_configs, request, modules, plot_label = 'Comparison plot', config_file = None):

    '''
    Purpose
    -------
    Ensure comparison runtime products.

    Inputs
    ------
    runtime_configs: object
        Input parameter.
    request: object
        Input parameter.
    modules: object
        Input parameter.
    plot_label: object
        Input parameter.
    config_file: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    runtime_configs = [copy.deepcopy(runtime_config) for runtime_config in runtime_configs if runtime_config is not None]

    if len(runtime_configs) == 0:
        return []

    execution_settings = resolve_comparison_execution_settings(request)
    record_lookup = {}
    ordered_records = []

    for runtime_config in runtime_configs:
        output_paths = comparison_runtime_output_paths(runtime_config)
        record_key = tuple(str(output_paths[output_key]) for output_key in COMPARISON_OUTPUT_KEYS)

        if record_key not in record_lookup:
            record_lookup[record_key] = {
                'runtime_config': runtime_config,
                'output_paths': output_paths,
                'label': describe_comparison_runtime_config(runtime_config, modules),
                'missing_outputs': [
                    str(path) for path in output_paths.values() if not path.exists()
                ],
            }

        ordered_records.append(record_lookup[record_key])

    unique_records = list(record_lookup.values())
    missing_records = [record for record in unique_records if len(record['missing_outputs']) > 0]

    if execution_settings['execution_mode'] == 'load':
        if len(missing_records) > 0:
            if execution_settings['missing_data_behavior'] == 'recompute':
                print(f"{plot_label}: recomputing {len(missing_records)} missing runtime product set(s).")
                recompute_comparison_runtime_records(missing_records, modules, config_file = config_file)
            else:
                raise FileNotFoundError(build_missing_comparison_outputs_error(plot_label, missing_records))
    else:
        print(f"{plot_label}: recomputing {len(unique_records)} runtime product set(s).")
        recompute_comparison_runtime_records(unique_records, modules, config_file = config_file)

    return [record['runtime_config'] for record in ordered_records]



def build_plot_requests(config, use_config = True, direct_plot_requests = None, comparison_execution_mode = 'load', comparison_missing_data_behavior = 'error'):

    '''
    Purpose
    -------
    Build plot requests.

    Inputs
    ------
    config: object
        Input parameter.
    use_config: object
        Input parameter.
    direct_plot_requests: object
        Input parameter.
    comparison_execution_mode: object
        Input parameter.
    comparison_missing_data_behavior: object
        Input parameter.

    Outputs
    -------
    result: object
        Helper return value.

    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    plot_requests = []
    generate_config = config.get('plots', {}).get('generate', {})

    if use_config:
        if isinstance(generate_config, dict):
            for flag_name, enabled in generate_config.items():
                if not enabled:
                    continue
                if flag_name not in PLOT_FLAG_MAP:
                    raise KeyError(f"Unknown plot flag in config['plots']['generate']: {flag_name}")
                plot_type = PLOT_FLAG_MAP[flag_name]
                request = build_plot_request_defaults(plot_type, config)
                request = apply_comparison_execution_defaults(
                    request,
                    execution_mode = comparison_execution_mode,
                    missing_data_behavior = comparison_missing_data_behavior,
                    explicit_request = {},
                )
                plot_requests.append(request)
        else:
            for plot_type in generate_config:
                request = build_plot_request_defaults(plot_type, config)
                request = apply_comparison_execution_defaults(
                    request,
                    execution_mode = comparison_execution_mode,
                    missing_data_behavior = comparison_missing_data_behavior,
                    explicit_request = {},
                )
                plot_requests.append(request)
    else:
        direct_plot_requests = direct_plot_requests or []
        for item in direct_plot_requests:
            if isinstance(item, str):
                request = {'plot_type': item}
            else:
                request = copy.deepcopy(item)
                if 'plot_type' not in request:
                    raise KeyError('Each direct plot request must include a plot_type entry.')

            defaults = build_plot_request_defaults(request['plot_type'], config)
            defaults.update(request)
            defaults['plot_type'] = request['plot_type']
            defaults = apply_comparison_execution_defaults(
                defaults,
                execution_mode = comparison_execution_mode,
                missing_data_behavior = comparison_missing_data_behavior,
                explicit_request = request,
            )

            if defaults.get('data_file', '') in ['', None]:
                if defaults['plot_type'] in ['time_distance', 'correlation_radius_animation', 'correlation_vertical_animation']:
                    defaults['data_file'] = config['data']['outfile']
                elif defaults['plot_type'] in ['phase_difference', 'phase_radius_animation', 'phase_vertical_animation']:
                    defaults['data_file'] = config['data']['phase_outfile']
                elif defaults['plot_type'] == 'komega_diagram':
                    defaults['data_file'] = config['data']['komega_outfile']

            plot_requests.append(defaults)

    return plot_requests


def run_plot_request(request, config, modules, plot_cache):

    '''
    Purpose
    -------
    Run a single plotting request.

    Inputs
    ------
    request: dict
        Dictionary containing the plot type and parameters.

    config: dict
        Shared configuration dictionary.

    modules: dict
        Dictionary containing the local modules needed for plotting.

    plot_cache: dict
        In-memory cache for the plotting notebook.

    Outputs
    -------
    result: dict or pathlib.Path
        Output from the selected plot function.

    Author(s)
    ---------
    Julio M. Morales, March 13th, 2026
    '''

    # Route the request to the corresponding plot function
    plot_type = request['plot_type']
    AGWFilter = modules['AGWFilter']
    spectral_analysis = modules['spectral_analysis']
    isothermal_dispersion_equations = modules['isothermal_dispersion_equations']
    naming_module = modules['naming_module']

    if plot_type == 'time_distance':
        return make_time_distance_plot(request, config, naming_module)
    if plot_type == 'phase_difference':
        return make_phase_difference_plot(request, config, naming_module)
    if plot_type == 'komega_diagram':
        return make_komega_plot(request, config, naming_module)
    if plot_type == 'correlation_radius_animation':
        return make_correlation_radius_animation(request, config, naming_module)
    if plot_type == 'correlation_vertical_animation':
        return make_correlation_vertical_animation(request, config, naming_module)
    if plot_type == 'phase_radius_animation':
        return make_phase_radius_animation(request, config, naming_module)
    if plot_type == 'phase_vertical_animation':
        return make_phase_vertical_animation(request, config, naming_module)
    if plot_type == 'gaussian_filter':
        return make_gaussian_filter_plot(request, config, AGWFilter, spectral_analysis, isothermal_dispersion_equations, naming_module, plot_cache)
    if plot_type == 'filtered_dopplergram':
        return make_filtered_dopplergram_plot(request, config, AGWFilter, naming_module, plot_cache)
    if plot_type == 'composite_diagnostics':
        return make_composite_diagnostics_plot(request, config, AGWFilter, spectral_analysis, isothermal_dispersion_equations, naming_module, plot_cache)
    if plot_type == 'field_strength_comparison':
        return make_field_strength_comparison_plot(request, config, modules, plot_cache)
    if plot_type == 'field_orientation_comparison':
        return make_field_orientation_comparison_plot(request, config, modules, plot_cache)
    if plot_type == 'gaussian_filter_comparison':
        return make_gaussian_filter_comparison_plot(request, config, modules, plot_cache)

    raise ValueError(f'Unsupported plot type: {plot_type}')


In [ ]:


def normalize_cube_paths(cube_paths):

    '''
    Purpose
    -------
    Normalize cube paths.
    
    Inputs
    ------
    cube_paths: object
        Cube paths used by this helper.
    
    Outputs
    -------
    normalized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    normalized_paths = []
    seen = set()

    for cube_path in cube_paths:
        if cube_path in ['', None]:
            continue

        resolved_path = str(Path(cube_path).expanduser().resolve())

        if resolved_path in seen:
            continue

        seen.add(resolved_path)
        normalized_paths.append(resolved_path)

    return normalized_paths



def parse_single_cube_field_strength_case(cube_path):

    '''
    Purpose
    -------
    Parse single cube field-strength case.
    
    Inputs
    ------
    cube_path: object
        Cube path used by this helper.
    
    Outputs
    -------
    parsed_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    resolved_path = str(Path(cube_path).expanduser().resolve())
    path = Path(resolved_path)
    component = ''
    strength_token = ''

    for part in path.parts:
        lower_part = part.lower()

        if lower_part in ['hx', 'vx', 'z0']:
            component = lower_part

        if re.fullmatch(r'\d+(?:[._]\d+)?g', lower_part):
            strength_token = lower_part

    if component == '' or strength_token == '':
        match = re.search(
            r'(hx|vx|z0)[_\-/](\d+(?:[._]\d+)?g)',
            resolved_path,
            flags = re.IGNORECASE,
        )

        if match is not None:
            component = match.group(1).lower()
            strength_token = match.group(2).lower()

    if component not in ['hx', 'vx']:
        raise ValueError(
            "Field-strength comparison requires cube paths that encode either 'hx' or 'vx' geometry. "
            f'Could not infer a supported geometry from {resolved_path}.'
        )

    if strength_token == '':
        raise ValueError(
            'Field-strength comparison requires cube paths that encode a field-strength folder such as '
            f"'10G'. Could not infer the field strength from {resolved_path}."
        )

    field_strength_G = float(strength_token[:-1].replace('_', '.'))
    strength_label_value = (
        f'{int(round(field_strength_G))}'
        if np.isclose(field_strength_G, round(field_strength_G))
        else f'{field_strength_G:g}'
    )

    return {
        'cube_path': resolved_path,
        'component': component,
        'geometry': 'horizontal' if component == 'hx' else 'vertical',
        'field_strength_G': field_strength_G,
        'field_strength_token': strength_token,
        'field_strength_label': f'{strength_label_value} G'}



def organize_single_cube_field_strength_cases(cube_paths):

    '''
    Purpose
    -------
    Organize single cube field-strength cases.
    
    Inputs
    ------
    cube_paths: object
        Cube paths used by this helper.
    
    Outputs
    -------
    organized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    normalized_paths = normalize_cube_paths(cube_paths)

    if len(normalized_paths) == 0:
        raise ValueError('Field-strength comparison requires at least one single_cube path.')

    cases_by_geometry = {
        'horizontal': [],
        'vertical': []}
    seen_case_keys = {}

    for cube_path in normalized_paths:
        case = parse_single_cube_field_strength_case(cube_path)
        case_key = (case['geometry'], case['field_strength_token'])

        if case_key in seen_case_keys:
            raise ValueError(
                'Duplicate field-strength comparison case detected for '
                f"{case['geometry']} {case['field_strength_label']}: "
                f"{seen_case_keys[case_key]} and {case['cube_path']}"
            )

        seen_case_keys[case_key] = case['cube_path']
        cases_by_geometry[case['geometry']].append(case)

    for geometry_cases in cases_by_geometry.values():
        geometry_cases.sort(key = lambda item: (item['field_strength_G'], item['cube_path']))

    return {
        'cube_paths': normalized_paths,
        'cases_by_geometry': cases_by_geometry,
        'field_strengths_by_geometry': {
            geometry: [case['field_strength_G'] for case in geometry_cases]
            for geometry, geometry_cases in cases_by_geometry.items()
        },
        'ordered_cases': cases_by_geometry['horizontal'] + cases_by_geometry['vertical']}



def normalize_gaussian_filter_param_set(filter_params, reference_config = None, filter_index = None):

    '''
    Purpose
    -------
    Normalize Gaussian-filter param set.
    
    Inputs
    ------
    filter_params: object
        Gaussian-filter parameters used by this helper.
    
    reference_config: object
        Reference config used by this helper.
    
    filter_index: object
        Filter index used by this helper.
    
    Outputs
    -------
    normalized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    if not isinstance(filter_params, dict):
        raise TypeError('Each Gaussian filter parameter set must be provided as a dictionary.')

    raw_gaussian = filter_params.get('gaussian', filter_params)
    if not isinstance(raw_gaussian, dict):
        raise TypeError('Gaussian filter parameters must be stored in a dictionary.')

    gaussian_defaults = {}
    if reference_config is not None:
        gaussian_defaults = copy.deepcopy(reference_config.get('filtering', {}).get('gaussian', {}))

    gaussian_config = copy.deepcopy(gaussian_defaults)
    for key, value in raw_gaussian.items():
        if key in ['label', 'name', 'slug']:
            continue
        gaussian_config[key] = value

    gaussian_config['enabled'] = bool(gaussian_config.get('enabled', True))

    required_keys = ['central_k', 'width_k', 'central_f', 'width_f']
    missing_keys = [key for key in required_keys if gaussian_config.get(key, None) in ['', None]]
    if len(missing_keys) > 0:
        raise ValueError(
            'Each Gaussian filter parameter set must define '
            f"{', '.join(missing_keys)}."
        )

    return {
        'filter_index': None if filter_index is None else int(filter_index),
        'label': str(filter_params.get('label', filter_params.get('name', ''))),
        'gaussian': gaussian_config}



def normalize_gaussian_filter_param_list(filter_params_list, reference_config = None):

    '''
    Purpose
    -------
    Normalize Gaussian-filter param list.
    
    Inputs
    ------
    filter_params_list: object
        Gaussian-filter parameter sets used by this helper.
    
    reference_config: object
        Reference config used by this helper.
    
    Outputs
    -------
    normalized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    normalized_filter_params = []

    for filter_index, filter_params in enumerate(filter_params_list):
        normalized_filter_params.append(
            normalize_gaussian_filter_param_set(
                filter_params,
                reference_config = reference_config,
                filter_index = filter_index,
            )
        )

    if len(normalized_filter_params) == 0:
        raise ValueError('Gaussian-filter comparison requires at least one Gaussian filter parameter set.')

    return normalized_filter_params



def parse_single_cube_gaussian_filter_comparison_case(cube_path):

    '''
    Purpose
    -------
    Parse single cube Gaussian-filter comparison case.
    
    Inputs
    ------
    cube_path: object
        Cube path used by this helper.
    
    Outputs
    -------
    parsed_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    resolved_path = str(Path(cube_path).expanduser().resolve())
    path = Path(resolved_path)
    component = ''
    strength_token = ''

    for part in path.parts:
        lower_part = part.lower()

        if lower_part in ['hx', 'vx', 'z0']:
            component = lower_part

        if re.fullmatch(r'\d+(?:[._]\d+)?g', lower_part):
            strength_token = lower_part

    if component == '' or strength_token == '':
        match = re.search(
            r'(hx|vx|z0)[_\-/](\d+(?:[._]\d+)?g)',
            resolved_path,
            flags = re.IGNORECASE,
        )

        if match is not None:
            component = match.group(1).lower()
            strength_token = match.group(2).lower()

    if component == '' and strength_token == '0g':
        component = 'z0'

    if component == '' or strength_token == '':
        raise ValueError(
            'Gaussian-filter comparison requires cube paths that encode a supported geometry folder '
            "(hx, vx, or z0) and a field-strength folder such as '10G'. "
            f'Could not infer those values from {resolved_path}.'
        )

    field_strength_G = float(strength_token[:-1].replace('_', '.'))
    strength_label_value = (
        f'{int(round(field_strength_G))}'
        if np.isclose(field_strength_G, round(field_strength_G))
        else f'{field_strength_G:g}'
    )

    if component == 'z0' or np.isclose(field_strength_G, 0.0, rtol = 1.0e-9, atol = 1.0e-9):
        case_key = '0g'
        comparison_label = '0G'
    elif component == 'hx':
        if not any(np.isclose(field_strength_G, value, rtol = 1.0e-9, atol = 1.0e-9) for value in [10.0, 50.0, 100.0]):
            raise ValueError(
                'Gaussian-filter comparison supports only the horizontal 10G, 50G, and 100G cases. '
                f'Received {resolved_path}.'
            )

        case_key = f'h{strength_label_value.lower()}g'
        comparison_label = f'h{strength_label_value}G'
    elif component == 'vx':
        if not any(np.isclose(field_strength_G, value, rtol = 1.0e-9, atol = 1.0e-9) for value in [10.0, 50.0, 100.0]):
            raise ValueError(
                'Gaussian-filter comparison supports only the vertical 10G, 50G, and 100G cases. '
                f'Received {resolved_path}.'
            )

        case_key = f'v{strength_label_value.lower()}g'
        comparison_label = f'v{strength_label_value}G'
    else:
        raise ValueError(
            'Gaussian-filter comparison requires hx, vx, or z0 simulation paths. '
            f'Could not infer a supported case from {resolved_path}.'
        )

    return {
        'cube_path': resolved_path,
        'component': component,
        'field_strength_G': field_strength_G,
        'field_strength_token': strength_token,
        'field_strength_label': f'{strength_label_value} G',
        'case_key': case_key,
        'comparison_label': comparison_label}



def organize_single_cube_gaussian_filter_comparison_cases(cube_paths):

    '''
    Purpose
    -------
    Organize single cube Gaussian-filter comparison cases.
    
    Inputs
    ------
    cube_paths: object
        Cube paths used by this helper.
    
    Outputs
    -------
    organized_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    normalized_paths = normalize_cube_paths(cube_paths)
    if len(normalized_paths) == 0:
        raise ValueError('Gaussian-filter comparison requires at least one single_cube path.')

    required_order = ['0g', 'h10g', 'h50g', 'h100g', 'v10g', 'v50g', 'v100g']
    case_lookup = {}

    for cube_path in normalized_paths:
        case = parse_single_cube_gaussian_filter_comparison_case(cube_path)

        if case['case_key'] in case_lookup:
            raise ValueError(
                'Duplicate Gaussian-filter comparison case detected for '
                f"{case['comparison_label']}: {case_lookup[case['case_key']]['cube_path']} and {case['cube_path']}"
            )

        case_lookup[case['case_key']] = case

    missing_cases = [case_key.upper() for case_key in required_order if case_key not in case_lookup]
    if len(missing_cases) > 0:
        missing_labels = [case_lookup_key.replace('H', 'h').replace('V', 'v') for case_lookup_key in missing_cases]
        raise ValueError(
            'Gaussian-filter comparison requires the seven simulation cases '
            '0G, h10G, h50G, h100G, v10G, v50G, and v100G. '
            f"Missing cases: {', '.join(missing_labels)}."
        )

    ordered_cases = [case_lookup[case_key] for case_key in required_order]

    return {
        'cube_paths': [case['cube_path'] for case in ordered_cases],
        'case_lookup': case_lookup,
        'ordered_cases': ordered_cases,
        'ordered_labels': [case['comparison_label'] for case in ordered_cases]}

def apply_gaussian_filter_params_to_runtime_config(runtime_config, filter_params = None):

    '''
    Purpose
    -------
    Apply Gaussian-filter params to runtime config.
    
    Inputs
    ------
    runtime_config: object
        Runtime config used by this helper.
    
    filter_params: object
        Gaussian-filter parameters used by this helper.
    
    Outputs
    -------
    updated_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    if filter_params in ['', None]:
        return runtime_config

    runtime_config = copy.deepcopy(runtime_config)
    raw_gaussian = filter_params.get('gaussian', filter_params)

    if not isinstance(raw_gaussian, dict):
        raise TypeError('Gaussian filter parameters must be stored in a dictionary.')

    filtering = runtime_config.setdefault('filtering', {})
    filter_sequence = list(filtering.get('filter_sequence', []))
    if 'gaussian' not in filter_sequence:
        filter_sequence.append('gaussian')
    filtering['filter_sequence'] = filter_sequence
    filtering['enabled'] = True

    gaussian_config = copy.deepcopy(filtering.get('gaussian', {}))
    for key, value in raw_gaussian.items():
        if key in ['label', 'name', 'slug', 'filter_index']:
            continue
        gaussian_config[key] = value

    gaussian_config['enabled'] = bool(gaussian_config.get('enabled', True))
    filtering['gaussian'] = gaussian_config

    return runtime_config



def build_field_strength_comparison_runtime_config(base_config, cube_path, h1, h2, observable = None, model_atmosphere_path = None, filter_params = None):

    '''
    Purpose
    -------
    Build field-strength comparison runtime config.
    
    Inputs
    ------
    base_config: object
        Base configuration used to build the requested result.
    
    cube_path: object
        Cube path used by this helper.
    
    h1: object
        H1 used by this helper.
    
    h2: object
        H2 used by this helper.
    
    observable: object
        Observable name used by this helper.
    
    model_atmosphere_path: object
        Path to the model-atmosphere file.
    
    filter_params: object
        Gaussian-filter parameters used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    runtime_config = copy.deepcopy(base_config)
    runtime_config['data']['source_type'] = 'single_cube'
    runtime_config['data'].setdefault('single_cube', {})
    runtime_config['data']['single_cube']['file'] = str(Path(cube_path).expanduser().resolve())
    runtime_config['data']['single_cube']['h1'] = int(h1)
    runtime_config['data']['single_cube']['h2'] = int(h2)

    if observable not in ['', None]:
        runtime_config['data']['single_cube']['observable'] = str(observable)

    if model_atmosphere_path not in ['', None]:
        runtime_config['data']['single_cube']['model_atmosphere_path'] = str(Path(model_atmosphere_path).expanduser().resolve())

    runtime_config = apply_gaussian_filter_params_to_runtime_config(runtime_config, filter_params = filter_params)

    return prepare_runtime_config(runtime_config)



def resolve_field_strength_comparison_observable(request, config):

    '''
    Purpose
    -------
    Resolve field-strength comparison observable.
    
    Inputs
    ------
    request: object
        Request used by this helper.
    
    config: object
        Runtime configuration used by this helper.
    
    Outputs
    -------
    resolved_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    if request.get('observable', '') not in ['', None]:
        return str(request['observable'])

    data = config.get('data', {})
    single_cube = data.get('single_cube', {})

    return str(data.get('observable', single_cube.get('observable', '')))



def build_field_strength_comparison_column_cases(organized_cases):

    '''
    Purpose
    -------
    Build field-strength comparison column cases.
    
    Inputs
    ------
    organized_cases: object
        Organized cases used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    cases_by_geometry = organized_cases['cases_by_geometry']

    if len(cases_by_geometry['horizontal']) == 0 or len(cases_by_geometry['vertical']) == 0:
        raise ValueError('Field-strength comparison requires at least one horizontal and one vertical cube path.')

    column_cases = []

    for geometry in ['horizontal', 'vertical']:
        geometry_cases = list(cases_by_geometry[geometry])

        if len(geometry_cases) > 4:
            raise ValueError(
                f"Field-strength comparison supports up to four {geometry} field-strength cases; "
                f"received {len(geometry_cases)}."
            )

        column_cases.extend(geometry_cases + [None]*(4 - len(geometry_cases)))

    return column_cases



def build_field_strength_comparison_suptitle(h1_km, h2_km):

    '''
    Purpose
    -------
    Build field-strength comparison suptitle.
    
    Inputs
    ------
    h1_km: object
        H1 km used by this helper.
    
    h2_km: object
        H2 km used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    return f'Field Strength Comparison\n{_format_height_km(h1_km)} km vs {_format_height_km(h2_km)} km'



def build_field_strength_comparison_output_stem(config, organized_cases, h1_km, h2_km, observable):

    '''
    Purpose
    -------
    Build field-strength comparison output stem.
    
    Inputs
    ------
    config: object
        Runtime configuration used by this helper.
    
    organized_cases: object
        Organized cases used by this helper.
    
    h1_km: object
        H1 km used by this helper.
    
    h2_km: object
        H2 km used by this helper.
    
    observable: object
        Observable name used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    observable_slug = normalize_observable_slug(observable)
    processing_slug = build_processing_labels(config)['slug']
    stem_parts = [
        'field_strength_comparison',
        observable_slug,
        f"{_slugify(_format_height_km(h1_km))}km",
        f"{_slugify(_format_height_km(h2_km))}km",
    ]

    for geometry in ['horizontal', 'vertical']:
        geometry_slug = 'hx' if geometry == 'horizontal' else 'vx'
        strength_slug = [case['field_strength_token'] for case in organized_cases['cases_by_geometry'][geometry]]
        stem_parts.append(_join_slug([geometry_slug] + strength_slug))

    if processing_slug != '':
        stem_parts.append(processing_slug)

    return _join_slug(stem_parts)



def build_field_strength_comparison_gaussian_title(config):

    '''
    Purpose
    -------
    Build field-strength comparison gaussian title.
    
    Inputs
    ------
    config: object
        Runtime configuration used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    filtering = config.get('filtering', {})
    filter_sequence = filtering.get('filter_sequence', [])
    gaussian_config = filtering.get('gaussian', {})

    if not filtering.get('enabled', False):
        return 'Gaussian Filter'

    if 'gaussian' not in filter_sequence or not gaussian_config.get('enabled', False):
        return 'Gaussian Filter'

    sigma_k = float(gaussian_config['width_k'])/2.355
    sigma_nu = float(gaussian_config['width_f'])/2.355

    return (
        'Gaussian Filter: '
        rf'$k_0 = {float(gaussian_config["central_k"]):g}\,\mathrm{{Mm^{{-1}}}},\ '
        rf'\sigma_k = {sigma_k:g}\,\mathrm{{Mm^{{-1}}}},\ '
        rf'\nu_0 = {float(gaussian_config["central_f"]):g}\,\mathrm{{mHz}},\ '
        rf'\sigma_\nu = {sigma_nu:g}\,\mathrm{{mHz}}$'
    )



def load_field_strength_comparison_gaussian_panel_data(config, AGWFilter, spectral_analysis, plot_cache, dataset = 'v1'):

    '''
    Purpose
    -------
    Load field-strength comparison gaussian panel data.
    
    Inputs
    ------
    config: object
        Runtime configuration used by this helper.
    
    AGWFilter: object
        AGWFilter used by this helper.
    
    spectral_analysis: object
        Spectral analysis used by this helper.
    
    plot_cache: object
        Plot cache used by this helper.
    
    dataset: object
        Dataset used by this helper.
    
    Outputs
    -------
    loaded_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    filtering = config.get('filtering', {})
    gaussian_config = filtering.get('gaussian', {})
    dataset = str(dataset).lower()
    cache_key = (
        'field_strength_comparison_gaussian_panel',
        str(config.get('data', {}).get('file', '')),
        str(config.get('data', {}).get('observable', '')),
        dataset,
        float(config['time_distance']['dt']),
        float(config['time_distance']['p_dx_Mm']),
        tuple(filtering.get('filter_sequence', [])),
        tuple(sorted((key, gaussian_config.get(key)) for key in gaussian_config)),
    )

    if cache_key not in plot_cache:
        gaussian_enabled = filtering.get('enabled', False) and ('gaussian' in filtering.get('filter_sequence', [])) and gaussian_config.get('enabled', False)

        if not gaussian_enabled:
            plot_cache[cache_key] = {'enabled': False}
        else:
            raw_key = (
                'field_strength_comparison_raw_dopplergrams',
                str(config.get('data', {}).get('file', '')),
                str(config.get('data', {}).get('observable', '')),
                dataset,
            )

            if raw_key not in plot_cache:
                plot_cache[raw_key] = AGWFilter(config).load_dopplergrams()

            v1, v2 = plot_cache[raw_key]
            cube = v1 if dataset == 'v1' else v2
            filtering_module = AGWFilter(config)
            filter3D = filtering_module.build_gaussian_filter(cube)
            cube_xyt = np.transpose(cube, (2, 1, 0))
            _, kx_array, _, freq_array = spectral_analysis.make_fourier_grid(
                config['time_distance']['dt'],
                config['time_distance']['p_dx_Mm'],
                cube_xyt,
                threeD = True,
            )

            filter_slice = np.asarray(filter3D[:, filter3D.shape[1]//2, :], dtype = np.float64)
            positive_k = np.asarray(kx_array) >= 0.0
            k_plot = np.asarray(kx_array, dtype = np.float64)[positive_k]
            filter_slice = filter_slice[positive_k, :]

            plot_cache[cache_key] = {
                'enabled': True,
                'filter_slice': filter_slice,
                'k_plot': k_plot,
                'freq_array': np.asarray(freq_array, dtype = np.float64),
            }

    return plot_cache[cache_key]



def make_field_strength_comparison_plot(request, config, modules, plot_cache):

    '''
    Purpose
    -------
    Create field-strength comparison plot.
    
    Inputs
    ------
    request: object
        Request used by this helper.
    
    config: object
        Runtime configuration used by this helper.
    
    modules: object
        Loaded project modules used by this helper.
    
    plot_cache: object
        Plot cache used by this helper.
    
    Outputs
    -------
    created_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    request = copy.deepcopy(request)
    AGWFilter = modules['AGWFilter']
    spectral_analysis = modules['spectral_analysis']

    h1 = request.get('h1', None)
    h2 = request.get('h2', None)

    if h1 in ['', None] or h2 in ['', None]:
        raise ValueError("field_strength_comparison requires integer 'h1' and 'h2' entries in the request.")

    cube_paths = request.get('cube_paths', [])
    organized_cases = organize_single_cube_field_strength_cases(cube_paths)
    column_cases = build_field_strength_comparison_column_cases(organized_cases)
    observable = resolve_field_strength_comparison_observable(request, config)
    model_atmosphere_path = request.get(
        'model_atmosphere_path',
        config.get('data', {}).get('single_cube', {}).get('model_atmosphere_path', ''),
    )

    comparison_cases = []
    comparison_runtime_configs = []

    for case in column_cases:
        if case is None:
            comparison_cases.append(None)
            continue

        runtime_config = build_field_strength_comparison_runtime_config(
            config,
            case['cube_path'],
            h1,
            h2,
            observable = observable,
            model_atmosphere_path = model_atmosphere_path,
        )

        comparison_case = copy.deepcopy(case)
        comparison_case['runtime_config'] = runtime_config
        comparison_cases.append(comparison_case)
        comparison_runtime_configs.append(runtime_config)

    resolved_runtime_configs = ensure_comparison_runtime_products(
        comparison_runtime_configs,
        request,
        modules,
        plot_label = 'Field-strength comparison',
    )

    resolved_index = 0
    for case in comparison_cases:
        if case is None:
            continue

        case['runtime_config'] = resolved_runtime_configs[resolved_index]
        resolved_index += 1

    reference_case = next(case for case in comparison_cases if case is not None)
    reference_config = reference_case['runtime_config']
    h1_km = float(reference_config['data']['resolved_h1_km'])
    h2_km = float(reference_config['data']['resolved_h2_km'])
    reference_dt = float(reference_config['time_distance']['dt'])
    reference_dx = float(reference_config['time_distance']['p_dx_Mm'])

    for case in comparison_cases:
        if case is None:
            continue

        case_config = case['runtime_config']
        if not np.isclose(float(case_config['time_distance']['dt']), reference_dt, rtol = 1.0e-9, atol = 1.0e-12):
            raise ValueError('All field-strength comparison cubes must share the same cadence dt.')
        if not np.isclose(float(case_config['time_distance']['p_dx_Mm']), reference_dx, rtol = 1.0e-9, atol = 1.0e-12):
            raise ValueError('All field-strength comparison cubes must share the same horizontal sampling p_dx_Mm.')
        if not np.isclose(float(case_config['data']['resolved_h1_km']), h1_km, rtol = 1.0e-9, atol = 1.0e-6):
            raise ValueError('All field-strength comparison cubes must resolve the same lower height for the requested h1 index.')
        if not np.isclose(float(case_config['data']['resolved_h2_km']), h2_km, rtol = 1.0e-9, atol = 1.0e-6):
            raise ValueError('All field-strength comparison cubes must resolve the same upper height for the requested h2 index.')

    komega_request = build_plot_request_defaults('komega_diagram', reference_config)
    komega_request.update(copy.deepcopy(request.get('komega', {})))
    time_distance_request = build_plot_request_defaults('time_distance', reference_config)
    time_distance_request.update(copy.deepcopy(request.get('cross_correlation', {})))
    if time_distance_request.get('normalize_display', False):
        time_distance_request.setdefault('vmin', -1.0)
        time_distance_request.setdefault('vmax', 1.0)
    phase_request = build_plot_request_defaults('phase_difference', reference_config)
    phase_request.update(copy.deepcopy(request.get('phase_difference', {})))
    phase_request['cmap'] = resolve_phase_cmap(phase_request.get('cmap', 'seismic'))
    gaussian_request = build_plot_request_defaults('gaussian_filter', reference_config)
    gaussian_request.update(copy.deepcopy(request.get('gaussian_filter', {})))

    reference_komega = load_komega_data(reference_config['data']['komega_outfile'], reference_config)
    default_komega_xlim = get_axis_limits(
        reference_komega['k_axis'],
        np.median(np.diff(reference_komega['k_axis'])) if reference_komega['k_axis'].size > 1 else 1.0,
    )
    default_komega_ylim = get_axis_limits(
        reference_komega['nu_axis'],
        np.median(np.diff(reference_komega['nu_axis'])) if reference_komega['nu_axis'].size > 1 else 1.0,
    )
    komega_xlim = default_komega_xlim if komega_request.get('xlim', None) in ['', None] else komega_request['xlim']
    komega_ylim = default_komega_ylim if komega_request.get('ylim', None) in ['', None] else komega_request['ylim']

    reference_xc = load_time_distance_data(
        reference_config['data']['outfile'],
        reference_dt,
        reference_dx,
        axis_type = time_distance_request['axis_type'],
    )
    xc_xlim = get_axis_limits(reference_xc['radii'], reference_dx)
    default_xc_ylim = get_axis_limits(reference_xc['vertical_axis'], reference_dt/60.0)
    xc_ylim = default_xc_ylim if time_distance_request.get('ylim', None) in ['', None] else time_distance_request['ylim']

    reference_phase = load_time_distance_data(
        reference_config['data']['phase_outfile'],
        reference_dt,
        reference_dx,
        axis_type = phase_request['axis_type'],
    )
    phase_xlim = get_axis_limits(reference_phase['radii'], reference_dx)
    phase_vertical_step = 1.0e3/(reference_phase['values'].shape[1]*reference_dt)
    default_phase_ylim = get_axis_limits(reference_phase['vertical_axis'], phase_vertical_step)
    phase_ylim = default_phase_ylim if phase_request.get('ylim', None) in ['', None] else phase_request['ylim']

    fig = plt.figure(figsize = tuple(request.get('figsize', [31.5, 14.5])))
    grid = fig.add_gridspec(
        4,
        8,
        height_ratios = [1.0, 1.0, 1.0, 0.95],
        left = 0.04,
        right = 0.965,
        bottom = 0.06,
        top = 0.88,
        wspace = 0.22,
        hspace = 0.28,
    )

    axes = np.empty((3, 8), dtype = object)

    for column_index in range(8):
        axes[0, column_index] = fig.add_subplot(
            grid[0, column_index],
            sharey = axes[0, 0] if column_index > 0 else None,
        )
        axes[1, column_index] = fig.add_subplot(
            grid[1, column_index],
            sharey = axes[1, 0] if column_index > 0 else None,
        )
        axes[2, column_index] = fig.add_subplot(
            grid[2, column_index],
            sharex = axes[1, column_index],
            sharey = axes[2, 0] if column_index > 0 else None,
        )

    filter_grid = grid[3, 3:5].subgridspec(1, 2, width_ratios = [1.0, 0.06], wspace = 0.12)
    ax_filter = fig.add_subplot(filter_grid[0, 0])
    cax_filter = fig.add_subplot(filter_grid[0, 1])

    last_komega_mesh = None
    last_xc_mesh = None
    last_phase_mesh = None

    for column_index, case in enumerate(comparison_cases):
        ax_komega = axes[0, column_index]
        ax_xc = axes[1, column_index]
        ax_phase = axes[2, column_index]

        if case is None:
            ax_komega.axis('off')
            ax_xc.axis('off')
            ax_phase.axis('off')
            continue

        case_config = case['runtime_config']
        komega_data = load_komega_data(case_config['data']['komega_outfile'], case_config)
        last_komega_mesh = ax_komega.pcolormesh(
            komega_data['k_axis'],
            komega_data['nu_axis'],
            komega_data['values'].T,
            cmap = komega_request.get('cmap', cmr.fusion),
            norm = TwoSlopeNorm(
                vmin = komega_request.get('vmin', -20.0),
                vcenter = 0.0,
                vmax = komega_request.get('vmax', 80.0),
            ),
            shading = 'auto',
        )
        if komega_request.get('overlay_curves', True):
            add_komega_dispersion_overlays(ax_komega, komega_data['k_axis'])
        ax_komega.set_xlim(*komega_xlim)
        ax_komega.set_ylim(*komega_ylim)
        ax_komega.set_xlabel(komega_data['horizontal_label'])
        if column_index == 0:
            ax_komega.set_ylabel(komega_data['vertical_label'])
        else:
            ax_komega.tick_params(axis = 'y', labelleft = False)
        ax_komega.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
        ax_komega.set_title(case['field_strength_label'], pad = 8.0)

        xc_data = load_time_distance_data(
            case_config['data']['outfile'],
            float(case_config['time_distance']['dt']),
            float(case_config['time_distance']['p_dx_Mm']),
            axis_type = time_distance_request['axis_type'],
        )
        xc_values = np.asarray(xc_data['values'], dtype = np.float64)
        if time_distance_request.get('normalize_display', False):
            xc_values = normalize_cross_correlation_display(xc_values)
        last_xc_mesh = ax_xc.pcolormesh(
            xc_data['radii'],
            xc_data['vertical_axis'],
            xc_values.T,
            cmap = time_distance_request.get('cmap', 'viridis'),
            vmin = time_distance_request.get('vmin', None),
            vmax = time_distance_request.get('vmax', None),
            shading = 'auto',
        )
        ax_xc.set_xlim(*xc_xlim)
        ax_xc.set_ylim(*xc_ylim)
        if column_index == 0:
            ax_xc.set_ylabel(xc_data['vertical_label'])
        else:
            ax_xc.tick_params(axis = 'y', labelleft = False)
        ax_xc.tick_params(axis = 'x', which = 'both', bottom = False, labelbottom = False)
        ax_xc.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
        add_reference_angle_line(ax_xc, case_config)

        phase_data = load_time_distance_data(
            case_config['data']['phase_outfile'],
            float(case_config['time_distance']['dt']),
            float(case_config['time_distance']['p_dx_Mm']),
            axis_type = phase_request['axis_type'],
        )
        last_phase_mesh = ax_phase.pcolormesh(
            phase_data['radii'],
            phase_data['vertical_axis'],
            phase_data['values'].T,
            cmap = phase_request.get('cmap', 'seismic_r'),
            vmin = phase_request.get('vmin', None),
            vmax = phase_request.get('vmax', None),
            shading = 'auto',
        )
        ax_phase.set_xlim(*phase_xlim)
        ax_phase.set_ylim(*phase_ylim)
        ax_phase.set_xlabel('Radius [Mm]')
        if column_index == 0:
            ax_phase.set_ylabel(phase_data['vertical_label'])
        else:
            ax_phase.tick_params(axis = 'y', labelleft = False)
        ax_phase.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
        add_reference_angle_line(ax_phase, case_config)

    cax_komega = axes[0, 7].inset_axes([1.05, 0.0, 0.04, 1.0], transform = axes[0, 7].transAxes)
    cax_xc = axes[1, 7].inset_axes([1.05, 0.0, 0.04, 1.0], transform = axes[1, 7].transAxes)
    cax_phase = axes[2, 7].inset_axes([1.05, 0.0, 0.04, 1.0], transform = axes[2, 7].transAxes)
    fig.colorbar(last_komega_mesh, cax = cax_komega, label = komega_request.get('value_label', 'Value'))
    fig.colorbar(last_xc_mesh, cax = cax_xc, label = time_distance_request.get('value_label', 'Value'))
    fig.colorbar(last_phase_mesh, cax = cax_phase, label = phase_request.get('value_label', 'Value'))

    gaussian_panel = load_field_strength_comparison_gaussian_panel_data(
        reference_config,
        AGWFilter,
        spectral_analysis,
        plot_cache,
        dataset = gaussian_request.get('dataset', 'v1'),
    )

    if gaussian_panel.get('enabled', False):
        default_filter_xlim = get_axis_limits(
            gaussian_panel['k_plot'],
            np.median(np.diff(gaussian_panel['k_plot'])) if gaussian_panel['k_plot'].size > 1 else 1.0,
        )
        default_filter_ylim = get_axis_limits(
            gaussian_panel['freq_array'],
            np.median(np.diff(gaussian_panel['freq_array'])) if gaussian_panel['freq_array'].size > 1 else 1.0,
        )
        filter_xlim = default_filter_xlim if gaussian_request.get('xlim', None) in ['', None] else gaussian_request['xlim']
        filter_ylim = default_filter_ylim if gaussian_request.get('ylim', None) in ['', None] else gaussian_request['ylim']
        filter_image = ax_filter.imshow(
            gaussian_panel['filter_slice'].T,
            origin = 'lower',
            aspect = 'auto',
            extent = [default_filter_xlim[0], default_filter_xlim[1], default_filter_ylim[0], default_filter_ylim[1]],
            cmap = gaussian_request.get('cmap', 'viridis'),
        )
        if gaussian_request.get('overlay_lamb_line', True):
            add_komega_dispersion_overlays(ax_filter, gaussian_panel['k_plot'])
        ax_filter.set_xlim(*filter_xlim)
        ax_filter.set_ylim(*filter_ylim)
        fig.colorbar(filter_image, cax = cax_filter, label = 'Filter Transmission')
    else:
        ax_filter.text(0.5, 0.5, 'Gaussian disabled', ha = 'center', va = 'center', transform = ax_filter.transAxes)
        cax_filter.axis('off')

    ax_filter.set_xlabel(r'$k_h$ [Mm$^{-1}$]')
    ax_filter.set_ylabel(r'$\nu / 2\pi$ [mHz]')
    ax_filter.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
    ax_filter.set_title(build_field_strength_comparison_gaussian_title(reference_config), pad = 10.0)

    matrix_top = max(axes[0, column_index].get_position().y1 for column_index in range(8))
    matrix_bottom = min(axes[2, column_index].get_position().y0 for column_index in range(8))
    divider_x = 0.5*(axes[0, 3].get_position().x1 + axes[0, 4].get_position().x0)
    horizontal_center = 0.5*(axes[0, 0].get_position().x0 + axes[0, 3].get_position().x1)
    vertical_center = 0.5*(axes[0, 4].get_position().x0 + axes[0, 7].get_position().x1)
    fig.add_artist(Line2D([divider_x, divider_x], [matrix_bottom, matrix_top], transform = fig.transFigure, color = '0.45', linewidth = 1.2))
    fig.text(horizontal_center, matrix_top + 0.018, 'Horizontal', ha = 'center', va = 'bottom', fontsize = 14.0)
    fig.text(vertical_center, matrix_top + 0.018, 'Vertical', ha = 'center', va = 'bottom', fontsize = 14.0)
    set_centered_figure_title(fig, build_field_strength_comparison_suptitle(h1_km, h2_km), y = 0.975)

    output_stem = build_field_strength_comparison_output_stem(reference_config, organized_cases, h1_km, h2_km, observable)
    saved_file = maybe_save_figure(
        fig,
        request.get('output_file', ''),
        default_name = f'{output_stem}.jpeg',
        figure_dir = request.get('figure_dir', get_figure_dir(reference_config)),
    )

    if request.get('show', True):
        plt.show()
    else:
        plt.close(fig)

    return {
        'figure': fig,
        'axes': {'matrix': axes, 'gaussian_filter': ax_filter},
        'saved_file': saved_file,
        'h1_km': h1_km,
        'h2_km': h2_km,
        'cases': comparison_cases,
    }



def build_field_orientation_comparison_column_cases(organized_cases):

    '''
    Purpose
    -------
    Build field-orientation comparison column cases.
    
    Inputs
    ------
    organized_cases: object
        Organized cases used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    cases_by_geometry = organized_cases['cases_by_geometry']
    required_strengths_G = [10.0, 50.0, 100.0]
    required_columns = [
        ('horizontal', 10.0, 'h10G'),
        ('vertical', 10.0, 'v10G'),
        ('horizontal', 50.0, 'h50G'),
        ('vertical', 50.0, 'v50G'),
        ('horizontal', 100.0, 'h100G'),
        ('vertical', 100.0, 'v100G'),
    ]

    unsupported_labels = []

    for geometry in ['horizontal', 'vertical']:
        for case in cases_by_geometry[geometry]:
            strength_G = float(case['field_strength_G'])

            if np.isclose(strength_G, 0.0, rtol = 1.0e-9, atol = 1.0e-9):
                continue

            if not any(np.isclose(strength_G, required_strength_G, rtol = 1.0e-9, atol = 1.0e-9) for required_strength_G in required_strengths_G):
                prefix = 'h' if case['component'] == 'hx' else 'v'
                unsupported_labels.append(f"{prefix}{case['field_strength_token'].upper()}")

    if len(unsupported_labels) > 0:
        raise ValueError(
            'Field-orientation comparison supports only the 10G, 50G, and 100G cases. '
            f"Unsupported cases: {', '.join(sorted(unsupported_labels))}."
        )

    column_cases = []
    missing_labels = []

    for geometry, strength_G, column_label in required_columns:
        matching_cases = [
            case
            for case in cases_by_geometry[geometry]
            if np.isclose(float(case['field_strength_G']), strength_G, rtol = 1.0e-9, atol = 1.0e-9)
        ]

        if len(matching_cases) == 0:
            missing_labels.append(column_label)
            continue

        comparison_case = copy.deepcopy(matching_cases[0])
        comparison_case['column_label'] = column_label
        column_cases.append(comparison_case)

    if len(missing_labels) > 0:
        raise ValueError(
            'Field-orientation comparison requires all six nonzero simulation cases. '
            f"Missing cases: {', '.join(missing_labels)}."
        )

    return column_cases



def build_field_orientation_comparison_suptitle(h1_km, h2_km):

    '''
    Purpose
    -------
    Build field-orientation comparison suptitle.
    
    Inputs
    ------
    h1_km: object
        H1 km used by this helper.
    
    h2_km: object
        H2 km used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    return f'Field Orientation Comparison\n{_format_height_km(h1_km)} km vs {_format_height_km(h2_km)} km'



def build_field_orientation_comparison_output_stem(config, column_cases, h1_km, h2_km, observable):

    '''
    Purpose
    -------
    Build field-orientation comparison output stem.
    
    Inputs
    ------
    config: object
        Runtime configuration used by this helper.
    
    column_cases: object
        Column cases used by this helper.
    
    h1_km: object
        H1 km used by this helper.
    
    h2_km: object
        H2 km used by this helper.
    
    observable: object
        Observable name used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    observable_slug = normalize_observable_slug(observable)
    processing_slug = build_processing_labels(config)['slug']
    stem_parts = [
        'field_orientation_comparison',
        observable_slug,
        f"{_slugify(_format_height_km(h1_km))}km",
        f"{_slugify(_format_height_km(h2_km))}km",
    ]

    stem_parts.extend(case['column_label'].lower() for case in column_cases)

    if processing_slug != '':
        stem_parts.append(processing_slug)

    return _join_slug(stem_parts)



def make_field_orientation_comparison_plot(request, config, modules, plot_cache):

    '''
    Purpose
    -------
    Create field-orientation comparison plot.
    
    Inputs
    ------
    request: object
        Request used by this helper.
    
    config: object
        Runtime configuration used by this helper.
    
    modules: object
        Loaded project modules used by this helper.
    
    plot_cache: object
        Plot cache used by this helper.
    
    Outputs
    -------
    created_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    request = copy.deepcopy(request)
    AGWFilter = modules['AGWFilter']
    spectral_analysis = modules['spectral_analysis']

    h1 = request.get('h1', None)
    h2 = request.get('h2', None)

    if h1 in ['', None] or h2 in ['', None]:
        raise ValueError("field_orientation_comparison requires integer 'h1' and 'h2' entries in the request.")

    cube_paths = request.get('cube_paths', [])
    organized_cases = organize_single_cube_field_strength_cases(cube_paths)
    column_cases = build_field_orientation_comparison_column_cases(organized_cases)
    observable = resolve_field_strength_comparison_observable(request, config)
    model_atmosphere_path = request.get(
        'model_atmosphere_path',
        config.get('data', {}).get('single_cube', {}).get('model_atmosphere_path', ''),
    )

    comparison_cases = []
    comparison_runtime_configs = []

    for case in column_cases:
        runtime_config = build_field_strength_comparison_runtime_config(
            config,
            case['cube_path'],
            h1,
            h2,
            observable = observable,
            model_atmosphere_path = model_atmosphere_path,
        )

        comparison_case = copy.deepcopy(case)
        comparison_case['runtime_config'] = runtime_config
        comparison_cases.append(comparison_case)
        comparison_runtime_configs.append(runtime_config)

    resolved_runtime_configs = ensure_comparison_runtime_products(
        comparison_runtime_configs,
        request,
        modules,
        plot_label = 'Field-orientation comparison',
    )

    for comparison_case, runtime_config in zip(comparison_cases, resolved_runtime_configs):
        comparison_case['runtime_config'] = runtime_config

    reference_case = comparison_cases[0]
    reference_config = reference_case['runtime_config']
    h1_km = float(reference_config['data']['resolved_h1_km'])
    h2_km = float(reference_config['data']['resolved_h2_km'])
    reference_dt = float(reference_config['time_distance']['dt'])
    reference_dx = float(reference_config['time_distance']['p_dx_Mm'])

    for case in comparison_cases:
        case_config = case['runtime_config']
        if not np.isclose(float(case_config['time_distance']['dt']), reference_dt, rtol = 1.0e-9, atol = 1.0e-12):
            raise ValueError('All field-orientation comparison cubes must share the same cadence dt.')
        if not np.isclose(float(case_config['time_distance']['p_dx_Mm']), reference_dx, rtol = 1.0e-9, atol = 1.0e-12):
            raise ValueError('All field-orientation comparison cubes must share the same horizontal sampling p_dx_Mm.')
        if not np.isclose(float(case_config['data']['resolved_h1_km']), h1_km, rtol = 1.0e-9, atol = 1.0e-6):
            raise ValueError('All field-orientation comparison cubes must resolve the same lower height for the requested h1 index.')
        if not np.isclose(float(case_config['data']['resolved_h2_km']), h2_km, rtol = 1.0e-9, atol = 1.0e-6):
            raise ValueError('All field-orientation comparison cubes must resolve the same upper height for the requested h2 index.')

    komega_request = build_plot_request_defaults('komega_diagram', reference_config)
    komega_request.update(copy.deepcopy(request.get('komega', {})))
    time_distance_request = build_plot_request_defaults('time_distance', reference_config)
    time_distance_request.update(copy.deepcopy(request.get('cross_correlation', {})))
    if time_distance_request.get('normalize_display', False):
        time_distance_request.setdefault('vmin', -1.0)
        time_distance_request.setdefault('vmax', 1.0)
    phase_request = build_plot_request_defaults('phase_difference', reference_config)
    phase_request.update(copy.deepcopy(request.get('phase_difference', {})))
    phase_request['cmap'] = resolve_phase_cmap(phase_request.get('cmap', 'seismic'))
    gaussian_request = build_plot_request_defaults('gaussian_filter', reference_config)
    gaussian_request.update(copy.deepcopy(request.get('gaussian_filter', {})))

    reference_komega = load_komega_data(reference_config['data']['komega_outfile'], reference_config)
    default_komega_xlim = get_axis_limits(
        reference_komega['k_axis'],
        np.median(np.diff(reference_komega['k_axis'])) if reference_komega['k_axis'].size > 1 else 1.0,
    )
    default_komega_ylim = get_axis_limits(
        reference_komega['nu_axis'],
        np.median(np.diff(reference_komega['nu_axis'])) if reference_komega['nu_axis'].size > 1 else 1.0,
    )
    komega_xlim = default_komega_xlim if komega_request.get('xlim', None) in ['', None] else komega_request['xlim']
    komega_ylim = default_komega_ylim if komega_request.get('ylim', None) in ['', None] else komega_request['ylim']

    reference_xc = load_time_distance_data(
        reference_config['data']['outfile'],
        reference_dt,
        reference_dx,
        axis_type = time_distance_request['axis_type'],
    )
    xc_xlim = get_axis_limits(reference_xc['radii'], reference_dx)
    default_xc_ylim = get_axis_limits(reference_xc['vertical_axis'], reference_dt/60.0)
    xc_ylim = default_xc_ylim if time_distance_request.get('ylim', None) in ['', None] else time_distance_request['ylim']

    reference_phase = load_time_distance_data(
        reference_config['data']['phase_outfile'],
        reference_dt,
        reference_dx,
        axis_type = phase_request['axis_type'],
    )
    phase_xlim = get_axis_limits(reference_phase['radii'], reference_dx)
    phase_vertical_step = 1.0e3/(reference_phase['values'].shape[1]*reference_dt)
    default_phase_ylim = get_axis_limits(reference_phase['vertical_axis'], phase_vertical_step)
    phase_ylim = default_phase_ylim if phase_request.get('ylim', None) in ['', None] else phase_request['ylim']

    fig = plt.figure(figsize = tuple(request.get('figsize', [24.0, 14.5])))
    grid = fig.add_gridspec(
        4,
        6,
        height_ratios = [1.0, 1.0, 1.0, 0.95],
        left = 0.05,
        right = 0.955,
        bottom = 0.06,
        top = 0.88,
        wspace = 0.24,
        hspace = 0.28,
    )

    axes = np.empty((3, 6), dtype = object)

    for column_index in range(6):
        axes[0, column_index] = fig.add_subplot(
            grid[0, column_index],
            sharey = axes[0, 0] if column_index > 0 else None,
        )
        axes[1, column_index] = fig.add_subplot(
            grid[1, column_index],
            sharey = axes[1, 0] if column_index > 0 else None,
        )
        axes[2, column_index] = fig.add_subplot(
            grid[2, column_index],
            sharex = axes[1, column_index],
            sharey = axes[2, 0] if column_index > 0 else None,
        )

    filter_grid = grid[3, 2:4].subgridspec(1, 2, width_ratios = [1.0, 0.06], wspace = 0.12)
    ax_filter = fig.add_subplot(filter_grid[0, 0])
    cax_filter = fig.add_subplot(filter_grid[0, 1])

    last_komega_mesh = None
    last_xc_mesh = None
    last_phase_mesh = None

    for column_index, case in enumerate(comparison_cases):
        ax_komega = axes[0, column_index]
        ax_xc = axes[1, column_index]
        ax_phase = axes[2, column_index]

        case_config = case['runtime_config']
        komega_data = load_komega_data(case_config['data']['komega_outfile'], case_config)
        last_komega_mesh = ax_komega.pcolormesh(
            komega_data['k_axis'],
            komega_data['nu_axis'],
            komega_data['values'].T,
            cmap = komega_request.get('cmap', cmr.fusion),
            norm = TwoSlopeNorm(
                vmin = komega_request.get('vmin', -20.0),
                vcenter = 0.0,
                vmax = komega_request.get('vmax', 80.0),
            ),
            shading = 'auto',
        )
        if komega_request.get('overlay_curves', True):
            add_komega_dispersion_overlays(ax_komega, komega_data['k_axis'])
        ax_komega.set_xlim(*komega_xlim)
        ax_komega.set_ylim(*komega_ylim)
        ax_komega.set_xlabel(komega_data['horizontal_label'])
        if column_index == 0:
            ax_komega.set_ylabel(komega_data['vertical_label'])
        else:
            ax_komega.tick_params(axis = 'y', labelleft = False)
        ax_komega.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
        ax_komega.set_title(case['column_label'], pad = 8.0)

        xc_data = load_time_distance_data(
            case_config['data']['outfile'],
            float(case_config['time_distance']['dt']),
            float(case_config['time_distance']['p_dx_Mm']),
            axis_type = time_distance_request['axis_type'],
        )
        xc_values = np.asarray(xc_data['values'], dtype = np.float64)
        if time_distance_request.get('normalize_display', False):
            xc_values = normalize_cross_correlation_display(xc_values)
        last_xc_mesh = ax_xc.pcolormesh(
            xc_data['radii'],
            xc_data['vertical_axis'],
            xc_values.T,
            cmap = time_distance_request.get('cmap', 'viridis'),
            vmin = time_distance_request.get('vmin', None),
            vmax = time_distance_request.get('vmax', None),
            shading = 'auto',
        )
        ax_xc.set_xlim(*xc_xlim)
        ax_xc.set_ylim(*xc_ylim)
        if column_index == 0:
            ax_xc.set_ylabel(xc_data['vertical_label'])
        else:
            ax_xc.tick_params(axis = 'y', labelleft = False)
        ax_xc.tick_params(axis = 'x', which = 'both', bottom = False, labelbottom = False)
        ax_xc.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
        add_reference_angle_line(ax_xc, case_config)

        phase_data = load_time_distance_data(
            case_config['data']['phase_outfile'],
            float(case_config['time_distance']['dt']),
            float(case_config['time_distance']['p_dx_Mm']),
            axis_type = phase_request['axis_type'],
        )
        last_phase_mesh = ax_phase.pcolormesh(
            phase_data['radii'],
            phase_data['vertical_axis'],
            phase_data['values'].T,
            cmap = phase_request.get('cmap', 'seismic_r'),
            vmin = phase_request.get('vmin', None),
            vmax = phase_request.get('vmax', None),
            shading = 'auto',
        )
        ax_phase.set_xlim(*phase_xlim)
        ax_phase.set_ylim(*phase_ylim)
        ax_phase.set_xlabel('Radius [Mm]')
        if column_index == 0:
            ax_phase.set_ylabel(phase_data['vertical_label'])
        else:
            ax_phase.tick_params(axis = 'y', labelleft = False)
        ax_phase.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
        add_reference_angle_line(ax_phase, case_config)

    cax_komega = axes[0, 5].inset_axes([1.05, 0.0, 0.04, 1.0], transform = axes[0, 5].transAxes)
    cax_xc = axes[1, 5].inset_axes([1.05, 0.0, 0.04, 1.0], transform = axes[1, 5].transAxes)
    cax_phase = axes[2, 5].inset_axes([1.05, 0.0, 0.04, 1.0], transform = axes[2, 5].transAxes)
    fig.colorbar(last_komega_mesh, cax = cax_komega, label = komega_request.get('value_label', 'Value'))
    fig.colorbar(last_xc_mesh, cax = cax_xc, label = time_distance_request.get('value_label', 'Value'))
    fig.colorbar(last_phase_mesh, cax = cax_phase, label = phase_request.get('value_label', 'Value'))

    gaussian_panel = load_field_strength_comparison_gaussian_panel_data(
        reference_config,
        AGWFilter,
        spectral_analysis,
        plot_cache,
        dataset = gaussian_request.get('dataset', 'v1'),
    )

    if gaussian_panel.get('enabled', False):
        default_filter_xlim = get_axis_limits(
            gaussian_panel['k_plot'],
            np.median(np.diff(gaussian_panel['k_plot'])) if gaussian_panel['k_plot'].size > 1 else 1.0,
        )
        default_filter_ylim = get_axis_limits(
            gaussian_panel['freq_array'],
            np.median(np.diff(gaussian_panel['freq_array'])) if gaussian_panel['freq_array'].size > 1 else 1.0,
        )
        filter_xlim = default_filter_xlim if gaussian_request.get('xlim', None) in ['', None] else gaussian_request['xlim']
        filter_ylim = default_filter_ylim if gaussian_request.get('ylim', None) in ['', None] else gaussian_request['ylim']
        filter_image = ax_filter.imshow(
            gaussian_panel['filter_slice'].T,
            origin = 'lower',
            aspect = 'auto',
            extent = [default_filter_xlim[0], default_filter_xlim[1], default_filter_ylim[0], default_filter_ylim[1]],
            cmap = gaussian_request.get('cmap', 'viridis'),
        )
        if gaussian_request.get('overlay_lamb_line', True):
            add_komega_dispersion_overlays(ax_filter, gaussian_panel['k_plot'])
        ax_filter.set_xlim(*filter_xlim)
        ax_filter.set_ylim(*filter_ylim)
        fig.colorbar(filter_image, cax = cax_filter, label = 'Filter Transmission')
    else:
        ax_filter.text(0.5, 0.5, 'Gaussian disabled', ha = 'center', va = 'center', transform = ax_filter.transAxes)
        cax_filter.axis('off')

    ax_filter.set_xlabel(r'$k_h$ [Mm$^{-1}$]')
    ax_filter.set_ylabel(r'$\nu / 2\pi$ [mHz]')
    ax_filter.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
    ax_filter.set_title(build_field_strength_comparison_gaussian_title(reference_config), pad = 10.0)

    matrix_top = max(axes[0, column_index].get_position().y1 for column_index in range(6))
    matrix_bottom = min(axes[2, column_index].get_position().y0 for column_index in range(6))
    divider_positions = [
        0.5*(axes[0, 1].get_position().x1 + axes[0, 2].get_position().x0),
        0.5*(axes[0, 3].get_position().x1 + axes[0, 4].get_position().x0),
    ]

    for divider_x in divider_positions:
        fig.add_artist(Line2D([divider_x, divider_x], [matrix_bottom, matrix_top], transform = fig.transFigure, color = '0.45', linewidth = 1.2))

    set_centered_figure_title(fig, build_field_orientation_comparison_suptitle(h1_km, h2_km), y = 0.975)

    output_stem = build_field_orientation_comparison_output_stem(reference_config, comparison_cases, h1_km, h2_km, observable)
    saved_file = maybe_save_figure(
        fig,
        request.get('output_file', ''),
        default_name = f'{output_stem}.jpeg',
        figure_dir = request.get('figure_dir', get_figure_dir(reference_config)),
    )

    if request.get('show', True):
        plt.show()
    else:
        plt.close(fig)

    return {
        'figure': fig,
        'axes': {'matrix': axes, 'gaussian_filter': ax_filter},
        'saved_file': saved_file,
        'h1_km': h1_km,
        'h2_km': h2_km,
        'cases': comparison_cases,
    }



def build_gaussian_filter_comparison_filter_slug(filter_param_spec):

    '''
    Purpose
    -------
    Build Gaussian-filter comparison filter slug.
    
    Inputs
    ------
    filter_param_spec: object
        Filter param spec used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    gaussian_config = filter_param_spec.get('gaussian', filter_param_spec)

    return _join_slug([
        'gauss',
        'ck', f"{float(gaussian_config['central_k']):g}",
        'wk', f"{float(gaussian_config['width_k']):g}",
        'cf', f"{float(gaussian_config['central_f']):g}",
        'wf', f"{float(gaussian_config['width_f']):g}",
    ])



def build_gaussian_filter_comparison_filter_label(filter_param_spec):

    '''
    Purpose
    -------
    Build Gaussian-filter comparison filter label.
    
    Inputs
    ------
    filter_param_spec: object
        Filter param spec used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    gaussian_config = filter_param_spec.get('gaussian', filter_param_spec)
    sigma_k = float(gaussian_config['width_k'])/2.355
    sigma_nu = float(gaussian_config['width_f'])/2.355

    return (
        rf'$k_0 = {float(gaussian_config["central_k"]):g}\,\mathrm{{Mm^{{-1}}}},\ '
        rf'\sigma_k = {sigma_k:g}\,\mathrm{{Mm^{{-1}}}},\ '
        rf'\nu_0 = {float(gaussian_config["central_f"]):g}\,\mathrm{{mHz}},\ '
        rf'\sigma_\nu = {sigma_nu:g}\,\mathrm{{mHz}}$'
    )



def build_gaussian_filter_comparison_suptitle(h1_km, h2_km):

    '''
    Purpose
    -------
    Build Gaussian-filter comparison suptitle.
    
    Inputs
    ------
    h1_km: object
        H1 km used by this helper.
    
    h2_km: object
        H2 km used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    return f'Gaussian Filter Comparison\n{_format_height_km(h1_km)} km vs {_format_height_km(h2_km)} km'



def build_gaussian_filter_comparison_output_stem(config, filter_param_specs, h1_km, h2_km, observable):

    '''
    Purpose
    -------
    Build Gaussian-filter comparison output stem.
    
    Inputs
    ------
    config: object
        Runtime configuration used by this helper.
    
    filter_param_specs: object
        Filter param specs used by this helper.
    
    h1_km: object
        H1 km used by this helper.
    
    h2_km: object
        H2 km used by this helper.
    
    observable: object
        Observable name used by this helper.
    
    Outputs
    -------
    built_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    observable_slug = normalize_observable_slug(observable)
    stem_parts = [
        'gaussian_filter_comparison',
        observable_slug,
        f"{_slugify(_format_height_km(h1_km))}km",
        f"{_slugify(_format_height_km(h2_km))}km",
        f'filters_{len(filter_param_specs)}',
    ]

    for filter_index, filter_param_spec in enumerate(filter_param_specs, start = 1):
        stem_parts.append(_join_slug([f'f{filter_index}', build_gaussian_filter_comparison_filter_slug(filter_param_spec)]))

    processing_slug = build_processing_labels(config)['slug']
    if processing_slug != '':
        stem_parts.append(processing_slug)

    return _join_slug(stem_parts)



def add_gaussian_filter_comparison_legend(ax, filter_param_spec, filter_index):

    '''
    Purpose
    -------
    Add Gaussian-filter comparison legend.
    
    Inputs
    ------
    ax: object
        Ax used by this helper.
    
    filter_param_spec: object
        Filter param spec used by this helper.
    
    filter_index: object
        Filter index used by this helper.
    
    Outputs
    -------
    output: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    legend_title = str(filter_param_spec.get('label', '')).strip()
    if legend_title == '':
        legend_title = f'Filter {filter_index + 1}'

    proxy_handle = Line2D([], [], linestyle = '', linewidth = 0.0, color = 'none', label = build_gaussian_filter_comparison_filter_label(filter_param_spec))
    legend = ax.legend(
        handles = [proxy_handle],
        loc = 'upper left',
        frameon = True,
        framealpha = 0.92,
        handlelength = 0.0,
        handletextpad = 0.0,
        borderpad = 0.45,
        fontsize = 9.5,
        title = legend_title,
    )

    for handle in legend.legendHandles:
        handle.set_visible(False)

    legend._legend_box.align = 'left'

    return legend



def make_gaussian_filter_comparison_plot(request, config, modules, plot_cache):

    '''
    Purpose
    -------
    Create Gaussian-filter comparison plot.
    
    Inputs
    ------
    request: object
        Request used by this helper.
    
    config: object
        Runtime configuration used by this helper.
    
    modules: object
        Loaded project modules used by this helper.
    
    plot_cache: object
        Plot cache used by this helper.
    
    Outputs
    -------
    created_value: object
        Output returned by this helper.
    
    Author(s)
    ---------
    Julio M. Morales, March 22nd, 2026.
    '''

    request = copy.deepcopy(request)
    AGWFilter = modules['AGWFilter']
    spectral_analysis = modules['spectral_analysis']

    h1 = request.get('h1', None)
    h2 = request.get('h2', None)

    if h1 in ['', None] or h2 in ['', None]:
        raise ValueError("gaussian_filter_comparison requires integer 'h1' and 'h2' entries in the request.")

    cube_paths = request.get('cube_paths', [])
    organized_cases = organize_single_cube_gaussian_filter_comparison_cases(cube_paths)
    filter_param_specs = normalize_gaussian_filter_param_list(
        request.get('filter_params_list', []),
        reference_config = config,
    )
    observable = resolve_field_strength_comparison_observable(request, config)
    model_atmosphere_path = request.get(
        'model_atmosphere_path',
        config.get('data', {}).get('single_cube', {}).get('model_atmosphere_path', ''),
    )

    case_filter_records = []
    comparison_runtime_configs = []

    for case in organized_cases['ordered_cases']:
        filter_records = []
        for filter_param_spec in filter_param_specs:
            runtime_config = build_field_strength_comparison_runtime_config(
                config,
                case['cube_path'],
                h1,
                h2,
                observable = observable,
                model_atmosphere_path = model_atmosphere_path,
                filter_params = filter_param_spec,
            )

            filter_records.append(
                {
                    'case': copy.deepcopy(case),
                    'filter_param_spec': copy.deepcopy(filter_param_spec),
                    'runtime_config': runtime_config,
                }
            )
            comparison_runtime_configs.append(runtime_config)
        case_filter_records.append(filter_records)

    resolved_runtime_configs = ensure_comparison_runtime_products(
        comparison_runtime_configs,
        request,
        modules,
        plot_label = 'Gaussian-filter comparison',
    )

    resolved_index = 0
    for filter_records in case_filter_records:
        for record in filter_records:
            record['runtime_config'] = resolved_runtime_configs[resolved_index]
            resolved_index += 1

    reference_record = case_filter_records[0][0]
    reference_config = reference_record['runtime_config']
    h1_km = float(reference_config['data']['resolved_h1_km'])
    h2_km = float(reference_config['data']['resolved_h2_km'])
    reference_dt = float(reference_config['time_distance']['dt'])
    reference_dx = float(reference_config['time_distance']['p_dx_Mm'])

    for filter_records in case_filter_records:
        for record in filter_records:
            case_config = record['runtime_config']
            if not np.isclose(float(case_config['time_distance']['dt']), reference_dt, rtol = 1.0e-9, atol = 1.0e-12):
                raise ValueError('All Gaussian-filter comparison cubes must share the same cadence dt.')
            if not np.isclose(float(case_config['time_distance']['p_dx_Mm']), reference_dx, rtol = 1.0e-9, atol = 1.0e-12):
                raise ValueError('All Gaussian-filter comparison cubes must share the same horizontal sampling p_dx_Mm.')
            if not np.isclose(float(case_config['data']['resolved_h1_km']), h1_km, rtol = 1.0e-9, atol = 1.0e-6):
                raise ValueError('All Gaussian-filter comparison cubes must resolve the same lower height for the requested h1 index.')
            if not np.isclose(float(case_config['data']['resolved_h2_km']), h2_km, rtol = 1.0e-9, atol = 1.0e-6):
                raise ValueError('All Gaussian-filter comparison cubes must resolve the same upper height for the requested h2 index.')

    komega_request = build_plot_request_defaults('komega_diagram', reference_config)
    komega_request.update(copy.deepcopy(request.get('komega', {})))
    time_distance_request = build_plot_request_defaults('time_distance', reference_config)
    time_distance_request.update(copy.deepcopy(request.get('cross_correlation', {})))
    if time_distance_request.get('normalize_display', False):
        time_distance_request.setdefault('vmin', -1.0)
        time_distance_request.setdefault('vmax', 1.0)
    phase_request = build_plot_request_defaults('phase_difference', reference_config)
    phase_request.update(copy.deepcopy(request.get('phase_difference', {})))
    phase_request['cmap'] = resolve_phase_cmap(phase_request.get('cmap', 'seismic'))
    gaussian_request = build_plot_request_defaults('gaussian_filter', reference_config)
    gaussian_request.update(copy.deepcopy(request.get('gaussian_filter', {})))

    reference_komega = load_komega_data(reference_config['data']['komega_outfile'], reference_config)
    default_komega_xlim = get_axis_limits(
        reference_komega['k_axis'],
        np.median(np.diff(reference_komega['k_axis'])) if reference_komega['k_axis'].size > 1 else 1.0,
    )
    default_komega_ylim = get_axis_limits(
        reference_komega['nu_axis'],
        np.median(np.diff(reference_komega['nu_axis'])) if reference_komega['nu_axis'].size > 1 else 1.0,
    )
    komega_xlim = default_komega_xlim if komega_request.get('xlim', None) in ['', None] else komega_request['xlim']
    komega_ylim = default_komega_ylim if komega_request.get('ylim', None) in ['', None] else komega_request['ylim']

    reference_xc = load_time_distance_data(
        reference_config['data']['outfile'],
        reference_dt,
        reference_dx,
        axis_type = time_distance_request['axis_type'],
    )
    xc_xlim = get_axis_limits(reference_xc['radii'], reference_dx)
    default_xc_ylim = get_axis_limits(reference_xc['vertical_axis'], reference_dt/60.0)
    xc_ylim = default_xc_ylim if time_distance_request.get('ylim', None) in ['', None] else time_distance_request['ylim']

    reference_phase = load_time_distance_data(
        reference_config['data']['phase_outfile'],
        reference_dt,
        reference_dx,
        axis_type = phase_request['axis_type'],
    )
    phase_xlim = get_axis_limits(reference_phase['radii'], reference_dx)
    phase_vertical_step = 1.0e3/(reference_phase['values'].shape[1]*reference_dt)
    default_phase_ylim = get_axis_limits(reference_phase['vertical_axis'], phase_vertical_step)
    phase_ylim = default_phase_ylim if phase_request.get('ylim', None) in ['', None] else phase_request['ylim']

    filter_panels = []
    filter_values = []
    for filter_index, filter_param_spec in enumerate(filter_param_specs):
        filter_config = case_filter_records[0][filter_index]['runtime_config']
        gaussian_panel = load_field_strength_comparison_gaussian_panel_data(
            filter_config,
            AGWFilter,
            spectral_analysis,
            plot_cache,
            dataset = gaussian_request.get('dataset', 'v1'),
        )
        filter_panels.append(gaussian_panel)
        if gaussian_panel.get('enabled', False):
            filter_values.append(np.asarray(gaussian_panel['filter_slice'], dtype = np.float64))

    if len(filter_values) > 0:
        default_filter_xlim = get_axis_limits(
            filter_panels[0]['k_plot'],
            np.median(np.diff(filter_panels[0]['k_plot'])) if filter_panels[0]['k_plot'].size > 1 else 1.0,
        )
        default_filter_ylim = get_axis_limits(
            filter_panels[0]['freq_array'],
            np.median(np.diff(filter_panels[0]['freq_array'])) if filter_panels[0]['freq_array'].size > 1 else 1.0,
        )
        filter_vmin = gaussian_request.get('vmin', None)
        filter_vmax = gaussian_request.get('vmax', None)
        if filter_vmin in ['', None]:
            filter_vmin = float(min(np.nanmin(values) for values in filter_values))
        if filter_vmax in ['', None]:
            filter_vmax = float(max(np.nanmax(values) for values in filter_values))
    else:
        default_filter_xlim = [0.0, 1.0]
        default_filter_ylim = [0.0, 1.0]
        filter_vmin = gaussian_request.get('vmin', 0.0)
        filter_vmax = gaussian_request.get('vmax', 1.0)

    filter_xlim = default_filter_xlim if gaussian_request.get('xlim', None) in ['', None] else gaussian_request['xlim']
    filter_ylim = default_filter_ylim if gaussian_request.get('ylim', None) in ['', None] else gaussian_request['ylim']

    num_filters = len(filter_param_specs)
    num_cases = len(organized_cases['ordered_cases'])
    total_rows = 1 + 3*num_cases
    default_figsize = [max(4.2*num_filters + 2.0, 12.0), 32.0]
    figsize = default_figsize if request.get('figsize', None) in ['', None] else request['figsize']

    fig = plt.figure(figsize = tuple(figsize))
    grid = fig.add_gridspec(
        total_rows,
        num_filters,
        left = 0.11,
        right = 0.955,
        bottom = 0.025,
        top = 0.955,
        wspace = 0.24,
        hspace = 0.18,
        height_ratios = [1.1] + [1.0]*(total_rows - 1),
    )

    filter_axes = []
    for filter_index in range(num_filters):
        filter_axes.append(
            fig.add_subplot(
                grid[0, filter_index],
                sharey = filter_axes[0] if filter_index > 0 else None,
            )
        )

    matrix_axes = np.empty((num_cases, 3, num_filters), dtype = object)
    for case_index in range(num_cases):
        for filter_index in range(num_filters):
            row_start = 1 + 3*case_index
            matrix_axes[case_index, 0, filter_index] = fig.add_subplot(
                grid[row_start, filter_index],
                sharey = matrix_axes[case_index, 0, 0] if filter_index > 0 else None,
            )
            matrix_axes[case_index, 1, filter_index] = fig.add_subplot(
                grid[row_start + 1, filter_index],
                sharey = matrix_axes[case_index, 1, 0] if filter_index > 0 else None,
            )
            matrix_axes[case_index, 2, filter_index] = fig.add_subplot(
                grid[row_start + 2, filter_index],
                sharex = matrix_axes[case_index, 1, filter_index],
                sharey = matrix_axes[case_index, 2, 0] if filter_index > 0 else None,
            )

    for filter_index, (ax_filter, filter_param_spec, gaussian_panel) in enumerate(zip(filter_axes, filter_param_specs, filter_panels)):
        if gaussian_panel.get('enabled', False):
            filter_image = ax_filter.imshow(
                gaussian_panel['filter_slice'].T,
                origin = 'lower',
                aspect = 'auto',
                extent = [default_filter_xlim[0], default_filter_xlim[1], default_filter_ylim[0], default_filter_ylim[1]],
                cmap = gaussian_request.get('cmap', 'viridis'),
                vmin = filter_vmin,
                vmax = filter_vmax,
            )
            if gaussian_request.get('overlay_lamb_line', True):
                add_komega_dispersion_overlays(ax_filter, gaussian_panel['k_plot'])
            if filter_index == num_filters - 1:
                cax_filter = ax_filter.inset_axes([1.05, 0.0, 0.04, 1.0], transform = ax_filter.transAxes)
                fig.colorbar(filter_image, cax = cax_filter, label = 'Filter Transmission')
        else:
            ax_filter.text(0.5, 0.5, 'Gaussian disabled', ha = 'center', va = 'center', transform = ax_filter.transAxes)
            if filter_index == num_filters - 1:
                cax_filter = ax_filter.inset_axes([1.05, 0.0, 0.04, 1.0], transform = ax_filter.transAxes)
                cax_filter.axis('off')

        ax_filter.set_xlim(*filter_xlim)
        ax_filter.set_ylim(*filter_ylim)
        ax_filter.set_xlabel(r'$k_h$ [Mm$^{-1}$]')
        if filter_index == 0:
            ax_filter.set_ylabel(r'$\nu / 2\pi$ [mHz]')
        else:
            ax_filter.tick_params(axis = 'y', labelleft = False)
        ax_filter.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
        add_gaussian_filter_comparison_legend(ax_filter, filter_param_spec, filter_index)

    for case_index, filter_records in enumerate(case_filter_records):
        for filter_index, record in enumerate(filter_records):
            ax_komega = matrix_axes[case_index, 0, filter_index]
            ax_xc = matrix_axes[case_index, 1, filter_index]
            ax_phase = matrix_axes[case_index, 2, filter_index]
            case_config = record['runtime_config']

            komega_data = load_komega_data(case_config['data']['komega_outfile'], case_config)
            komega_mesh = ax_komega.pcolormesh(
                komega_data['k_axis'],
                komega_data['nu_axis'],
                komega_data['values'].T,
                cmap = komega_request.get('cmap', cmr.fusion),
                norm = TwoSlopeNorm(
                    vmin = komega_request.get('vmin', -20.0),
                    vcenter = 0.0,
                    vmax = komega_request.get('vmax', 80.0),
                ),
                shading = 'auto',
            )
            if komega_request.get('overlay_curves', True):
                add_komega_dispersion_overlays(ax_komega, komega_data['k_axis'])
            ax_komega.set_xlim(*komega_xlim)
            ax_komega.set_ylim(*komega_ylim)
            ax_komega.set_xlabel(komega_data['horizontal_label'])
            if filter_index == 0:
                ax_komega.set_ylabel(komega_data['vertical_label'])
            else:
                ax_komega.tick_params(axis = 'y', labelleft = False)
            ax_komega.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
            if filter_index == num_filters - 1:
                cax_komega = ax_komega.inset_axes([1.05, 0.0, 0.04, 1.0], transform = ax_komega.transAxes)
                fig.colorbar(komega_mesh, cax = cax_komega, label = komega_request.get('value_label', 'Value'))

            xc_data = load_time_distance_data(
                case_config['data']['outfile'],
                float(case_config['time_distance']['dt']),
                float(case_config['time_distance']['p_dx_Mm']),
                axis_type = time_distance_request['axis_type'],
            )
            xc_values = np.asarray(xc_data['values'], dtype = np.float64)
            if time_distance_request.get('normalize_display', False):
                xc_values = normalize_cross_correlation_display(xc_values)
            xc_mesh = ax_xc.pcolormesh(
                xc_data['radii'],
                xc_data['vertical_axis'],
                xc_values.T,
                cmap = time_distance_request.get('cmap', 'viridis'),
                vmin = time_distance_request.get('vmin', None),
                vmax = time_distance_request.get('vmax', None),
                shading = 'auto',
            )
            ax_xc.set_xlim(*xc_xlim)
            ax_xc.set_ylim(*xc_ylim)
            if filter_index == 0:
                ax_xc.set_ylabel(xc_data['vertical_label'])
            else:
                ax_xc.tick_params(axis = 'y', labelleft = False)
            ax_xc.tick_params(axis = 'x', which = 'both', bottom = False, labelbottom = False)
            ax_xc.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
            add_reference_angle_line(ax_xc, case_config)
            if filter_index == num_filters - 1:
                cax_xc = ax_xc.inset_axes([1.05, 0.0, 0.04, 1.0], transform = ax_xc.transAxes)
                fig.colorbar(xc_mesh, cax = cax_xc, label = time_distance_request.get('value_label', 'Value'))

            phase_data = load_time_distance_data(
                case_config['data']['phase_outfile'],
                float(case_config['time_distance']['dt']),
                float(case_config['time_distance']['p_dx_Mm']),
                axis_type = phase_request['axis_type'],
            )
            phase_mesh = ax_phase.pcolormesh(
                phase_data['radii'],
                phase_data['vertical_axis'],
                phase_data['values'].T,
                cmap = phase_request.get('cmap', 'seismic_r'),
                vmin = phase_request.get('vmin', None),
                vmax = phase_request.get('vmax', None),
                shading = 'auto',
            )
            ax_phase.set_xlim(*phase_xlim)
            ax_phase.set_ylim(*phase_ylim)
            ax_phase.set_xlabel('Radius [Mm]')
            if filter_index == 0:
                ax_phase.set_ylabel(phase_data['vertical_label'])
            else:
                ax_phase.tick_params(axis = 'y', labelleft = False)
            ax_phase.set_box_aspect(TARGET_PANEL_BOX_ASPECT)
            add_reference_angle_line(ax_phase, case_config)
            if filter_index == num_filters - 1:
                cax_phase = ax_phase.inset_axes([1.05, 0.0, 0.04, 1.0], transform = ax_phase.transAxes)
                fig.colorbar(phase_mesh, cax = cax_phase, label = phase_request.get('value_label', 'Value'))

    left_edge = filter_axes[0].get_position().x0
    right_edge = filter_axes[-1].get_position().x1

    first_matrix_top = max(matrix_axes[0, 0, filter_index].get_position().y1 for filter_index in range(num_filters))
    filter_bottom = min(ax.get_position().y0 for ax in filter_axes)
    divider_ys = [0.5*(filter_bottom + first_matrix_top)]

    for case_index in range(num_cases - 1):
        upper_bottom = min(matrix_axes[case_index, 2, filter_index].get_position().y0 for filter_index in range(num_filters))
        lower_top = max(matrix_axes[case_index + 1, 0, filter_index].get_position().y1 for filter_index in range(num_filters))
        divider_ys.append(0.5*(upper_bottom + lower_top))

    for divider_y in divider_ys:
        fig.add_artist(Line2D([left_edge, right_edge], [divider_y, divider_y], transform = fig.transFigure, color = '0.45', linewidth = 1.1))

    for case_index, case in enumerate(organized_cases['ordered_cases']):
        chunk_top = max(matrix_axes[case_index, 0, filter_index].get_position().y1 for filter_index in range(num_filters))
        chunk_bottom = min(matrix_axes[case_index, 2, filter_index].get_position().y0 for filter_index in range(num_filters))
        fig.text(
            left_edge - 0.055,
            0.5*(chunk_top + chunk_bottom),
            case['comparison_label'],
            rotation = 90,
            ha = 'center',
            va = 'center',
            fontsize = 12.5,
        )

    set_centered_figure_title(fig, build_gaussian_filter_comparison_suptitle(h1_km, h2_km), y = 0.985)

    output_stem = build_gaussian_filter_comparison_output_stem(reference_config, filter_param_specs, h1_km, h2_km, observable)
    saved_file = maybe_save_figure(
        fig,
        request.get('output_file', ''),
        default_name = f'{output_stem}.jpeg',
        figure_dir = request.get('figure_dir', get_figure_dir(reference_config)),
    )

    if request.get('show', True):
        plt.show()
    else:
        plt.close(fig)

    return {
        'figure': fig,
        'axes': {'filters': filter_axes, 'matrix': matrix_axes},
        'saved_file': saved_file,
        'h1_km': h1_km,
        'h2_km': h2_km,
        'filter_params_list': filter_param_specs,
        'cases': organized_cases['ordered_labels'],
    }


In [ ]:
# Load the shared configuration file and define the plot requests
# Resolve the shared notebook configuration before loading any modules.
config_file = resolve_config_file()
config = load_time_distance_config(config_file)
modules = load_project_modules(config_file)
# Expand the loaded config into the runtime form expected by the plotting helpers.
config = prepare_runtime_config(config)

# Use a wider k-omega range for simulation cubes
if config.get('data', {}).get('source_type', '') == 'single_cube':
    PLOT_DEFAULTS['komega_diagram']['vmin'] = -80.0

# Set the execution policy for comparison plots and missing-data handling.
comparison_execution_mode = 'load'  # 'load' or 'recompute'
comparison_missing_data_behavior = 'error'  # 'error' or 'recompute'

# Set this to False when you want to specify the plot requests directly below
use_config = True

# Example direct requests:
# direct_plot_requests = [
#     {
#         'plot_type': 'field_strength_comparison',
#         'cube_paths': ['/path/to/hx/10G/cube.nc', '/path/to/vx/10G/cube.nc'],
#         'h1': 0,
#         'h2': 1,
#         'execution_mode': 'load',
#     },
#     {
#         'plot_type': 'gaussian_filter_comparison',
#         'cube_paths': ['/path/to/z0/0G/cube.nc', '/path/to/hx/10G/cube.nc', '/path/to/hx/50G/cube.nc', '/path/to/hx/100G/cube.nc', '/path/to/vx/10G/cube.nc', '/path/to/vx/50G/cube.nc', '/path/to/vx/100G/cube.nc'],
#         'filter_params_list': [{'central_k': 2.0, 'width_k': 2.0, 'central_f': 1.5, 'width_f': 1.5}],
#         'h1': 0,
#         'h2': 1,
#         'execution_mode': 'recompute',
#     },
# ]
# Leave this list empty when all plot requests should come from the config file.
direct_plot_requests = []

print(config_file)


In [ ]:
# Generate only the requested plots and animations
# Cache expensive intermediate products so repeated plots can reuse them.
plot_cache = {}
# Resolve the final list of plot requests from the config and any direct overrides.
plot_requests = build_plot_requests(
    config,
    use_config = use_config,
    direct_plot_requests = direct_plot_requests,
    comparison_execution_mode = comparison_execution_mode,
    comparison_missing_data_behavior = comparison_missing_data_behavior,
)
# Collect the generated plot outputs for later inspection in the notebook.
generated_products = []

# Run each requested plot in sequence and collect the saved outputs.
if len(plot_requests) == 0:
    print('No plot requests were selected.')
else:
    for request in plot_requests:
        print(f"Running {request['plot_type']}")
        result = run_plot_request(request, config, modules, plot_cache)
        generated_products.append({'plot_type': request['plot_type'], 'result': result})
        if isinstance(result, dict):
            if result.get('saved_file', None) is not None:
                print(result['saved_file'])
        else:
            print(result)

generated_products
